# NEON GRSM·ORNL 2021: 항공사진 해상도 실험 추가

**Google Colab / T4 GPU. 기본은 `RESOLUTION_ONLY=True`: 지난주 3차 완료 결과만 불러옵니다.**
상단에서 하단까지 실행하면 기존 자료 다운로드·10cm baseline/A/B/C 추론을 건너뛰고,
Drive에 있는 원본 RGB/CHM crop으로25cm·51cm RGB 실험만 추가합니다.
같은23개 조사구(GRSM15+ORNL8), 같은 매칭·공식·CHM을 유지합니다.
첫 설치 후 런타임 재시작 안내가 나오면 재시작하고 Drive 설정 셀부터 실행하세요.

`RESOLUTION_BASE_DIR=None`은 완료된3차 결과 폴더나 Drive의 `outputs/neon_results.zip`을 자동 탐색합니다.
없거나 서로 다른 결과가 여러 개면 중단하며, 새 영상 다운로드나 기존 방법 재계산으로 대체하지 않습니다.
필요하면02번 셀의 `RESOLUTION_BASE_DIR`에3차 결과 폴더 경로를 넣으세요.
새 결과는 별도 `outputs/resolution_…/`, RGB·탐지 캐시는 `data/cache/resolution/…/`에 저장합니다.
HF 캐시는 `/content/hf_cache`, 마지막 다운로드는 `/content/neon_results.zip`입니다.

원본 설명과 기존 분석 코드는 아래에 보존했으며 기본 실행에서는 정의와 import만 읽습니다.

# NEON GRSM·ORNL 2021: 나무 탐지 → 수고·수관 → DBH → AGB·탄소

**실행 환경: Google Colab. Windows에서는 실행하지 않습니다.**

실행 순서: `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하고 위에서 아래로 실행합니다.
첫 실행은 설치 5–15분, 실측·계수 다운로드 5–20분, RGB/CHM 다운로드 20분–수 시간,
탐지·비교 10–60분 정도를 예상합니다. 이는 계획용 추정이며 타일 수·Drive 속도·GPU에 따라 달라집니다.
10cm RGB 한 타일은 압축 전 약 300MB이므로 Drive 여유 공간 10–30GB 이상을 권장합니다.
다운로드 전에 예상 타일 수를 출력합니다. 작은 실행은 `max_plots_per_site=2`로 시작하세요.

NEON API 토큰이 필요할 수 있습니다. [NEON 계정](https://data.neonscience.org/)의 API Tokens에서 발급하고
Colab 보안 비밀 `NEON_TOKEN`에 등록하거나 숨김 입력창에 입력하세요. 토큰은 Drive에 저장하지 않습니다.

데이터·결과는 `/content/drive/MyDrive/neon_grsm_ornl/`에 저장합니다.
HF 모델 캐시는 Drive와 분리된 `/content/hf_cache`이며 새 런타임에서는 다시 받을 수 있습니다.
완료된 다운로드와 매개변수가 같은 중간 결과는 재사용합니다. 설정이 바뀌면 별도 실행 폴더에 결과를 만듭니다.

**비교 대상의 범위:** 기본은 2021년 실측, 살아 있는 DBH≥10cm의 나무입니다.
작은 나무의 중첩 소구획 조사와 전체 조사 면적을 혼합하지 않기 위한 공통 비교 범위입니다.
실측 좌표가 없는 개체도 조사구 참조 총량에 포함합니다. 좌표는 개체 매칭에만 필요합니다.
ORNL의 2021-06 RGB/CHM은 API에서 PROVISIONAL로 제공되므로 이를 명시적으로 포함합니다.
NEONForestAGB는 실측 DBH로 계산한 참조 추정값이며 직접 무게를 잰 정답이 아닙니다.
탐지 정확도는 수관의 정답 윤곽이 아닌 **줄기 위치 기반 대리지표**입니다.
모델은 NEON 사전학습 가중치를 그대로 사용하므로 완전히 독립적인 외부 지역 검증은 아닙니다.

## 01. 패키지 설치

첫 코드 셀입니다. 설치 후 Colab이 런타임 재시작을 요청하면 재시작하고 다음 셀부터 실행하세요.

In [ ]:
%pip -q install "deepforest==2.1.0" "neonutilities>=2.0.1" rasterio geopandas shapely pyproj "scikit-learn==1.6.1" matplotlib scipy scikit-image requests tabulate

## 02. Drive 마운트와 재사용 설정

한국 자료에는 아래 입력·공식 설정과 자료 로더를 바꾸고, 뒤의 탐지·추정·매칭·평가 함수를 재사용합니다.
`ALLOW_ADJACENT_YEARS=False`가 기본입니다. 켜면 2021년 조사구를 유지하고 2021년 없는 조사구만
2020→2022 순서로 추가합니다. 서로 다른 해의 개체를 한 조사구에 섞지 않습니다.
보완 자료는 2021 결과와 분리 집계합니다.

**해상도 추가 실행:** RESOLUTION_ONLY=True를 유지하세요. NEON 토큰은 묻지 않습니다. CFG의 .15/400은 보존된 baseline용이며, 새 고정 실험은 별도 .2/600입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, hashlib, inspect, re, time, warnings, importlib.metadata, getpass, tempfile

ROOT = Path('/content/drive/MyDrive/neon_grsm_ornl')
RESOLUTION_ONLY = True  # 기존3차 자료만 재사용; False로 바꾸지 마세요.
RESOLUTION_BASE_DIR = None  # 예: ROOT/'outputs/f55cc4636e94'
CFG = dict(
    sites=['GRSM','ORNL'], year=2021, allow_adjacent_years=False,
    # 실측 연도 보완과 별개인 영상 대체 정책. 2021 공통 자료가 있으면 항상 우선합니다.
    imagery_allow_adjacent_years={'GRSM':False,'ORNL':True},
    imagery_adjacent_priority=[2022,2020],
    aop_include_provisional=True, api_refresh=True,
    adjacent_priority=[2020,2022], min_plots_per_site=5,
    max_plots_per_site=None, # 2로 바꾸면 소규모 점검
    release='current', buffer_m=50, patch_size=400, patch_overlap=0.25,
    score_threshold=0.15, nms_iou=0.15, min_height_m=3.0,
    expected_plot_count=23,
    comparison_plot_ids=['GRSM_001','GRSM_009','GRSM_012','GRSM_014','GRSM_016','GRSM_017',
        'GRSM_047','GRSM_049','GRSM_050','GRSM_051','GRSM_055','GRSM_057','GRSM_058','GRSM_059','GRSM_060',
        'ORNL_001','ORNL_004','ORNL_006','ORNL_009','ORNL_010','ORNL_027','ORNL_049','ORNL_057'],
    search_score_thresholds=[0.1,0.2,0.3,0.4,0.5],search_patch_sizes=[300,400,600],
    search_inner_folds=4,chm_sigma_m=1.0,chm_window_a=2.51,chm_window_b=0.00901,
    fusion_dedup_m=1.0,default_method_metric='carbon_RMSE_tC_ha',
    comparison_overlay_plots=['GRSM_016','GRSM_051','GRSM_012','ORNL_010'],
    min_field_dbh_cm=10.0, visible_height_ratio=0.8, visible_radius_m=3.0,
    match_size='stemDiameter', # 가장 큰 DBH 우선, 동률 때 중심거리
    jucker=dict(a=0.367, b=0.887, g=0.0, biome=11),
    # itcSegment/R/dbh.R biome 11: Nearctic-Temperate mixed forests-Angiosperm
    agb_default=dict(beta0=-2.2118, beta1=2.4133,
                     group='Chojnacky general hardwood surrogate (Cor/Eri/Lau/etc)'),
    carbon_fraction=0.5, carbon_alternative=0.47, cv_folds=5,
    model_name='weecology/deepforest-tree', model_revision='main',
    # 한국 자료용: site/plot/year/area_m2와 미터 투영좌표 polygon이 있는 GPKG/GeoJSON
    footprint_override=None,
    local_rgb={}, local_chm={}, # 예: {'KOREA_plot1':'/content/drive/.../rgb.tif'}
)
for d in ['data/raw','data/cache','data/crops','models','outputs']:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT/'data/raw')
tempfile.tempdir = str(ROOT/'data/cache')
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_CACHE'] = '/content/hf_cache/hub'
Path('/content/hf_cache').mkdir(parents=True,exist_ok=True)
os.environ['TORCH_HOME'] = str(ROOT/'models/torch')
os.environ['MPLCONFIGDIR'] = str(ROOT/'data/cache/matplotlib')
RUN_ID = hashlib.sha256(json.dumps(dict(config=CFG,versions={p:importlib.metadata.version(p)
    for p in ['deepforest','neonutilities','rasterio','geopandas']}),sort_keys=True).encode()).hexdigest()[:12]
OUT = ROOT/'outputs'/('resolution_setup' if RESOLUTION_ONLY else RUN_ID)
OUT.mkdir(exist_ok=True)
(OUT/'config.json').write_text(json.dumps(CFG,ensure_ascii=False,indent=2))
TOKEN = ''  # 해상도 실험에는 NEON 토큰/영상 다운로드 불필요
if not RESOLUTION_ONLY:
    try:
        from google.colab import userdata
        TOKEN = userdata.get('NEON_TOKEN')
    except Exception:
        TOKEN = os.environ.get('NEON_TOKEN') or getpass.getpass('NEON API token (숨김 입력): ')
    os.environ['NEON_TOKEN'] = TOKEN
print('결과 폴더:', OUT)

## 03. 공통 라이브러리·캐시·버전 기록

파일은 임시 파일로 받은 뒤 완료 시 이름을 바꿉니다. 위치 API의 성공 응답을 캐시하고 인증 오류를 숨기지 않습니다.

In [ ]:
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.windows import from_bounds, Window
from rasterio.plot import show
from shapely.geometry import Point, box, Polygon
from shapely.ops import unary_union
from pyproj import CRS, Transformer
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import neonutilities as nu

session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=Retry(total=5,backoff_factor=1,
    status_forcelist=[429,500,502,503,504])))
VERSIONS = {p:importlib.metadata.version(p) for p in
    ['deepforest','neonutilities','numpy','pandas','rasterio','geopandas','shapely','pyproj','scikit-learn']}
(OUT/'versions.json').write_text(json.dumps(VERSIONS,indent=2))
AUDIT = []
def audit(stage, item, reason):
    AUDIT.append(dict(stage=stage,item=str(item),reason=str(reason)))

def cached_download(url, dest):
    dest=Path(dest)
    if dest.exists() and dest.stat().st_size>0: return dest
    tmp=dest.with_suffix(dest.suffix+'.part')
    with session.get(url,stream=True,timeout=180) as r:
        r.raise_for_status()
        with tmp.open('wb') as f:
            for block in r.iter_content(1024*1024): f.write(block)
        # requests가 gzip을 해제한 길이는 전송된 Content-Length와 다릅니다.
        if not r.headers.get('Content-Encoding') and r.headers.get('Content-Length') and tmp.stat().st_size != int(r.headers['Content-Length']):
            raise IOError('다운로드 크기 불일치: '+url)
    tmp.replace(dest)
    return dest

def cached_location(name):
    key=hashlib.sha256(name.encode()).hexdigest()
    path=ROOT/'data/cache'/f'location_{key}.json'
    if path.exists(): return json.loads(path.read_text())
    url='https://data.neonscience.org/api/v0/locations/'+requests.utils.quote(name,safe='._')
    r=session.get(url,headers={'X-API-Token':TOKEN},timeout=60)
    r.raise_for_status()
    data=r.json()['data']
    path.write_text(json.dumps(data,ensure_ascii=False))
    return data

def location_xy(d):
    zone=int(d['locationUtmZone'])
    south=d.get('locationUtmHemisphere','N')=='S'
    return float(d['locationUtmEasting']),float(d['locationUtmNorthing']),CRS.from_epsg((32700 if south else 32600)+zone)

def require_columns(df, cols, label):
    missing=set(cols)-set(df.columns)
    if missing: raise ValueError(f'{label} 필드 누락: {missing}; 실제 필드={list(df.columns)}')

def choose_table(tables, name):
    keys=[k for k in tables if k==name or k.endswith(name)]
    if len(keys)!=1: raise KeyError(f'{name}: {keys}; tables={list(tables)}')
    return tables[keys[0]].copy()

print(VERSIONS)
print('load_by_product',inspect.signature(nu.load_by_product))
print('by_tile_aop',inspect.signature(nu.by_tile_aop))

## 04. 실측 자료 다운로드

공식 `load_by_product`로 연도별 자료를 받고 표를 Drive에 저장합니다.
2024년 이후 배포본은 과거 자료의 subplotID도 `31_100`, `21_400` 같은 형식으로 수정했습니다.
DBH≥10cm 복수 줄기는 NEON에서 개별 individualID가 부여되므로 서로 합치지 않습니다.
같은 ID의 중복 방문 기록은 마지막 측정만 사용합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def load_field_year(site, year):
    folder=ROOT/'data/raw'/f'vst_{site}_{year}_{CFG["release"]}'
    folder.mkdir(exist_ok=True)
    done=folder/'complete.json'
    if done.exists():
        return {t:pd.read_csv(folder/f'{t}.csv',low_memory=False) for t in TABLES}
    # load_by_product는 savepath가 없으며 사전 캐시 확인 후 실행합니다.
    tables=nu.load_by_product(dpid='DP1.10098.001',site=[site],
        startdate=f'{year}-01',enddate=f'{year}-12',release=CFG['release'],
        package='expanded',check_size=False,token=TOKEN)
    result={t:choose_table(tables,t) for t in TABLES}
    for t,df in result.items(): df.to_csv(folder/f'{t}.csv',index=False)
    # 변수·품질·인용 정보도 보존
    for k,v in tables.items():
        if isinstance(v,pd.DataFrame) and k not in TABLES:
            v.to_csv(folder/(re.sub(r'[^\w.-]','_',k)+'.csv'),index=False)
    done.write_text(json.dumps({'site':site,'year':year,'keys':list(tables)}))
    return result

def prepare_field(tables, site, year):
    a=tables[TABLES[0]].copy(); m=tables[TABLES[1]].copy(); p=tables[TABLES[2]].copy()
    require_columns(a,['individualID','plotID','date','stemDiameter','height','plantStatus','subplotID'],TABLES[0])
    require_columns(m,['individualID','pointID','stemDistance','stemAzimuth','taxonID'],TABLES[1])
    require_columns(p,['plotID','date','totalSampledAreaTrees'],TABLES[2])
    a=a[pd.to_datetime(a.date,errors='coerce').dt.year.eq(year)].copy()
    p=p[pd.to_datetime(p.date,errors='coerce').dt.year.eq(year)].copy()
    if 'dataCollected' in p:
        p=p[p.dataCollected.fillna('').str.lower().eq('allgrowthforms')]
    if 'samplingImpractical' in p:
        p=p[p.samplingImpractical.isna() | p.samplingImpractical.str.lower().eq('ok')]
    p=p.sort_values('date').drop_duplicates('plotID',keep='last')
    # 데이터 수집이 완료된 가장 최근 event만 사용
    if 'eventID' in a and 'eventID' in p:
        a=a.merge(p[['plotID','eventID']].drop_duplicates(),on=['plotID','eventID'],how='inner')
    else:
        a=a[a.plotID.isin(p.plotID)]
    for c in ['stemDiameter','height','maxCrownDiameter','ninetyCrownDiameter']:
        if c not in a: a[c]=np.nan
        a[c]=pd.to_numeric(a[c],errors='coerce')
    for c in ['stemDistance','stemAzimuth']:
        m[c]=pd.to_numeric(m[c],errors='coerce')
    # mapping에 여러 이력이 있으면 관측연도까지의 가장 최근 행
    if 'date' in m:
        m['_mapping_date']=pd.to_datetime(m.date,errors='coerce')
        m=m[m._mapping_date.isna() | m._mapping_date.dt.year.le(year)]
        m=m.sort_values('_mapping_date',na_position='first')
    key=['individualID']
    if 'stemID' in m and 'stemID' in a: key.append('stemID')
    m=m.drop_duplicates(key,keep='last')
    # apparent 표의 정체성 필드는 보존하고 위치·분류 필드만 보강
    mergecols=key+[c for c in ['pointID','stemDistance','stemAzimuth','taxonID','namedLocation'] if c in m and c not in key]
    for c in mergecols:
        if c in a and c not in key and c not in ['taxonID','namedLocation']: a=a.drop(columns=c)
    a=a.merge(m[mergecols],on=key,how='left',suffixes=('','_map'),validate='many_to_one')
    for c in ['taxonID','namedLocation']:
        if c+'_map' in a: a[c]=a[c].fillna(a[c+'_map']) if c in a else a[c+'_map']
    a['siteID']=site; a['field_year']=year
    a['plotNamedLocation']=a['plotID']+'.basePlot.vst'
    a['field_CD_m']=(a.maxCrownDiameter+a.ninetyCrownDiameter)/2
    # footprint 산정용: 살아 있는 나무로 제한하기 전의 모든 조사 나무
    footprint_evidence=a.copy()
    live=a.plantStatus.fillna('').str.startswith('Live')
    treeform=a.get('growthForm',pd.Series('tree',index=a.index)).fillna('').str.contains('tree',case=False)
    a=a[live & treeform & a.stemDiameter.ge(CFG['min_field_dbh_cm'])].copy()
    rows=[]
    for individual,g in a.groupby('individualID'):
        # 큰 나무의 multi-bole은 ID별로 구분된다. 같은 ID를 다시 합산하지 않는다.
        g=g.sort_values('date')
        r=g.iloc[-1].copy()
        r['duplicate_measurements']=len(g)
        r['field_DBH_cm']=r['stemDiameter']
        r['field_H_m']=r['height']
        rows.append(r)
    return pd.DataFrame(rows),p,footprint_evidence

if not RESOLUTION_ONLY:
    TABLES=['vst_apparentindividual','vst_mappingandtagging','vst_perplotperyear']


    FIELD_PARTS=[]; PLOT_PARTS=[]; EVIDENCE_PARTS=[]
    for site in CFG['sites']:
        years=[CFG['year']]+(CFG['adjacent_priority'] if CFG['allow_adjacent_years'] else [])
        selected=set()
        for year in years:
            if year!=CFG['year'] and len(selected)>=CFG['min_plots_per_site']: break
            tables=load_field_year(site,year)
            f,p,e=prepare_field(tables,site,year)
            available=[x for x in p.plotID if x not in selected and x in set(f.plotID)]
            if year!=CFG['year']:
                available=available[:max(0,CFG['min_plots_per_site']-len(selected))]
            if CFG['max_plots_per_site'] is not None:
                available=available[:max(0,CFG['max_plots_per_site']-len(selected))]
            selected.update(available)
            FIELD_PARTS.append(f[f.plotID.isin(available)])
            PLOT_PARTS.append(p[p.plotID.isin(available)].assign(siteID=site,field_year=year))
            EVIDENCE_PARTS.append(e[e.plotID.isin(available)])
    field=pd.concat(FIELD_PARTS,ignore_index=True)
    plotmeta=pd.concat(PLOT_PARTS,ignore_index=True)
    evidence=pd.concat(EVIDENCE_PARTS,ignore_index=True)
    if field.empty: raise RuntimeError('해당 연도 실측 나무 없음. 인접연도 옵션/자료 가용성 확인')
    display(field.groupby(['siteID','field_year']).agg(trees=('individualID','size'),plots=('plotID','nunique')))

## 05. 기준점 좌표와 실제 조사 경계 복원

사용자 안내서 §3.1, §3.5, §6.1을 따릅니다.
동쪽 = 기준점 E + 거리×sin(방위각), 북쪽 = 기준점 N + 거리×cos(방위각).
큰 나무가 기록된 `100/400㎡ subplotID`의 API 중심점으로 정사각형 조사 범위를 합칩니다.
살아 있는 나무가 없어도 기록된 고사목을 경계 증거에 포함합니다. 면적은 perplot의
`totalSampledAreaTrees`와 검산합니다. **빈 소구획을 추측해서 확장하지 않습니다.**
모든 소구획을 복원하지 못하면 해당 조사구를 제외하고 진단 CSV에 남깁니다.
그 경우 정확한 조사 경계를 별도 파일로 제공하는 `footprint_override`를 사용할 수 있습니다.
경계의 소구획 모서리는 정북 기준이며 실제 경사면 면적 대신 수평 투영면적을 사용합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def offset_xy(e,n,distance,azimuth_deg):
    rad=np.deg2rad(azimuth_deg)
    return e+distance*np.sin(rad), n+distance*np.cos(rad)

def map_individuals(df):
    out=[]
    for _,r in df.iterrows():
        try:
            if not np.isfinite(r.stemDistance) or not np.isfinite(r.stemAzimuth) or pd.isna(r.pointID):
                raise ValueError('정확한 stem offset 없음')
            pid=str(r.pointID).removesuffix('.0')
            loc=cached_location(r.plotNamedLocation+'.'+pid)
            e,n,crs=location_xy(loc)
            x,y=offset_xy(e,n,r.stemDistance,r.stemAzimuth)
            rr=r.to_dict(); rr.update(x=x,y=y,source_epsg=crs.to_epsg())
            # 사이트마다 CRS가 달라도 공통 작업 CRS로 변환
            xx,yy=Transformer.from_crs(crs,WORK_CRS,always_xy=True).transform(x,y)
            rr['geometry']=Point(xx,yy)
            out.append(rr)
        except requests.HTTPError as exc:
            if exc.response.status_code in [401,403,429]: raise
            audit('coordinates',r.individualID,exc)
        except (ValueError,KeyError,TypeError) as exc: audit('coordinates',r.individualID,exc)
    if not out:
        pd.DataFrame(AUDIT).to_csv(OUT/'audit.csv',index=False)
        raise RuntimeError('매핑된 실측 나무가 없습니다. pointID·API 진단은 audit.csv 확인')
    return gpd.GeoDataFrame(out,geometry='geometry',crs=WORK_CRS)

def build_footprints(meta,evidence):
    plots=[]
    override=None
    if CFG['footprint_override']:
        override=gpd.read_file(CFG['footprint_override']).to_crs(WORK_CRS)
        require_columns(override,['plotID','field_year','geometry'],'footprint_override')
    for _,p in meta.iterrows():
        try:
            loc=cached_location(p.plotID+'.basePlot.vst')
            cx,cy,crs=location_xy(loc)
            ids=evidence.loc[(evidence.plotID==p.plotID)&(evidence.field_year==p.field_year)&
                (evidence.stemDiameter>=CFG['min_field_dbh_cm']),'subplotID'].dropna().unique()
            polygons=[]
            for sub in ids:
                match=re.fullmatch(r'(\d+)_(100|400)',str(sub))
                if not match: continue
                d=cached_location(p.plotID+'.basePlot.vst.'+str(sub))
                x,y,c=location_xy(d); half=np.sqrt(int(match.group(2)))/2
                geo=gpd.GeoSeries([box(x-half,y-half,x+half,y+half)],crs=c).to_crs(WORK_CRS).iloc[0]
                polygons.append(geo)
            expected=float(p.totalSampledAreaTrees)
            if override is not None:
                q=override[(override.plotID==p.plotID)&(override.field_year==p.field_year)]
                if not q.empty: polygons=list(q.geometry)
            if not polygons: raise ValueError('100/400㎡ 소구획 경계 증거 없음')
            geom=unary_union(polygons)
            if not np.isfinite(expected) or expected<=0 or abs(geom.area-expected)/expected>0.05:
                raise ValueError(f'조사 면적 불일치: polygon={geom.area:.1f}, totalSampledAreaTrees={expected}; 빈 소구획/조사 범위 확인')
            x,y=Transformer.from_crs(crs,WORK_CRS,always_xy=True).transform(cx,cy)
            plots.append(dict(plotID=p.plotID,siteID=p.siteID,field_year=int(p.field_year),
                area_m2=expected,reported_area_m2=expected,polygon_area_m2=geom.area,
                area_difference_m2=geom.area-expected,center_x=x,center_y=y,geometry=geom))
        except requests.HTTPError as exc:
            if exc.response.status_code in [401,403,429]: raise
            audit('footprints',p.plotID,exc)
        except (ValueError,KeyError,TypeError) as exc: audit('footprints',p.plotID,exc)
    pd.DataFrame(AUDIT).to_csv(OUT/'audit.csv',index=False)
    if not plots: raise RuntimeError('검증된 조사 경계 없음: audit.csv와 subplotID/API 확인')
    return gpd.GeoDataFrame(plots,geometry='geometry',crs=WORK_CRS)

if not RESOLUTION_ONLY:


    WORK_CRS=CRS.from_epsg(32617) # GRSM·ORNL; 한국에는 현지 미터 투영 CRS 사용
    field_geo=map_individuals(field)


    plots=build_footprints(plotmeta,evidence)
    requested_cohort=set(CFG['comparison_plot_ids'])
    missing_cohort=requested_cohort-set(plots.plotID)
    if missing_cohort: raise RuntimeError('기존 비교 조사구 경계 복원 실패: '+str(sorted(missing_cohort)))
    plots=plots[plots.plotID.isin(requested_cohort)].copy()
    if plots.empty: raise RuntimeError('검증된 조사 경계가 없습니다. AUDIT와 subplotID/API 확인 필요')
    eligible=[]
    for _,p in plots.iterrows():
        f=field_geo[(field_geo.plotID==p.plotID)&(field_geo.field_year==p.field_year)].copy()
        inside=f.geometry.apply(p.geometry.covers)
        for ind in f.loc[~inside,'individualID']: audit('outside_sampled_area',ind,p.plotID)
        eligible.append(f[inside])
    field_geo=pd.concat(eligible,ignore_index=True)
    field_geo=gpd.GeoDataFrame(field_geo,geometry='geometry',crs=WORK_CRS)
    plots.to_file(OUT/'sampled_footprints.gpkg',driver='GPKG')
    pd.DataFrame(AUDIT).to_csv(OUT/'audit.csv',index=False)
    display(plots.drop(columns='geometry'))

## 06. NEONForestAGB 참조값과 수종 계수

152MB 파일은 청크 단위로 읽습니다. Chojnacky 유효값을 우선하고 없으면 Jenkins를 선택합니다.
중복 개체는 날짜·공식 우선순위를 기록해 한 줄로 만듭니다. 참조값은 각 조사구의 선택된 실측 연도와 맞춥니다.
2020/2022 보완값은 2021년 값으로 바꾸어 부르지 않습니다.

Chojnacky에는 Jenkins의 “mixed hardwood”와 같은 단일 명칭의 계수가 없습니다.
따라서 기본은 **일반 활엽수 대체식** `exp(-2.2118 + 2.4133 ln(DBH_cm))`이며
Cor/Eri/Lau/Platanaceae/Rosaceae/Ulmaceae 그룹임을 명시합니다. 모든 활엽수에 검증된 혼합식이라는 뜻이 아닙니다.
실측 taxonID에 대응하는 β0·β1은 MasterTaxonList에서 읽고, 미대응 수종은 대체 여부를 표시합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
default=CFG['agb_default']

if not RESOLUTION_ONLY:
    AGB_URL='https://ndownloader.figshare.com/files/45710721'
    TAXON_URL='https://raw.githubusercontent.com/atkinsjeff/NEONForestAGB/main/data/MasterTaxonList.csv'
    agbp=cached_download(AGB_URL,ROOT/'data/raw/NEONForestAGBv2.csv')
    taxp=cached_download(TAXON_URL,ROOT/'data/raw/MasterTaxonList.csv')
    tax=pd.read_csv(taxp,encoding='cp1252')
    require_columns(tax,['PLANTS.Code','beta0','beta1','chojnacky'],'MasterTaxonList')
    for c in ['beta0','beta1']: tax[c]=pd.to_numeric(tax[c],errors='coerce')
    valid=tax.dropna(subset=['beta0','beta1'])
    conflicts=valid.groupby('PLANTS.Code')[['beta0','beta1']].nunique().max(axis=1)
    if conflicts.gt(1).any():
        bad_taxa=conflicts[conflicts.gt(1)].index.tolist()
        warnings.warn('Chojnacky 계수 충돌(QUDO 등): 경고 후 해당 taxon 계수 행 제외 '+str(bad_taxa))
        valid[valid['PLANTS.Code'].isin(bad_taxa)].to_csv(OUT/'excluded_taxon_coefficients.csv',index=False)
        valid=valid[~valid['PLANTS.Code'].isin(bad_taxa)].copy()
    taxlookup=valid.drop_duplicates('PLANTS.Code').set_index('PLANTS.Code')
    assert ((valid.beta0-default['beta0']).abs()<1e-8).any(), '기본 계수가 출처 표에 없음'
    parts=[]
    for chunk in pd.read_csv(agbp,chunksize=100000,low_memory=False):
        require_columns(chunk,['individualID','siteID','date','plantStatus','allometry','AGB'],'NEONForestAGB')
        chunk['field_year']=pd.to_datetime(chunk.date,errors='coerce').dt.year
        z=chunk[chunk.siteID.isin(CFG['sites']) & chunk.field_year.isin(plots.field_year.unique()) &
            chunk.plantStatus.fillna('').str.startswith('Live') &
            chunk.allometry.isin(['AGBChojnacky','AGBJenkins'])].copy()
        z['AGB']=pd.to_numeric(z.AGB,errors='coerce')
        parts.append(z[z.AGB.gt(0)])
    ref=pd.concat(parts,ignore_index=True)
    ref['_priority']=ref.allometry.map({'AGBChojnacky':0,'AGBJenkins':1})
    ref=ref.sort_values(['individualID','field_year','_priority','date'],ascending=[True,True,True,False])
    ref=ref.drop_duplicates(['individualID','field_year'])
    ref=ref[['individualID','field_year','AGB','allometry','date']].rename(columns={'AGB':'ref_AGB_kg','date':'ref_date'})
    field_geo=field_geo.merge(ref,on=['individualID','field_year'],how='left',validate='one_to_one')
    field_all=field.merge(ref,on=['individualID','field_year'],how='left',validate='one_to_one')
    field_all=field_all[field_all.plotID.isin(plots.plotID)].copy()
    field_all['ref_fallback_jenkins']=field_all.allometry.eq('AGBJenkins')
    field_all['has_matching_coordinate']=field_all.individualID.isin(field_geo.individualID)
    field_geo['ref_fallback_jenkins']=field_geo.allometry.eq('AGBJenkins')
    for ind in field_all.loc[field_all.ref_AGB_kg.isna(),'individualID']: audit('missing_AGB',ind,'참조 AGB 없음; 총량 비교에서 완전성 확인')
    field_geo['species_beta0']=field_geo.taxonID.map(taxlookup.beta0)
    field_geo['species_beta1']=field_geo.taxonID.map(taxlookup.beta1)
    field_geo['species_group']=field_geo.taxonID.map(taxlookup.chojnacky)
    field_geo['species_coeff_missing']=field_geo.species_beta0.isna()|field_geo.species_beta1.isna()
    field_geo['species_beta0']=field_geo.species_beta0.fillna(default['beta0'])
    field_geo['species_beta1']=field_geo.species_beta1.fillna(default['beta1'])
    print('참조값 coverage:',field_geo.ref_AGB_kg.notna().mean())
    print('Jenkins 대체:',field_geo.ref_fallback_jenkins.sum())

## 07. 공식·단위 검산

H·CD는 m, DBH는 cm, AGB는 kg, 면적은 ㎡입니다.
Jucker/itcSegment biome=11: DBH=0.367(H×CD)^0.887×exp(0²/2).
로그는 자연로그입니다. 수관 지름은 사용자가 요청한 상자 가로·세로 평균을 사용합니다.
이는 Jucker 원자료의 실제 수관 지름과 기하학적으로 다를 수 있습니다.
출처 원문과 SHA256을 보존해 추후 계수 변경을 추적합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def estimate_dbh(H_m, CD_m, coefficients=None):
    c=coefficients or CFG['jucker']
    H=np.asarray(H_m,dtype=float); CD=np.asarray(CD_m,dtype=float)
    return np.where((H>0)&(CD>0),c['a']*(H*CD)**c['b']*np.exp(c.get('g',0)**2/2),np.nan)

def estimate_agb(dbh_cm,beta0=None,beta1=None):
    b0=default['beta0'] if beta0 is None else beta0
    b1=default['beta1'] if beta1 is None else beta1
    D=np.asarray(dbh_cm,dtype=float)
    with np.errstate(invalid='ignore',divide='ignore'):
        return np.where(D>0,np.exp(b0+b1*np.log(D)),np.nan)

def kg_to_t_ha(kg,area_m2): return np.asarray(kg)/1000*10000/np.asarray(area_m2)

def carbon_kg(agb_kg,fraction=0.5): return np.asarray(agb_kg)*fraction

if not RESOLUTION_ONLY:
    JUCKER_SOURCE='https://raw.githubusercontent.com/cran/itcSegment/master/R/dbh.R'
    jsp=cached_download(JUCKER_SOURCE,ROOT/'data/raw/itcSegment_dbh.R')
    source=jsp.read_text()
    assert re.search(r'11\s*,\s*0\.367\s*,\s*0\.887\s*,\s*0',source), 'Jucker 출처 계수 변경: 검토 필요'




    assert np.isclose(kg_to_t_ha(1000,10000),1)
    assert np.isclose(kg_to_t_ha(100,400),2.5)
    assert np.isclose(carbon_kg(100,0.5),50)
    assert np.isclose(carbon_kg(100,0.47),47)
    assert np.allclose(offset_xy(100,200,10,90),(110,200))
    assert np.allclose(offset_xy(100,200,10,0),(100,210))
    assert np.isclose(estimate_dbh([20],[5])[0],0.367*100**0.887)
    assert np.isclose(estimate_agb([10])[0],np.exp(-2.2118+2.4133*np.log(10)))
    provenance={str(p.name):hashlib.sha256(p.read_bytes()).hexdigest() for p in [taxp,jsp]}
    (OUT/'formula_provenance.json').write_text(json.dumps(dict(urls=[JUCKER_SOURCE,TAXON_URL,AGB_URL],sha256=provenance),indent=2))
    print('단위·방위각·공식 검산 통과')

## 08. AOP 제공 연월 조회·영상 연도 선택·타일 다운로드

공식 products API의 사이트별 availableMonths와 availableReleases를 출력합니다.
ORNL의 2021-06 RGB/CHM은 PROVISIONAL이므로 `include_provisional=True`가 필요합니다.
2021년 공통 자료가 실제로 없을 때만 ORNL에 한해 2022→2020 순서로 영상 연도를 선택합니다.
실측 연도는 바꾸지 않습니다. ORNL은 UTM 16N, 작업 좌표는 UTM 17N이므로 다운로드 좌표를 변환합니다.
다운로드 후 TIFF 파일 수·확장자·실제 저장 폴더를 출력하고, 다음 셀에서 조사구별 커버 여부를 확인합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def tile_coordinates(centers,buffer_m=50):
    tiles=set()
    for x,y in centers:
        for e in range(int((x-buffer_m)//1000),int((x+buffer_m)//1000)+1):
            for n in range(int((y-buffer_m)//1000),int((y+buffer_m)//1000)+1): tiles.add((e*1000,n*1000))
    return sorted(tiles)

def product_site_entry(product,site):
    records=product.get('data',product).get('siteCodes',[])
    return next((r for r in records if r.get('siteCode')==site),dict(siteCode=site,availableMonths=[],availableReleases=[]))

def select_aop_pair(rgb_entry,chm_entry,target_year,allow_adjacent=False,priority=(2022,2020),include_provisional=True):
    common=sorted(set(rgb_entry.get('availableMonths',[])) & set(chm_entry.get('availableMonths',[])))
    years=[int(target_year)]+([int(y) for y in priority] if allow_adjacent else [])
    selected=next((y for y in years if any(m.startswith(str(y)+'-') for m in common)),None)
    if selected is None:
        raise ValueError(f'RGB/CHM 공통 연월 없음: 요청 연도 {years}; 공통 월 {common}')
    months=[m for m in common if m.startswith(str(selected)+'-')]
    provisional=False
    for entry in [rgb_entry,chm_entry]:
        stable={m for r in entry.get('availableReleases',[]) if r.get('release')!='PROVISIONAL' for m in r.get('availableMonths',[])}
        prov={m for r in entry.get('availableReleases',[]) if r.get('release')=='PROVISIONAL' for m in r.get('availableMonths',[])}
        provisional |= any(m in prov and m not in stable for m in months)
    if provisional and not include_provisional:
        raise ValueError('선택 연도에 PROVISIONAL 자료가 있습니다. aop_include_provisional=True 필요; 다른 해로 숨겨 대체하지 않습니다.')
    return dict(imagery_year=selected,common_months=months,include_provisional=bool(provisional),
                provisional=bool(provisional),adjacent_imagery=selected!=int(target_year))

def fetch_product_metadata(dpid):
    path=ROOT/'data/cache'/f'product_{dpid}.json'
    if path.exists() and not CFG.get('api_refresh',True): return json.loads(path.read_text())
    url='https://data.neonscience.org/api/v0/products/'+dpid
    r=session.get(url,headers={'X-API-Token':TOKEN},timeout=90); r.raise_for_status()
    payload=r.json(); path.write_text(json.dumps(payload,ensure_ascii=False))
    return payload

def aop_tiff_files(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in ['.tif','.tiff'])

if not RESOLUTION_ONLY:





    PRODUCTS=['DP3.30010.001','DP3.30015.001']
    product_metadata={product:fetch_product_metadata(product) for product in PRODUCTS}
    AOP_SELECTION={}; AOP_DIRS={}; availability_rows=[]; inventory_rows=[]
    for site,g in plots.groupby('siteID'):
        entries={product:product_site_entry(product_metadata[product],site) for product in PRODUCTS}
        for product,entry in entries.items():
            print(site,product,'실제 제공 연월:',entry.get('availableMonths',[]))
            print('  release별:',entry.get('availableReleases',[]))
            for month in entry.get('availableMonths',[]):
                releases=[r['release'] for r in entry.get('availableReleases',[]) if month in r.get('availableMonths',[])]
                availability_rows.append(dict(siteID=site,product=product,month=month,releases=';'.join(releases)))
        selection=select_aop_pair(entries[PRODUCTS[0]],entries[PRODUCTS[1]],CFG['year'],
            CFG['imagery_allow_adjacent_years'].get(site,False),CFG['imagery_adjacent_priority'],CFG['aop_include_provisional'])
        AOP_SELECTION[site]=selection
        print(site,'영상 선택:',selection)
        if all(p in CFG['local_rgb'] and p in CFG['local_chm'] for p in g.plotID):
            print(site,'사용자 입력 RGB/CHM 사용: 다운로드 생략')
            continue
        loc=cached_location(g.iloc[0].plotID+'.basePlot.vst')
        _,_,site_crs=location_xy(loc)
        transform=Transformer.from_crs(WORK_CRS,site_crs,always_xy=True)
        centers=[transform.transform(r.center_x,r.center_y) for _,r in g.iterrows()]
        nt=len(tile_coordinates(centers,CFG['buffer_m']))
        print(site,'다운로드 CRS:',site_crs.to_string(),'첫 중심 좌표:',centers[0])
        print(site,'요청 예상 1km 타일:',nt,'RGB 압축 전 대략',round(nt*0.30,2),'GB')
        for product in PRODUCTS:
            legacy=ROOT/'data/raw/aop'/site
            # 기존 2021 캐시를 재사용. 인접연도는 별도 폴더에 격리.
            dest=legacy if selection['imagery_year']==CFG['year'] and (legacy/product).exists() else legacy/str(selection['imagery_year'])
            dest.mkdir(parents=True,exist_ok=True)
            args=dict(dpid=product,site=site,year=selection['imagery_year'],easting=[c[0] for c in centers],
                northing=[c[1] for c in centers],buffer=CFG['buffer_m'],check_size=False,
                savepath=str(dest),token=TOKEN,skip_if_exists=True,overwrite='yes',
                include_provisional=selection['include_provisional'],verbose=True)
            if 'skip_if_exists' not in inspect.signature(nu.by_tile_aop).parameters:
                raise RuntimeError('neonutilities를 업데이트하세요: 안전한 다운로드 재사용 API 없음')
            nu.by_tile_aop(**args)
            product_folder=dest/product
            # 확장자는 대소문자 구분 없이 .tif와 .tiff 모두 검색. 제품 디렉터리는 재귀적으로 검사.
            paths=aop_tiff_files(product_folder)
            if not paths:
                candidates=[q for q in dest.rglob('*') if q.is_dir() and q.name==product]
                if len(candidates)==1: product_folder=candidates[0]; paths=aop_tiff_files(product_folder)
            AOP_DIRS[(site,product)]=product_folder
            extensions=sorted({p.suffix for p in paths})
            print(site,product,'다운로드 후 TIFF 타일 수:',len(paths),'확장자:',extensions,'폴더:',product_folder)
            inventory_rows.append(dict(siteID=site,product=product,imagery_year=selection['imagery_year'],
                site_crs=site_crs.to_string(),expected_tiles=nt,tiff_tiles=len(paths),extensions=';'.join(extensions),folder=str(product_folder)))
            if not paths:
                print('폴더 진단:',[str(p.relative_to(dest)) for p in list(dest.rglob('*'))[:20]])
                raise RuntimeError(f'{site} {product}: 다운로드 후 TIFF 0개. 제공 월·PROVISIONAL·좌표·by_tile_aop 로그를 확인하세요.')
    pd.DataFrame(availability_rows).to_csv(OUT/'aop_availability.csv',index=False)
    pd.DataFrame(inventory_rows).to_csv(OUT/'aop_download_inventory.csv',index=False)
    (OUT/'aop_selection.json').write_text(json.dumps(AOP_SELECTION,ensure_ascii=False,indent=2))
    plots['imagery_year']=plots.siteID.map(lambda s:AOP_SELECTION[s]['imagery_year'])
    plots['imagery_provisional']=plots.siteID.map(lambda s:AOP_SELECTION[s]['provisional'])

## 09. 타일 색인·조사구 주변 잘라내기

1km RGB 전체를 한 번에 메모리에 읽지 않습니다.
조사구 주변 창만 mosaic하며, CRS·해상도·nodata를 검사합니다. 조사구 경계를 완전히 덮지 못하는 자료는
평가하지 않고 사유를 저장합니다. 여러 촬영일 자료가 겹치면 중복 합성을 피하기 위해 촬영 폴더를 먼저 확인하세요.
RGB·CHM 파일 경로와 CRS는 manifest에 남깁니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def raster_index(folder,expected_resolution):
    rows=[]
    for path in sorted(Path(folder).rglob('*')):
        if path.suffix.lower() not in ['.tif','.tiff']: continue
        with rasterio.open(path) as ds:
            if not ds.crs or not ds.crs.is_projected: raise ValueError(f'미터 투영 CRS 필요: {path}')
            if not np.allclose(ds.res,expected_resolution,atol=expected_resolution*0.05):
                audit('resolution',path,f'예상 {expected_resolution}; 실제 {ds.res}'); continue
            rows.append(dict(path=str(path),crs=ds.crs,geometry=box(*ds.bounds)))
    return rows

def crop_mosaic(index,region,region_crs,dest,bands):
    if dest.exists(): return dest
    datasets=[]; coverage=[]
    try:
        for r in index:
            geom=gpd.GeoSeries([region],crs=region_crs).to_crs(r['crs']).iloc[0]
            if r['geometry'].intersects(geom):
                datasets.append(rasterio.open(r['path'])); coverage.append(r['geometry'])
        if not datasets: raise ValueError('해당 위치를 덮는 타일 없음')
        crs=datasets[0].crs
        if any(d.crs!=crs for d in datasets): raise ValueError('mosaic 입력 CRS 불일치')
        geom=gpd.GeoSeries([region],crs=region_crs).to_crs(crs).iloc[0]
        if not unary_union(coverage).buffer(0.2).covers(geom): raise ValueError('조사구 주변 영상 coverage 불완전')
        data,trans=merge(datasets,bounds=geom.bounds,indexes=bands)
        profile=datasets[0].profile.copy()
        profile.update(driver='GTiff',height=data.shape[1],width=data.shape[2],count=len(bands),
            transform=trans,compress='deflate',tiled=True,blockxsize=256,blockysize=256)
        with rasterio.open(dest,'w',**profile) as out: out.write(data)
    finally:
        for d in datasets: d.close()
    return dest

def raster_covers_geometry(index,geometry,geometry_crs):
    if not index: return False
    # 타일 CRS가 여러 개일 수 있으므로 모든 경계를 작업 CRS로 옮겨 합칩니다.
    bounds=[gpd.GeoSeries([r['geometry']],crs=r['crs']).to_crs(geometry_crs).iloc[0] for r in index]
    return unary_union(bounds).buffer(0.2).covers(geometry)

if not RESOLUTION_ONLY:



    RASTERS={}; MANIFEST=[]; coverage_rows=[]
    for site,g in plots.groupby('siteID'):
        rgb_idx=raster_index(AOP_DIRS.get((site,'DP3.30010.001'),ROOT/'data/raw/aop'/site/'DP3.30010.001'),0.1)
        chm_idx=raster_index(AOP_DIRS.get((site,'DP3.30015.001'),ROOT/'data/raw/aop'/site/'DP3.30015.001'),1.0)
        print(site,'해상도 검사 후 RGB/CHM 타일 수:',len(rgb_idx),len(chm_idx))
        for _,p in g.iterrows():
            key=p.plotID
            status=dict(siteID=site,plotID=key,imagery_year=p.imagery_year,field_year=p.field_year,
                rgb_sampled_coverage=False,chm_sampled_coverage=False,usable=False,reason='')
            try:
                if key in CFG['local_rgb'] and key in CFG['local_chm']:
                    rgb=Path(CFG['local_rgb'][key]); chm=Path(CFG['local_chm'][key])
                    with rasterio.open(rgb) as ds:
                        if ds.dtypes[0]!='uint8' or ds.count<3: raise ValueError('사용자 RGB는 uint8 3밴드 필요')
                    RASTERS[key]=(rgb,chm)
                    MANIFEST.append(dict(plotID=key,rgb=str(rgb),chm=str(chm),field_year=p.field_year,
                        imagery_year=p.imagery_year,imagery_provisional=p.imagery_provisional))
                    status.update(usable=True,reason='사용자 입력 파일')
                    coverage_rows.append(status)
                    continue
                status['rgb_sampled_coverage']=raster_covers_geometry(rgb_idx,p.geometry,WORK_CRS)
                status['chm_sampled_coverage']=raster_covers_geometry(chm_idx,p.geometry,WORK_CRS)
                if not rgb_idx or not chm_idx: raise ValueError(f'{site}: {p.imagery_year} RGB/CHM TIFF 없음. AOP_DIRS 확인')
                region=box(p.center_x-CFG['buffer_m'],p.center_y-CFG['buffer_m'],
                           p.center_x+CFG['buffer_m'],p.center_y+CFG['buffer_m'])
                rgb=crop_mosaic(rgb_idx,region,WORK_CRS,ROOT/'data/crops'/f'{key}_{p.imagery_year}_rgb_{RUN_ID}.tif',[1,2,3])
                chm=crop_mosaic(chm_idx,region,WORK_CRS,ROOT/'data/crops'/f'{key}_{p.imagery_year}_chm_{RUN_ID}.tif',[1])
                with rasterio.open(rgb) as ds:
                    if ds.dtypes[0]!='uint8': raise ValueError('RGB는 uint8 입력 필요; 스케일링을 명시하세요')
                RASTERS[key]=(rgb,chm)
                MANIFEST.append(dict(plotID=key,rgb=str(rgb),chm=str(chm),field_year=p.field_year,
                    imagery_year=p.imagery_year,imagery_provisional=p.imagery_provisional))
                status['usable']=True
            except ValueError as exc:
                audit('raster_coverage',key,exc); status['reason']=str(exc)
            coverage_rows.append(status)
    pd.DataFrame(MANIFEST).to_csv(OUT/'raster_manifest.csv',index=False)
    coverage=pd.DataFrame(coverage_rows)
    coverage.to_csv(OUT/'aop_plot_coverage.csv',index=False)
    display(coverage)
    print('사이트별 분석 가능 조사구:'); display(coverage.groupby('siteID').usable.agg(['sum','count']))
    pd.DataFrame(AUDIT).to_csv(OUT/'audit.csv',index=False)
    for site in CFG['sites']:
        if not coverage.loc[coverage.siteID==site,'usable'].any():
            raise RuntimeError(f'{site} 분석 가능 조사구 0개. aop_plot_coverage.csv 확인 후 다운로드/좌표 문제를 해결하세요.')
    plots=plots[plots.plotID.isin(RASTERS)].copy()
    field_geo=field_geo[field_geo.plotID.isin(RASTERS)].copy()
    field_all=field_all[field_all.plotID.isin(RASTERS)].copy()
    if plots.empty: raise RuntimeError('영상 coverage를 만족하는 조사구가 없습니다')

## 10. DeepForest 사전학습 모델과 버전 호환

설치된 버전과 실제 메서드 존재 여부를 확인합니다.
새 버전은 `load_model`, 이전 버전은 `use_release`를 사용합니다.
네트워크·가중치 다운로드 오류를 구버전 오류로 오해해 숨기지 않습니다. 추가 학습은 하지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
from deepforest import main

def load_detector():
    version=importlib.metadata.version('deepforest')
    m=main.deepforest()
    if callable(getattr(m,'load_model',None)):
        m.load_model(model_name=CFG['model_name'],revision=CFG['model_revision'])
        method='load_model'
    elif callable(getattr(m,'use_release',None)):
        m.use_release(); method='use_release'
    else: raise RuntimeError(f'DeepForest {version}: 지원 모델 로더 없음')
    print('DeepForest',version,'로더',method,'predict_tile',inspect.signature(m.predict_tile))
    (OUT/'model.json').write_text(json.dumps(dict(version=version,loader=method,
        model=CFG['model_name'],revision=CFG['model_revision']),indent=2))
    return m

if not RESOLUTION_ONLY:
    model=load_detector()

## 11. 탐지·CHM 수고·수관 지름 추출

RGB 픽셀 좌표를 raster transform으로 지도 좌표로 바꿉니다.
수고는 상자 내부 CHM 최대값이고, 수관 지름은 상자 가로·세로 평균입니다.
CHM nodata/음수는 제외하며 유효 CHM이 없는 탐지는 따로 기록합니다.
조사구 안/밖 판정은 탐지 상자의 중심점 기준입니다. 실측은 줄기 위치 기준이므로 경계 편향이 남습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def zonal_max(path, geom, geom_crs):
    with rasterio.open(path) as ds:
        g=gpd.GeoSeries([geom],crs=geom_crs).to_crs(ds.crs).iloc[0]
        try: arr,_=mask(ds,[g],crop=True,filled=False,indexes=1)
        except ValueError: return np.nan
        v=arr.compressed(); v=v[np.isfinite(v)&(v>=0)]
        return float(v.max()) if len(v) else np.nan

def predict_crowns(m,rgb,chm,plot):
    cache=OUT/f'{plot.plotID}_raw_predictions.csv'
    if cache.exists(): raw=pd.read_csv(cache)
    else:
        params=inspect.signature(m.predict_tile).parameters
        path_key='path' if 'path' in params else 'raster_path'
        args={path_key:str(rgb),'patch_size':CFG['patch_size'],'patch_overlap':CFG['patch_overlap']}
        if 'iou_threshold' in params: args['iou_threshold']=CFG['nms_iou']
        raw=m.predict_tile(**args)
        if isinstance(raw,tuple): raw=raw[0]
        if raw is None or len(raw)==0:
            raw=pd.DataFrame(columns=['xmin','ymin','xmax','ymax','score'])
        raw.to_csv(cache,index=False)
    require_columns(raw,['xmin','ymin','xmax','ymax','score'],'DeepForest predictions')
    rows=[]
    with rasterio.open(rgb) as ds:
        trans=ds.transform; crs=ds.crs
        for idx,r in raw[raw.score>=CFG['score_threshold']].iterrows():
            corners=[trans*(x,y) for x,y in [(r.xmin,r.ymin),(r.xmax,r.ymin),(r.xmax,r.ymax),(r.xmin,r.ymax)]]
            geom=Polygon(corners)
            geom=gpd.GeoSeries([geom],crs=crs).to_crs(WORK_CRS).iloc[0]
            if not plot.geometry.covers(geom.centroid): continue
            h=zonal_max(chm,geom,WORK_CRS)
            if not np.isfinite(h): audit('CHM_nodata',f'{plot.plotID}_{idx}','CHM 없음'); continue
            if h<CFG['min_height_m']: continue
            width=(r.xmax-r.xmin)*abs(trans.a)
            length=(r.ymax-r.ymin)*abs(trans.e)
            rows.append(dict(det_id=f'{plot.plotID}_{idx}',plotID=plot.plotID,siteID=plot.siteID,
                field_year=plot.field_year,score=r.score,xmin_px=r.xmin,ymin_px=r.ymin,
                xmax_px=r.xmax,ymax_px=r.ymax,H_m=h,CD_m=(width+length)/2,
                width_m=width,length_m=length,geometry=geom))
    return rows

DET_COLUMNS=['det_id','plotID','siteID','field_year','score','xmin_px','ymin_px','xmax_px','ymax_px','H_m','CD_m','width_m','length_m','geometry']

if not RESOLUTION_ONLY:


    records=[]
    for _,p in plots.iterrows():
        print('탐지:',p.plotID)
        records.extend(predict_crowns(model,*RASTERS[p.plotID],p))
    detections=gpd.GeoDataFrame(records,columns=DET_COLUMNS,geometry='geometry',crs=WORK_CRS)
    detections['imagery_year']=detections.siteID.map(lambda s:AOP_SELECTION[s]['imagery_year'])
    detections['DBH_jucker_cm']=estimate_dbh(detections.H_m,detections.CD_m)
    detections['AGB_default_kg']=estimate_agb(detections.DBH_jucker_cm)
    detections['C_default_kg']=carbon_kg(detections.AGB_default_kg,CFG['carbon_fraction'])
    print('평가 영역 내 탐지:',len(detections))

## 12. 보이는 나무 정의와 1:1 매칭

실측 수관위치가 Full sun / Open grown / Partially shaded 이거나,
실측 수고가 주변 반경 3m CHM 최대값의 80% 이상이면 “보이는 나무”입니다.
이 정의는 관측 가능성의 근사이며 높이 검증 표본 선택 편향을 일으킬 수 있어 수관위치만 쓴 플래그도 보존합니다.

헝가리안 매칭의 후보는 **줄기 점이 상자 안에 있는 조합**입니다.
먼저 가능한 매칭 수를 최대화하고, 그 안에서 큰 실측 DBH를 우선하며 중심거리는 동률 해소에만 씁니다.
보이는 나무/전체 나무 두 모집단을 각각 매칭합니다. 전체 나무 기준은 선택된 공통 비교 범위(DBH≥10cm)의 전체입니다.
점 정답은 실제 수관 IoU 평가와 같지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def label_visibility(f,rasters):
    f=f.copy(); local=[]
    for _,r in f.iterrows():
        local.append(zonal_max(rasters[r.plotID][1],r.geometry.buffer(CFG['visible_radius_m']),WORK_CRS))
    f['local_CHM_m']=local
    canopy=f.get('canopyPosition',pd.Series('',index=f.index)).fillna('').str.strip().str.lower()
    f['visible_by_position']=canopy.isin(['full sun','open grown','partially shaded'])
    f['visible_by_height']=f.field_H_m.ge(CFG['visible_height_ratio']*f.local_CHM_m)&f.local_CHM_m.gt(0)
    f['visible']=f.visible_by_position|f.visible_by_height
    return f

def match_one_to_one(det,truth):
    if det.empty or truth.empty: return pd.DataFrame(columns=['det_id','individualID','plotID','distance_m'])
    d=det.reset_index(drop=True); t=truth.reset_index(drop=True)
    n,m=len(d),len(t)
    weights=np.zeros((n,m+n)) # dummy columns = unmatched
    valid=np.zeros((n,m),dtype=bool)
    sizes=pd.to_numeric(t[CFG['match_size']],errors='coerce').fillna(0).to_numpy()
    rank=pd.Series(sizes).rank(method='dense').to_numpy()/(m+1)
    for i,r in d.iterrows():
        for j,s in t.iterrows():
            if r.geometry.covers(s.geometry):
                valid[i,j]=True
                distance=r.geometry.centroid.distance(s.geometry)
                weights[i,j]=(n+1)*10 + rank[j] + 1e-6/(1+distance)
    ri,ci=linear_sum_assignment(-weights)
    rows=[]
    for i,j in zip(ri,ci):
        if j<m and valid[i,j]:
            rows.append(dict(det_id=d.iloc[i].det_id,individualID=t.iloc[j].individualID,
                plotID=d.iloc[i].plotID,distance_m=d.iloc[i].geometry.centroid.distance(t.iloc[j].geometry)))
    return pd.DataFrame(rows,columns=['det_id','individualID','plotID','distance_m'])

if not RESOLUTION_ONLY:


    field_geo=label_visibility(field_geo,RASTERS)
    MATCHES={}; detmetrics=[]
    for population in ['visible','all']:
        items=[]
        for _,p in plots.iterrows():
            d=detections[detections.plotID==p.plotID]
            t=field_geo[field_geo.plotID==p.plotID]
            if population=='visible': t=t[t.visible]
            pairs=match_one_to_one(d,t); items.append(pairs)
            tp=len(pairs); precision=tp/len(d) if len(d) else np.nan
            recall=tp/len(t) if len(t) else np.nan
            f1=2*tp/(len(d)+len(t)) if len(d)+len(t) else np.nan
            detmetrics.append(dict(plotID=p.plotID,siteID=p.siteID,field_year=p.field_year,
                population=population,TP=tp,FP=len(d)-tp,FN=len(t)-tp,
                n_detections=len(d),n_truth=len(t),precision=precision,recall=recall,F1=f1))
        MATCHES[population]=pd.concat(items,ignore_index=True)
    detmetrics=pd.DataFrame(detmetrics)
    pairs=MATCHES['visible']
    matched=pairs.merge(detections.drop(columns='geometry'),on=['det_id','plotID'],validate='one_to_one')
    matched=matched.merge(field_geo.drop(columns=['geometry','plotID','siteID','field_year']),on='individualID',validate='one_to_one')
    display(detmetrics)

## 13. 조사구 분리 DBH 회귀와 수종 민감도

log(DBH) ~ log(H) + log(CD)를 적합합니다. GroupKFold의 그룹은 조사구입니다.
동일 조사구 나무가 학습과 평가에 동시에 들어가지 않습니다. 로그 역변환의 Duan smearing도 학습 fold에서만 계산합니다.
공식 비교용 회귀값은 out-of-fold 값만 사용합니다. 전체 표본 적합 모델은 재사용 목적으로 별도 저장하며 평가값으로 쓰지 않습니다.
유효 조사구가 2개 미만이면 회귀 지표는 계산하지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def grouped_dbh_regression(df,n_splits=5):
    result=pd.Series(np.nan,index=df.index); fold_id=pd.Series(np.nan,index=df.index)
    good=df[['H_m','CD_m','field_DBH_cm']].gt(0).all(axis=1)
    good &= np.isfinite(df[['H_m','CD_m','field_DBH_cm']]).all(axis=1)
    z=df.loc[good]; groups=z.plotID
    if groups.nunique()<2:
        audit('regression','GroupKFold','독립 조사구 2개 미만'); return result,fold_id,None
    X=np.log(z[['H_m','CD_m']].to_numpy()); y=np.log(z.field_DBH_cm.to_numpy())
    folds=GroupKFold(n_splits=min(n_splits,groups.nunique()))
    for fold,(tr,te) in enumerate(folds.split(X,y,groups)):
        if len(tr)<4:
            audit('regression',fold,'학습 나무 4개 미만'); continue
        m=LinearRegression().fit(X[tr],y[tr])
        smear=float(np.mean(np.exp(y[tr]-m.predict(X[tr]))))
        result.loc[z.index[te]]=np.exp(m.predict(X[te]))*smear
        fold_id.loc[z.index[te]]=fold
    m=LinearRegression().fit(X,y)
    fit=dict(intercept=float(m.intercept_),log_H=float(m.coef_[0]),log_CD=float(m.coef_[1]),
             smearing=float(np.mean(np.exp(y-m.predict(X)))),n=len(z),n_plots=groups.nunique())
    return result,fold_id,fit

if not RESOLUTION_ONLY:

    if not matched.empty:
        matched['DBH_regression_oof_cm'],matched['cv_fold'],final_fit=grouped_dbh_regression(matched,CFG['cv_folds'])
        matched['AGB_regression_oof_kg']=estimate_agb(matched.DBH_regression_oof_cm)
        matched['AGB_known_species_kg']=estimate_agb(matched.DBH_jucker_cm,matched.species_beta0,matched.species_beta1)
        matched['species_effect_kg']=matched.AGB_known_species_kg-matched.AGB_default_kg
        matched['species_effect_pct']=100*(matched.AGB_known_species_kg/matched.AGB_default_kg-1)
        if final_fit: (OUT/'dbh_regression_full_fit.json').write_text(json.dumps(final_fit,indent=2))
    else:
        for c in ['DBH_regression_oof_cm','AGB_regression_oof_kg','AGB_known_species_kg','species_effect_kg','species_effect_pct']:
            matched[c]=pd.Series(dtype=float)
        audit('matching','visible','매칭 0건: 회귀·개체 오차표 계산 불가')

## 14. 단계별 오차와 조사구 총량·누락 분해

Bias=추정−참조입니다. 상관계수는 Pearson r이며 표본 수와 유효 조사구 수를 함께 봅니다.
수관 지름은 실측 두 축 평균과 비교합니다. 수관 측정이 없는 행은 그 단계에서만 제외합니다.

조사구 참조 총량은 좌표 필터 전 전체 살아 있는 DBH≥10cm 나무를 합칩니다.
좌표 누락은 총량을 NaN으로 만들지 않습니다. 참조 AGB 자체가 없는 나무가 있을 때만 NaN 처리합니다.
분모는 totalSampledAreaTrees를 우선하며 폴리곤 면적도 함께 기록합니다.
차이는 `매칭 나무 추정오차 + 미매칭 탐지량 − 매칭되지 않은 전체 실측량`으로 분해합니다.
매칭 불가 좌표 누락과 실제 탐지 누락을 혼동하지 않도록 별도 열로 나눕니다.
영상 연도와 실측 연도가 다르면 사이트·연도 조합별로 분리합니다.
수종 민감도는 같은 매칭 나무 표본끼리만 비교합니다. 미매칭 탐지의 수종을 아는 척하지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def errors(truth,pred):
    a=np.asarray(truth,dtype=float); b=np.asarray(pred,dtype=float)
    ok=np.isfinite(a)&np.isfinite(b); a=a[ok]; b=b[ok]
    if not len(a): return dict(n=0,MAE=np.nan,RMSE=np.nan,Bias=np.nan,r=np.nan)
    e=b-a
    r=np.corrcoef(a,b)[0,1] if len(a)>1 and np.std(a)>0 and np.std(b)>0 else np.nan
    return dict(n=len(a),MAE=float(np.mean(abs(e))),RMSE=float(np.sqrt(np.mean(e**2))),Bias=float(np.mean(e)),r=r)

def plot_biomass_totals(detections_all,truth_all,truth_coord,pairs,plot,carbon_fraction=0.5,carbon_alternative=0.47):
    # 일반 list[dict] 입력으로 동작: 좌표/지리 패키지 없이도 총량 계산을 재사용 가능.
    def number(value):
        try: return float(value)
        except (ValueError,TypeError): return float('nan')
    def sum_valid(rows,column):
        values=[number(r.get(column)) for r in rows]
        return sum(values) if all(math.isfinite(v) for v in values) else float('nan')
    area=number(plot.get('reported_area_m2'))
    if not math.isfinite(area) or area<=0: area=number(plot.get('area_m2'))
    if not math.isfinite(area) or area<=0: raise ValueError('양수 조사면적 필요')
    scale=lambda value:value/1000*10000/area
    by_id={r['individualID']:r for r in truth_all}
    if len(by_id)!=len(truth_all): raise ValueError('조사구 참조 개체 중복')
    det_by_id={r['det_id']:r for r in detections_all}
    paired_truth={r['individualID'] for r in pairs}
    paired_det={r['det_id'] for r in pairs}
    coord_ids={r['individualID'] for r in truth_coord}
    if not paired_truth<=by_id.keys() or not paired_det<=det_by_id.keys(): raise ValueError('매칭 ID가 조사구 모집단에 없음')
    reference=sum_valid(truth_all,'ref_AGB_kg')
    pred=sum_valid(detections_all,'AGB_default_kg')
    coord_reference=sum_valid(truth_coord,'ref_AGB_kg')
    missing_agb=sum(not math.isfinite(number(r.get('ref_AGB_kg'))) for r in truth_all)
    matched_ref=sum_valid([by_id[k] for k in paired_truth],'ref_AGB_kg')
    matched_pred=sum_valid([det_by_id[k] for k in paired_det],'AGB_default_kg')
    unmatched_pred=sum_valid([r for r in detections_all if r['det_id'] not in paired_det],'AGB_default_kg')
    missed_ref=sum_valid([r for r in truth_all if r['individualID'] not in paired_truth],'ref_AGB_kg')
    no_coord=sum_valid([r for r in truth_all if r['individualID'] not in coord_ids],'ref_AGB_kg')
    missed_coord=sum_valid([r for r in truth_coord if r['individualID'] not in paired_truth],'ref_AGB_kg')
    field_year=int(plot['field_year']); imagery_year=int(plot.get('imagery_year',field_year))
    row=dict(plotID=plot['plotID'],siteID=plot['siteID'],field_year=field_year,imagery_year=imagery_year,
        same_year=field_year==imagery_year,imagery_provisional=plot.get('imagery_provisional',False),
        area_m2=area,reported_area_m2=number(plot.get('reported_area_m2',area)),
        polygon_area_m2=number(plot.get('polygon_area_m2',area)),
        area_difference_m2=number(plot.get('polygon_area_m2',area))-area,
        n_truth=len(truth_all),n_truth_coord=len(truth_coord),n_detected=len(detections_all),n_matched=len(pairs),
        n_field_before_coordinate_filter=len(truth_all),n_without_matching_coordinate=len(truth_all)-len(truth_coord),
        n_missing_ref_AGB=missing_agb,AGB_reference_coverage=1-missing_agb/len(truth_all) if truth_all else 1.0,
        reference_complete=missing_agb==0,coordinates_complete=len(truth_all)==len(truth_coord),
        AGB_est_t_ha=scale(pred),AGB_ref_t_ha=scale(reference),AGB_ref_coordonly_t_ha=scale(coord_reference),
        matched_error_t_ha=scale(matched_pred-matched_ref),unmatched_detection_t_ha=scale(unmatched_pred),
        missed_truth_t_ha=scale(missed_ref),unmapped_truth_t_ha=scale(no_coord),missed_mapped_truth_t_ha=scale(missed_coord))
    row['C_est_tC_ha']=row['AGB_est_t_ha']*carbon_fraction
    row['C_ref_tC_ha']=row['AGB_ref_t_ha']*carbon_fraction
    row['C_ref_coordonly_tC_ha']=row['AGB_ref_coordonly_t_ha']*carbon_fraction
    row['C_est_047_tC_ha']=row['AGB_est_t_ha']*carbon_alternative
    row['C_ref_047_tC_ha']=row['AGB_ref_t_ha']*carbon_alternative
    row['C_bias_tC_ha']=row['C_est_tC_ha']-row['C_ref_tC_ha']
    row['C_missed_tC_ha']=row['missed_truth_t_ha']*carbon_fraction
    row['estimate_reference_ratio']=pred/reference if reference>0 else float('nan')
    if math.isfinite(reference) and math.isfinite(pred):
        assert math.isclose(pred-reference,matched_pred-matched_ref+unmatched_pred-missed_ref,rel_tol=1e-9,abs_tol=1e-6)
    return row

def comparison_statistics(truth,pred):
    result=errors(truth,pred)
    a=np.asarray(truth,dtype=float); b=np.asarray(pred,dtype=float)
    ok=np.isfinite(a)&np.isfinite(b); a=a[ok]; b=b[ok]
    mean_ref=float(a.mean()) if len(a) else np.nan
    mean_est=float(b.mean()) if len(b) else np.nan
    result.update(mean_reference=mean_ref,mean_estimate=mean_est,
        estimate_reference_ratio=mean_est/mean_ref if mean_ref>0 else np.nan)
    return result

if not RESOLUTION_ONLY:



    stage_rows=[]
    comparisons=[('height','m','field_H_m','H_m'),('crown_diameter','m','field_CD_m','CD_m'),
     ('DBH_Jucker','cm','field_DBH_cm','DBH_jucker_cm'),('DBH_regression_OOF','cm','field_DBH_cm','DBH_regression_oof_cm'),
     ('AGB_default','kg','ref_AGB_kg','AGB_default_kg'),('AGB_known_species','kg','ref_AGB_kg','AGB_known_species_kg'),
     ('AGB_regression_OOF','kg','ref_AGB_kg','AGB_regression_oof_kg')]
    for year in sorted(plots.field_year.unique()):
        z=matched[matched.field_year==year]
        for name,unit,refcol,predcol in comparisons:
            stage_rows.append(dict(stage=name,unit=unit,field_year=year,population='visible_matched',
                n_plots=z.plotID.nunique(),**errors(z[refcol],z[predcol])))
    stage_errors=pd.DataFrame(stage_rows)

    plot_rows=[]
    for _,p in plots.iterrows():
        d=detections[detections.plotID==p.plotID]
        fa=field_all[(field_all.plotID==p.plotID)&(field_all.field_year==p.field_year)]
        fc=field_geo[(field_geo.plotID==p.plotID)&(field_geo.field_year==p.field_year)]
        ma=MATCHES['all']; ma=ma[ma.plotID==p.plotID]
        plot_rows.append(plot_biomass_totals(d.to_dict('records'),fa.to_dict('records'),fc.to_dict('records'),
            ma.to_dict('records'),p.to_dict(),CFG['carbon_fraction'],CFG['carbon_alternative']))
    plot_comparison=pd.DataFrame(plot_rows)
    for (year,image_year),z in plot_comparison.groupby(['field_year','imagery_year']):
        for name,unit,refcol,predcol in [('plot_AGB','t/ha','AGB_ref_t_ha','AGB_est_t_ha'),
          ('plot_carbon','tC/ha','C_ref_tC_ha','C_est_tC_ha')]:
            stage_rows.append(dict(stage=name,unit=unit,siteID='ALL',field_year=year,imagery_year=image_year,
                population='plot',n_plots=len(z),**comparison_statistics(z[refcol],z[predcol])))
            for site,q in z.groupby('siteID'):
                stage_rows.append(dict(stage=name,unit=unit,siteID=site,field_year=year,imagery_year=image_year,
                    population='plot',n_plots=len(q),**comparison_statistics(q[refcol],q[predcol])))
    stage_errors=pd.DataFrame(stage_rows)
    for (year,pop),z in detmetrics.groupby(['field_year','population']):
        tp,fp,fn=z[['TP','FP','FN']].sum()
        stage_rows.append(dict(stage='detection',unit='ratio',field_year=year,population=pop,
            n=int(tp+fn),n_plots=z.plotID.nunique(),
            precision=tp/(tp+fp) if tp+fp else np.nan,recall=tp/(tp+fn) if tp+fn else np.nan,
            F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan))
    stage_errors=pd.DataFrame(stage_rows)
    display(stage_errors); display(plot_comparison)

## 14A. 나무 찾기 개선 함수: DF 탐색·CHM 꼭대기·watershed·결합

이전 실행의 유지사항(QUDO 충돌 경고/제외, gzip 검사 예외, /content/hf_cache, ORNL provisional,
좌표 없는 나무 포함 참조 총량)을 유지합니다. 새로운 함수들은 이 셀에서 정의만 합니다.
A는 고정 사전학습 모델의 score 5개×patch 3개 조합을 탐색합니다. 학습 가중치는 바꾸지 않습니다.
B는 nodata 정규화 Gaussian(sigma=1m) → 높이 가변 원형 최대점 → 시드 watershed입니다.
창 지름=2.51+0.00901H²(m): Popescu & Wynne (2004) eq.3의 2.51503을 요청대로 반올림합니다.
B의 H는 평활 후 탐지한 픽셀의 원본 CHM 높이, CD는 실제 watershed 면적의 등가원 지름입니다.
매칭은 모든 방법에서 기존의 줄기 점이 상자 안에 있는 헝가리안 규칙을 유지하므로 B의 매칭 상자는 수관 polygon의 envelope입니다.
C는 A의 상자 안에서 최고 CHM 꼭대기로 H를 보완(CD는 상자 기준), 밖의 꼭대기는 B 수관으로 추가하고 1m 중복을 제거합니다.
상자에 꼭대기가 없으면 원래 DF 탐지를 보존합니다. 이 선택은 보고서에도 명시합니다.

In [ ]:
"""Colab runtime extension. Insert after baseline evaluation and before sensitivity.
Do not execute this file on the local Windows machine.
"""
from scipy.ndimage import gaussian_filter, maximum_filter, label as ndi_label
from skimage.segmentation import watershed
from rasterio.features import shapes as raster_shapes
from shapely.geometry import shape
import copy


def finalize_detection_frame(records, method):
    cols=DET_COLUMNS if 'DET_COLUMNS' in globals() else ['det_id','plotID','siteID','field_year','score','H_m','CD_m','geometry']
    frame=gpd.GeoDataFrame(records,geometry='geometry',crs=WORK_CRS) if len(records) else gpd.GeoDataFrame(columns=cols,geometry='geometry',crs=WORK_CRS)
    frame['method']=method
    frame['DBH_jucker_cm']=estimate_dbh(frame.H_m,frame.CD_m)
    frame['AGB_default_kg']=estimate_agb(frame.DBH_jucker_cm)
    frame['C_default_kg']=carbon_kg(frame.AGB_default_kg,CFG['carbon_fraction'])
    return frame


def deepforest_patch_candidates(detector,rgb,chm,plot,patch_size,cache_dir):
    # Cache one inference per patch, then filter the same raw predictions at all five thresholds.
    saved=[]
    for obj in [detector.model,getattr(detector.model,'model',None)]:
        if obj is not None and hasattr(obj,'score_thresh'):
            saved.append((obj,'score_thresh',obj.score_thresh)); obj.score_thresh=0.05
    configs=[]
    for obj in [detector.config,getattr(detector.config,'retinanet',None)]:
        if obj is None: continue
        try:
            old=obj['score_thresh']; obj['score_thresh']=0.05; configs.append((obj,old))
        except (KeyError,TypeError,AttributeError): pass
    if not saved and not configs: raise RuntimeError('DeepForest 내부 score_thresh를 0.05로 설정하지 못했습니다. 낮은 점수 탐색을 숨겨 진행하지 않습니다.')
    oldcfg=CFG.copy(); oldout=globals()['OUT']
    directory=Path(cache_dir)/f'patch_{patch_size}'
    directory.mkdir(parents=True,exist_ok=True)
    fingerprint=dict(rgb=str(rgb),rgb_bytes=Path(rgb).stat().st_size,patch_size=int(patch_size),
        patch_overlap=CFG['patch_overlap'],nms_iou=CFG['nms_iou'],min_score=0.1,
        model_version=importlib.metadata.version('deepforest'),model=CFG['model_name'],revision=CFG['model_revision'])
    stamp=directory/f'{plot.plotID}_input.json'
    rawpath=directory/f'{plot.plotID}_raw_predictions.csv'
    if rawpath.exists() and (not stamp.exists() or json.loads(stamp.read_text())!=fingerprint): rawpath.unlink()
    try:
        CFG['patch_size']=int(patch_size); CFG['score_threshold']=0.1
        globals()['OUT']=directory
        rows=predict_crowns(detector,rgb,chm,plot)
        stamp.write_text(json.dumps(fingerprint,indent=2))
    finally:
        CFG.clear(); CFG.update(oldcfg); globals()['OUT']=oldout
        for obj,key,value in saved: setattr(obj,key,value)
        for obj,value in configs: obj['score_thresh']=value
    for row in rows: row['det_id']=f'{plot.plotID}_p{patch_size}_{row["det_id"].split("_")[-1]}'
    return finalize_detection_frame(rows,f'DF_patch_{patch_size}')


def variable_window_peaks(height,valid,resolution_m,diameter_a=2.51,diameter_b=0.00901,min_height=3.0):
    # Popescu & Wynne (2004) eq.3: combined crown WIDTH 2.51503 + .00901 H^2.
    # User-requested rounded coefficient 2.51 is used, diameter in metres (not radius).
    mask=valid & (height>=min_height)
    cross=np.array([[0,1,0],[1,1,1],[0,1,0]],dtype=bool)
    # At 1m, the smallest requested radius is ~1.30m: axial neighbours are inside
    # every variable window, but diagonal pixels are not necessarily inside it.
    if resolution_m>(diameter_a+diameter_b*min_height**2)/2:
        raise ValueError('해상도가 최소 가변 창 반지름보다 큽니다. 더 세밀한 CHM 또는 현지 창 계수를 사용하세요.')
    maxima=mask & (height>=maximum_filter(np.where(valid,height,-np.inf),footprint=cross,mode='constant',cval=-np.inf))
    components,count=ndi_label(maxima,structure=cross.astype(int))
    candidates=[]
    for k in range(1,count+1):
        coords=np.argwhere(components==k)
        order=np.lexsort((coords[:,1],coords[:,0],-height[coords[:,0],coords[:,1]]))
        r,c=coords[order[0]]; H=float(height[r,c])
        radius=(diameter_a+diameter_b*H*H)/2
        pixels=int(np.ceil(radius/resolution_m))
        r0=max(0,r-pixels); r1=min(height.shape[0],r+pixels+1)
        c0=max(0,c-pixels); c1=min(height.shape[1],c+pixels+1)
        rr,cc=np.ogrid[r0:r1,c0:c1]
        circle=((rr-r)*resolution_m)**2+((cc-c)*resolution_m)**2<=radius**2
        neighborhood=height[r0:r1,c0:c1][circle & valid[r0:r1,c0:c1]]
        if len(neighborhood) and H>=float(neighborhood.max())-1e-7:
            candidates.append((int(r),int(c),H,float(radius*2)))
    return sorted(candidates,key=lambda x:(-x[2],x[0],x[1]))


def chm_watershed_crowns(chm_path,plot,sigma_m=1.0):
    with rasterio.open(chm_path) as ds:
        raw=ds.read(1,masked=True); transform=ds.transform; source_crs=ds.crs
        resolution=float(abs(transform.a))
        if not np.isclose(abs(transform.a),abs(transform.e)): raise ValueError('CHM 정사각형 픽셀 필요')
        rawdata=np.asarray(raw.astype(float).filled(np.nan),dtype=float)
        valid=np.isfinite(rawdata)&(rawdata>=0)
        # Normalized Gaussian prevents nodata from biasing the smoothed canopy edge.
        weight=gaussian_filter(valid.astype(float),sigma=sigma_m/resolution)
        smooth=gaussian_filter(np.where(valid,rawdata,0),sigma=sigma_m/resolution)/np.maximum(weight,1e-12)
        peaks=variable_window_peaks(smooth,valid,resolution,CFG['chm_window_a'],CFG['chm_window_b'],CFG['min_height_m'])
        markers=np.zeros(rawdata.shape,dtype=np.int32)
        accepted=[]
        for r,c,H,diameter in peaks:
            # H is the unsmoothed CHM height at the detected (smoothed) treetop pixel.
            if rawdata[r,c]<CFG['min_height_m']: continue
            k=len(accepted)+1; markers[r,c]=k
            accepted.append((k,r,c,float(rawdata[r,c]),diameter))
        if not accepted: return finalize_detection_frame([],'B_CHM')
        segments=watershed(-smooth,markers,mask=valid&(rawdata>=CFG['min_height_m']))
        crown_parts={}
        for geom,value in raster_shapes(segments.astype('int32'),mask=segments>0,transform=transform):
            crown_parts.setdefault(int(value),[]).append(shape(geom))
    rows=[]
    for k,r,c,H,diameter in accepted:
        xy=transform*(c+0.5,r+0.5)
        point=gpd.GeoSeries([Point(*xy)],crs=source_crs).to_crs(WORK_CRS).iloc[0]
        if not plot.geometry.covers(point): continue
        polygon=gpd.GeoSeries([unary_union(crown_parts[k])],crs=source_crs).to_crs(WORK_CRS).iloc[0]
        area=float(polygon.area)
        if area<=0: continue
        # Keep the existing point-in-BOX Hungarian matching rule for every method.
        rows.append(dict(det_id=f'{plot.plotID}_CHM_{k}',plotID=plot.plotID,siteID=plot.siteID,
            field_year=plot.field_year,imagery_year=plot.imagery_year,score=1.0,H_m=H,
            CD_m=2*np.sqrt(area/np.pi),crown_area_m2=area,crown_geometry_wkt=polygon.wkt,
            peak_x=point.x,peak_y=point.y,window_diameter_m=diameter,geometry=polygon.envelope))
    return finalize_detection_frame(rows,'B_CHM')


def combine_boxes_and_peaks(boxes,chm,dedup_m=1.0):
    rows=[]; used=set()
    for _,boxrow in boxes.sort_values(['score','det_id'],ascending=[False,True]).iterrows():
        candidates=chm[chm.apply(lambda p:boxrow.geometry.covers(Point(p.peak_x,p.peak_y)),axis=1)] if len(chm) else chm
        had_seed=len(candidates)>0
        candidates=candidates[~candidates.det_id.isin(used)]
        if had_seed and candidates.empty: continue # the same CHM seed was already assigned to a higher-score box
        row=boxrow.to_dict(); row['det_id']='C_'+str(row['det_id'])
        if len(candidates):
            peak=candidates.sort_values(['H_m','det_id'],ascending=[False,True]).iloc[0]
            used.add(peak.det_id); row.update(H_m=peak.H_m,peak_x=peak.peak_x,peak_y=peak.peak_y)
        else:
            # Preserve DF boxes without a CHM seed; do not invent a peak.
            row.update(peak_x=boxrow.geometry.centroid.x,peak_y=boxrow.geometry.centroid.y)
        point=Point(row['peak_x'],row['peak_y'])
        if any(point.distance(Point(x['peak_x'],x['peak_y']))<dedup_m for x in rows): continue
        row['fusion_source']='DeepForest_box'; rows.append(row)
    for _,p in chm.iterrows():
        point=Point(p.peak_x,p.peak_y)
        if any(g.covers(point) for g in boxes.geometry): continue
        if any(point.distance(Point(x['peak_x'],x['peak_y']))<dedup_m for x in rows): continue
        row=p.to_dict(); row['det_id']='C_'+row['det_id']; row['fusion_source']='outside_CHM_peak'; rows.append(row)
    return finalize_detection_frame(rows,'C_fusion')


def detection_counts(detections,truth,plot_ids):
    tp=fp=fn=0
    for pid in plot_ids:
        d=detections[detections.plotID==pid]; f=truth[truth.plotID==pid]
        hits=len(match_one_to_one(d,f)); tp+=hits; fp+=len(d)-hits; fn+=len(f)-hits
    return dict(TP=tp,FP=fp,FN=fn,precision=tp/(tp+fp) if tp+fp else np.nan,
                recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan)


def nested_group_parameter_selection(candidates,plot_table,truth,n_splits=5):
    ids=np.array(sorted(plot_table.plotID.unique()))
    outer=GroupKFold(n_splits=min(n_splits,len(ids)))
    predictions=[]; selections=[]; inner_rows=[]; split_rows=[]
    keys=sorted(candidates)
    for fold,(train,test) in enumerate(outer.split(ids,groups=ids)):
        trainids=ids[train]; testids=ids[test]
        assert set(trainids).isdisjoint(testids)
        for pid in trainids: split_rows.append(dict(outer_fold=fold,plotID=pid,role='train'))
        for pid in testids: split_rows.append(dict(outer_fold=fold,plotID=pid,role='test'))
        inner=GroupKFold(n_splits=min(CFG['search_inner_folds'],len(trainids)))
        scores={key:dict(TP=0,FP=0,FN=0) for key in keys}
        for inner_fold,(itr,ival) in enumerate(inner.split(trainids,groups=trainids)):
            assert set(trainids[itr]).isdisjoint(trainids[ival])
            assert set(trainids[ival]).isdisjoint(testids)
            # Fixed pretrained inference has no fitting here; only held-out inner labels score parameters.
            for key in keys:
                result=detection_counts(candidates[key],truth,trainids[ival])
                inner_rows.append(dict(outer_fold=fold,inner_fold=inner_fold,patch_size=key[0],score_threshold=key[1],**result))
                for k in ['TP','FP','FN']: scores[key][k]+=result[k]
        def ranking(key):
            counts=scores[key]; den=2*counts['TP']+counts['FP']+counts['FN']
            f1=2*counts['TP']/den if den else -1
            return (-f1,key[0],key[1]) # deterministic ties
        best=min(keys,key=ranking)
        chosen=candidates[best][candidates[best].plotID.isin(testids)].copy()
        chosen['outer_fold']=fold; chosen['selected_patch_size']=best[0]; chosen['selected_score_threshold']=best[1]
        predictions.append(chosen)
        selections.append(dict(outer_fold=fold,patch_size=best[0],score_threshold=best[1],
            train_plots=';'.join(trainids),test_plots=';'.join(testids),**detection_counts(chosen,truth,testids)))
    oof=gpd.GeoDataFrame(pd.concat(predictions,ignore_index=True),geometry='geometry',crs=WORK_CRS)
    # Deployment selection uses all 23 plots; these scores are tuning scores, NOT unbiased test scores.
    deployment_rows=[]
    for key in keys: deployment_rows.append(dict(patch_size=key[0],score_threshold=key[1],**detection_counts(candidates[key],truth,ids)))
    deployment=pd.DataFrame(deployment_rows).sort_values(['F1','patch_size','score_threshold'],ascending=[False,True,True])
    return oof,pd.DataFrame(selections),pd.DataFrame(inner_rows),pd.DataFrame(split_rows),deployment


def evaluate_detection_method(det,plot_table,all_truth,coordinate_truth):
    metric_rows=[]; pair_tables={}; joined=None
    for population in ['visible','all']:
        pairs=[]
        for _,p in plot_table.iterrows():
            d=det[det.plotID==p.plotID]; f=coordinate_truth[coordinate_truth.plotID==p.plotID]
            if population=='visible': f=f[f.visible]
            match=match_one_to_one(d,f); pairs.append(match)
            metric_rows.append(dict(plotID=p.plotID,siteID=p.siteID,field_year=p.field_year,population=population,
                **detection_counts(d,f,[p.plotID])))
        pair_tables[population]=pd.concat(pairs,ignore_index=True)
    visible=pair_tables['visible']
    joined=visible.merge(det.drop(columns='geometry'),on=['det_id','plotID'],validate='one_to_one')
    joined=joined.merge(coordinate_truth.drop(columns=['geometry','plotID','siteID','field_year']),on='individualID',validate='one_to_one')
    joined['DBH_regression_oof_cm'],joined['cv_fold'],_=grouped_dbh_regression(joined,CFG['cv_folds'])
    joined['AGB_regression_oof_kg']=estimate_agb(joined.DBH_regression_oof_cm)
    joined['AGB_known_species_kg']=estimate_agb(joined.DBH_jucker_cm,joined.species_beta0,joined.species_beta1)
    joined['species_effect_kg']=joined.AGB_known_species_kg-joined.AGB_default_kg
    joined['species_effect_pct']=100*(joined.AGB_known_species_kg/joined.AGB_default_kg-1)
    plotrows=[]
    for _,p in plot_table.iterrows():
        ds=det[det.plotID==p.plotID]; fa=all_truth[all_truth.plotID==p.plotID]; fc=coordinate_truth[coordinate_truth.plotID==p.plotID]
        pairs=pair_tables['all']; pairs=pairs[pairs.plotID==p.plotID]
        plotrows.append(plot_biomass_totals(ds.to_dict('records'),fa.to_dict('records'),fc.to_dict('records'),pairs.to_dict('records'),
            p.to_dict(),CFG['carbon_fraction'],CFG['carbon_alternative']))
    return dict(detections=det,matched=joined,metrics=pd.DataFrame(metric_rows),pairs=pair_tables,plots=pd.DataFrame(plotrows))


def method_comparison_rows(method_results,requested_sites):
    rows=[]
    for method,result in method_results.items():
        for site in ['ALL']+list(requested_sites):
            pc=result['plots']; mt=result['matched']; dm=result['metrics']
            if site!='ALL': pc=pc[pc.siteID==site]; mt=mt[mt.siteID==site]; dm=dm[dm.siteID==site]
            stats=comparison_statistics(pc.C_ref_tC_ha,pc.C_est_tC_ha)
            row=dict(method=method,siteID=site,n_plots=len(pc),n_field=int(pc.n_truth.sum()),n_detected=int(pc.n_detected.sum()),
                n_carbon_plots=stats['n'],C_est_mean_tC_ha=stats['mean_estimate'],C_ref_mean_tC_ha=stats['mean_reference'],
                estimate_reference_ratio=stats['estimate_reference_ratio'],carbon_r=stats['r'],carbon_RMSE_tC_ha=stats['RMSE'],carbon_MAE_tC_ha=stats['MAE'])
            for pop in ['visible','all']:
                q=dm[dm.population==pop]; tp=q.TP.sum(); fp=q.FP.sum(); fn=q.FN.sum()
                row['F1_'+pop]=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
                row['precision_'+pop]=tp/(tp+fp) if tp+fp else np.nan; row['recall_'+pop]=tp/(tp+fn) if tp+fn else np.nan
            for name,a,b in [('height_Bias_m','field_H_m','H_m'),('crown_Bias_m','field_CD_m','CD_m'),
                ('DBH_Bias_cm','field_DBH_cm','DBH_jucker_cm'),('individual_AGB_Bias_kg','ref_AGB_kg','AGB_default_kg')]:
                row[name]=errors(mt[a],mt[b])['Bias']
            rows.append(row)
    return pd.DataFrame(rows)


def detection_algorithm_smoke_checks():
    # Colab-only: actual scipy/skimage/rasterio calls on two synthetic trees, no external data/GPU.
    from types import SimpleNamespace
    yy,xx=np.mgrid[:64,:64]
    canopy=20*np.exp(-((xx-18)**2+(yy-32)**2)/8)+30*np.exp(-((xx-46)**2+(yy-32)**2)/8)
    path=OUT/'synthetic_CHM_smoke.tif'
    with rasterio.open(path,'w',driver='GTiff',width=64,height=64,count=1,dtype='float32',
        crs=WORK_CRS,transform=rasterio.transform.from_origin(0,64,1,1),nodata=-9999) as ds:
        ds.write(canopy.astype('float32'),1)
    p=SimpleNamespace(plotID='SMOKE',siteID='SMOKE',field_year=2021,imagery_year=2021,geometry=box(0,0,64,64))
    result=chm_watershed_crowns(path,p,1.0)
    assert len(result)==2,'합성 두 나무의 꼭대기 2개를 찾아야 합니다'
    assert np.allclose(sorted(result.H_m),[20,30],atol=0.01)
    assert np.allclose(result.CD_m,2*np.sqrt(result.crown_area_m2/np.pi))
    assert np.isclose(result.crown_area_m2.sum(),np.count_nonzero(canopy>=CFG['min_height_m']))
    first_peak=result.iloc[0].to_dict()
    duplicate=dict(first_peak); duplicate['det_id']='duplicate_box'; duplicate['score']=0.8
    first_peak['score']=0.9
    boxes=gpd.GeoDataFrame([first_peak,duplicate],geometry='geometry',crs=WORK_CRS)
    fused=combine_boxes_and_peaks(boxes,result,1.0)
    assert len(fused)==2,'동일 꼭대기 중복 상자는 제거하고 상자 밖 두 번째 꼭대기는 추가해야 합니다'
    ids=['P0','P1','P2','P3']; drows=[]; frows=[]
    for i,pid in enumerate(ids):
        x=i*20
        drows.append(dict(det_id='d'+pid,plotID=pid,score=0.8,H_m=20,CD_m=5,geometry=box(x-3,-3,x+3,3)))
        frows.append(dict(individualID='f'+pid,plotID=pid,stemDiameter=30,geometry=Point(x,0)))
    ds=gpd.GeoDataFrame(drows,geometry='geometry',crs=WORK_CRS)
    fs=gpd.GeoDataFrame(frows,geometry='geometry',crs=WORK_CRS)
    plotdf=pd.DataFrame({'plotID':ids})
    grid={(300,0.1):ds,(400,0.2):ds.iloc[:2].copy()}
    _,first,_,_,_=nested_group_parameter_selection(grid,plotdf,fs,n_splits=2)
    changed=fs.copy(); changed.loc[changed.plotID=='P0','geometry']=Point(9999,9999)
    _,second,_,_,_=nested_group_parameter_selection(grid,plotdf,changed,n_splits=2)
    fold=first[first.test_plots.str.split(';').apply(lambda z:'P0' in z)].index[0]
    assert first.loc[fold,['patch_size','score_threshold']].equals(second.loc[fold,['patch_size','score_threshold']]),'바깥 평가 정답이 선택에 새어 들어갔습니다'
    for _,r in first.iterrows(): assert set(r.train_plots.split(';')).isdisjoint(r.test_plots.split(';'))
    path.unlink(missing_ok=True)
    print('합성 CHM 꼭대기·watershed 면적·CD, GroupKFold 정답 격리 검증 통과')

## 14B. 동일 23개 조사구에서 네 방법 비교·기본값 선택

A: 바깥 GroupKFold 평가 조사구의 정답은 조합 선택에 쓰지 않습니다.
각 바깥 학습 조사구 안의 안쪽 GroupKFold **보이는 나무 기준 F1**로 조합을 고르고, 바깥 평가 탐지만 합쳐 OOF 결과를 만듭니다.
전 표본으로 고른 단일 조합은 향후 새 자료 적용용으로 별도 저장하며 현재 성능표에는 쓰지 않습니다.
PR 그림의 곡선은 조합 선택 점수, 별표는 바깥 OOF 성능입니다. B는 고정 설정, C는 A의 바깥 OOF 상자를 사용합니다.
4개 방법×전체/사이트 비교표를 저장하고, 23개 공통 조사구 탄소 RMSE가 최소인 방법을 이후 민감도·보고서의 기본값으로 씁니다.
이는 같은 23개에서 선택한 기본값이며 새 지역의 독립 검증 결과라는 뜻은 아닙니다.
추가 GPU 추론은 patch별 1회씩(총 69 조사구 추론)이며 threshold는 동일 예측을 필터링해 재사용합니다.
정확히 23개가 아니거나 요청한 4개 그림 조사구가 없으면 멈추므로 비교 모집단이 조용히 바뀌지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
if not RESOLUTION_ONLY:
    # Same plot cohort for all methods; never silently remove an inconvenient plot.
    detection_algorithm_smoke_checks()
    if CFG['expected_plot_count'] is not None and len(plots)!=CFG['expected_plot_count']:
        raise RuntimeError(f'이번 비교는 동일 23개 조사구가 필요합니다. 현재 {len(plots)}개: 영상 커버·실측 선택 설정 확인')
    plots[['plotID','siteID','field_year','imagery_year']].sort_values('plotID').to_csv(OUT/'comparison_cohort.csv',index=False)
    BASELINE_RESULT=dict(detections=detections.copy(),matched=matched.copy(),metrics=detmetrics.copy(),
        pairs={k:v.copy() for k,v in MATCHES.items()},plots=plot_comparison.copy())
    all_candidates={}; patch_frames={}; CHM_FRAMES=[]
    for patch in CFG['search_patch_sizes']:
        frames=[]
        for _,p in plots.iterrows():
            print('A 후보 추론:',p.plotID,'patch',patch)
            frames.append(deepforest_patch_candidates(model,*RASTERS[p.plotID],p,patch,OUT/'detection_search_cache'))
        patch_frames[patch]=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry='geometry',crs=WORK_CRS)
        for threshold in CFG['search_score_thresholds']:
            q=patch_frames[patch]; all_candidates[(patch,threshold)]=q[q.score>=threshold].copy()
    A_oof,A_outer,A_inner,A_splits,A_deployment=nested_group_parameter_selection(all_candidates,plots,field_geo[field_geo.visible],CFG['cv_folds'])
    A_outer.to_csv(OUT/'A_outer_groupkfold.csv',index=False); A_inner.to_csv(OUT/'A_inner_grid_scores.csv',index=False)
    A_splits.to_csv(OUT/'A_fold_plot_assignments.csv',index=False); A_deployment.to_csv(OUT/'A_deployment_grid.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,5))
    for patch,q in A_deployment.groupby('patch_size'):
        q=q.sort_values('score_threshold'); ax.plot(q.recall,q.precision,'o-',label=f'patch {patch}')
        for _,r in q.iterrows(): ax.annotate(str(r.score_threshold),(r.recall,r.precision),fontsize=8)
    oof_counts=detection_counts(A_oof,field_geo[field_geo.visible],plots.plotID)
    ax.scatter(oof_counts['recall'],oof_counts['precision'],marker='*',s=180,c='black',label='Nested outer OOF')
    ax.set(xlabel='Recall',ylabel='Precision',title='A: tuning curves (selection scores) and outer OOF')
    ax.legend(); fig.savefig(OUT/'A_precision_recall.png',dpi=170,bbox_inches='tight'); plt.close(fig)
    for _,p in plots.iterrows():
        print('B CHM 꼭대기/watershed:',p.plotID)
        CHM_FRAMES.append(chm_watershed_crowns(RASTERS[p.plotID][1],p,CFG['chm_sigma_m']))
    B_chm=gpd.GeoDataFrame(pd.concat(CHM_FRAMES,ignore_index=True),geometry='geometry',crs=WORK_CRS)
    fusion=[]
    for _,p in plots.iterrows():
        q=combine_boxes_and_peaks(A_oof[A_oof.plotID==p.plotID],B_chm[B_chm.plotID==p.plotID],CFG['fusion_dedup_m'])
        fusion.append(q)
    C_fusion=gpd.GeoDataFrame(pd.concat(fusion,ignore_index=True),geometry='geometry',crs=WORK_CRS)
    METHOD_RESULTS={'baseline_015':BASELINE_RESULT}
    for name,det in [('A_DF_nested_OOF',A_oof),('B_CHM',B_chm),('C_fusion_nested_OOF',C_fusion)]:
        METHOD_RESULTS[name]=evaluate_detection_method(det,plots,field_all,field_geo)
    method_comparison=method_comparison_rows(METHOD_RESULTS,CFG['sites'])
    method_comparison.to_csv(OUT/'method_comparison.csv',index=False)
    for name,result in METHOD_RESULTS.items():
        folder=OUT/'methods'/name; folder.mkdir(parents=True,exist_ok=True)
        for key,filename in [('detections','detected_trees.csv'),('matched','matched_trees_visible.csv'),('metrics','detection_metrics.csv'),('plots','plot_comparison.csv')]:
            q=result[key].copy()
            if 'geometry' in q: q['geometry']=q.geometry.apply(lambda g:g.wkt)
            pd.DataFrame(q).to_csv(folder/filename,index=False)
        for pop,q in result['pairs'].items(): q.to_csv(folder/f'matches_{pop}.csv',index=False)

    for pid in CFG['comparison_overlay_plots']:
        if pid not in set(plots.plotID): raise ValueError(f'필수 나란히 비교 조사구 없음: {pid}')
        fig,axes=plt.subplots(1,4,figsize=(24,6),sharex=True,sharey=True)
        for ax,(name,result) in zip(axes,METHOD_RESULTS.items()):
            with rasterio.open(RASTERS[pid][0]) as ds: show(ds,ax=ax); crs=ds.crs
            q=result['detections']; q=q[q.plotID==pid].to_crs(crs)
            if len(q):
                q.boundary.plot(ax=ax,color='cyan',linewidth=0.6)
                if 'peak_x' in q and 'peak_y' in q:
                    peaks=q.dropna(subset=['peak_x','peak_y'])
                    points=gpd.GeoSeries([Point(r.peak_x,r.peak_y) for _,r in peaks.iterrows()],crs=WORK_CRS).to_crs(crs)
                    ax.scatter(points.x,points.y,c='yellow',s=10,marker='x')
            f=field_geo[field_geo.plotID==pid].to_crs(crs)
            for visible,color in [(True,'lime'),(False,'magenta')]:
                z=f[f.visible==visible]; ax.scatter(z.geometry.x,z.geometry.y,c=color,s=14,edgecolors='black',linewidths=0.3)
            p=plots[plots.plotID==pid].iloc[0]
            gpd.GeoSeries([p.geometry],crs=WORK_CRS).to_crs(crs).boundary.plot(ax=ax,color='yellow')
            ax.set_title(f'{name}\nn={len(q)}')
        fig.suptitle(pid+' | green: visible field stems; magenta: occluded')
        fig.savefig(OUT/f'{pid}_methods_side_by_side.png',dpi=160,bbox_inches='tight'); plt.close(fig)

    # Choose on plot carbon error, never fit/change Jucker or Chojnacky coefficients.
    ranking=method_comparison[method_comparison.siteID=='ALL'].copy()
    ranking=ranking[ranking.n_carbon_plots==len(plots)].sort_values([CFG['default_method_metric'],'method'])
    if ranking.empty: raise RuntimeError('23개 전체 조사구의 참조 AGB가 유효하지 않아 기본 방법을 선택할 수 없습니다.')
    DEFAULT_DETECTION_METHOD=ranking.iloc[0]['method']
    deployment=A_deployment.iloc[0]
    selection=dict(default_method=DEFAULT_DETECTION_METHOD,selection_metric=CFG['default_method_metric'],
        selection_scope='same 23 plot comparison; selection error is not independent generalization error',
        deployment_DF_patch_size=int(deployment.patch_size),deployment_DF_score_threshold=float(deployment.score_threshold),
        current_evaluation='outer GroupKFold OOF for A and C; globally selected parameters are deployment-only',
        chosen_carbon_RMSE_tC_ha=float(ranking.iloc[0].carbon_RMSE_tC_ha))
    (OUT/'selected_detection_method.json').write_text(json.dumps(selection,ensure_ascii=False,indent=2))
    chosen=METHOD_RESULTS[DEFAULT_DETECTION_METHOD]
    detections=chosen['detections']; matched=chosen['matched']; detmetrics=chosen['metrics']; MATCHES=chosen['pairs']; plot_comparison=chosen['plots']
    _,_,selected_dbh_fit=grouped_dbh_regression(matched,CFG['cv_folds'])
    if selected_dbh_fit:
        selected_dbh_fit['detection_method']=DEFAULT_DETECTION_METHOD
        (OUT/'dbh_regression_full_fit.json').write_text(json.dumps(selected_dbh_fit,indent=2))
    # Rebuild the existing stage table for the chosen default before downstream sensitivity/report cells.
    stage_rows=[]
    for year,z in matched.groupby('field_year'):
        for name,unit,refcol,predcol in comparisons:
            stage_rows.append(dict(stage=name,unit=unit,field_year=year,population='visible_matched',n_plots=z.plotID.nunique(),**errors(z[refcol],z[predcol])))
    for (year,image_year),z in plot_comparison.groupby(['field_year','imagery_year']):
        for name,unit,a,b in [('plot_AGB','t/ha','AGB_ref_t_ha','AGB_est_t_ha'),('plot_carbon','tC/ha','C_ref_tC_ha','C_est_tC_ha')]:
            for site,q in [('ALL',z)]+list(z.groupby('siteID')):
                stage_rows.append(dict(stage=name,unit=unit,siteID=site,field_year=year,imagery_year=image_year,population='plot',n_plots=len(q),**comparison_statistics(q[a],q[b])))
    for (year,pop),z in detmetrics.groupby(['field_year','population']):
        tp=z.TP.sum(); fp=z.FP.sum(); fn=z.FN.sum()
        stage_rows.append(dict(stage='detection',unit='ratio',field_year=year,population=pop,n=int(tp+fn),n_plots=z.plotID.nunique(),
            precision=tp/(tp+fp) if tp+fp else np.nan,recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan))
    stage_errors=pd.DataFrame(stage_rows)
    display(method_comparison.round(3)); print('이후 단계 기본 방법:',DEFAULT_DETECTION_METHOD)

## 15. 수고 오차 ±10%·±20%의 탄소 영향

탐지·매칭·수관 지름을 고정하고 H만 바꾸는 통제된 민감도입니다.
수고 오차가 탐지 필터나 매칭까지 바꾸는 효과는 포함하지 않습니다.
Jucker DBH와 기본 AGB 식을 연쇄 적용하므로 탄소 배율은 이론적으로 `(1+오차)^(0.887×2.4133)`입니다.
이는 한국의 낮은 정확도 높이 자료를 검토하는 시나리오이며 실제 한국 자료의 정확도 검증을 대신하지 않습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
if not RESOLUTION_ONLY:
    sensitivity=[]
    for _,p in plots.iterrows():
        z=detections[detections.plotID==p.plotID]
        baseline=float(kg_to_t_ha(z.AGB_default_kg.sum(),p.area_m2)*CFG['carbon_fraction'])
        for delta in [-0.2,-0.1,0.0,0.1,0.2]:
            D=estimate_dbh(z.H_m*(1+delta),z.CD_m)
            C=float(kg_to_t_ha(estimate_agb(D).sum(),p.area_m2)*CFG['carbon_fraction'])
            factor=(1+delta)**(CFG['jucker']['b']*default['beta1'])
            if baseline>0: assert np.isclose(C/baseline,factor)
            sensitivity.append(dict(plotID=p.plotID,field_year=p.field_year,height_error_pct=100*delta,
                C_tC_ha=C,baseline_tC_ha=baseline,change_tC_ha=C-baseline,
                change_pct=100*(C/baseline-1) if baseline>0 else np.nan,theoretical_factor=factor))
    sensitivity=pd.DataFrame(sensitivity)
    display(sensitivity)

## 16. 조사구 지도와 단계별 산점도

RGB 위의 상자는 조사 범위 내 탐지, 점은 실측 줄기 위치입니다.
녹색은 보이는 나무, 자홍색은 가려진 나무이며 조사 경계를 노란색으로 표시합니다.
그림의 라벨은 Colab 기본 폰트에서도 깨지지 않도록 영어로 씁니다. 보고서는 한국어입니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def scatter_identity(df,x,y,label,filename):
    if x not in df or y not in df: return
    z=df[[x,y]].replace([np.inf,-np.inf],np.nan).dropna()
    fig,ax=plt.subplots(figsize=(5,5))
    if len(z):
        ax.scatter(z[x],z[y],s=12,alpha=0.6)
        lo=min(z.min()); hi=max(z.max())
        ax.plot([lo,hi],[lo,hi],'k--',label='1:1')
    ax.set(xlabel='Reference '+label,ylabel='Estimated '+label,title=f'{label} (n={len(z)})')
    fig.savefig(FIGDIR/filename,dpi=170,bbox_inches='tight'); plt.close(fig)

def plot_carbon_scatter(plot_table,output_folder):
    for (year,image_year),cohort in plot_table.groupby(['field_year','imagery_year']):
        z=cohort.replace([np.inf,-np.inf],np.nan).dropna(subset=['C_ref_tC_ha','C_est_tC_ha'])
        fig,ax=plt.subplots(figsize=(6,6))
        for site,q in z.groupby('siteID'):
            ax.scatter(q.C_ref_tC_ha,q.C_est_tC_ha,s=35,alpha=0.8,label=site,
                color={'GRSM':'tab:blue','ORNL':'tab:orange'}.get(site))
        if len(z):
            hi=max(z[['C_ref_tC_ha','C_est_tC_ha']].max())*1.05
            hi=max(hi,1.0); ax.plot([0,hi],[0,hi],'k--',label='1:1')
            ax.set_xlim(0,hi); ax.set_ylim(0,hi)
        ax.set(xlabel='Field-based reference carbon (tC/ha)',ylabel='Estimated carbon (tC/ha)',
            title=f'Plot carbon: field {year} / imagery {image_year} (n={len(z)})')
        ax.legend(); ax.set_aspect('equal',adjustable='box')
        filename=f'plot_carbon_{year}.png' if year==image_year else f'plot_carbon_field{year}_imagery{image_year}.png'
        fig.savefig(Path(output_folder)/filename,dpi=170,bbox_inches='tight'); plt.close(fig)

if not RESOLUTION_ONLY:
    FIGDIR=OUT/'figures'; FIGDIR.mkdir(exist_ok=True)
    for _,p in plots.iterrows():
        fig,ax=plt.subplots(figsize=(9,9))
        rgb,_=RASTERS[p.plotID]
        with rasterio.open(rgb) as ds:
            show(ds,ax=ax)
            crs=ds.crs
        gpd.GeoSeries([p.geometry],crs=WORK_CRS).to_crs(crs).boundary.plot(ax=ax,color='yellow',linewidth=2)
        d=detections[detections.plotID==p.plotID].to_crs(crs)
        if not d.empty: d.boundary.plot(ax=ax,color='cyan',linewidth=0.7)
        f=field_geo[field_geo.plotID==p.plotID].to_crs(crs)
        for v,color,label in [(True,'lime','Visible field stem'),(False,'magenta','Occluded field stem')]:
            q=f[f.visible==v]
            if not q.empty: ax.scatter(q.geometry.x,q.geometry.y,c=color,s=18,label=label,edgecolors='black',linewidths=0.3)
        ax.set_title(f'{p.plotID}: imagery {p.imagery_year} / field {p.field_year}')
        ax.legend(); fig.savefig(FIGDIR/f'{p.plotID}_overlay.png',dpi=160,bbox_inches='tight'); plt.close(fig)


    # 같은 해 검증 산점도와 인접연도 보완 그림을 섞지 않음
    for year in sorted(plots.field_year.unique()):
        z=matched[matched.field_year==year]
        for x,y,label,fn in [('field_H_m','H_m','Height (m)','height'),
          ('field_CD_m','CD_m','Crown diameter (m)','crown'),
          ('field_DBH_cm','DBH_jucker_cm','DBH Jucker (cm)','dbh_jucker'),
          ('field_DBH_cm','DBH_regression_oof_cm','DBH OOF (cm)','dbh_oof')]:
            scatter_identity(z,x,y,label,f'{fn}_{year}.png')

    plot_carbon_scatter(plot_comparison,FIGDIR)

## 17. CSV·한국어 보고서·재사용 함수 저장

CSV의 기하정보는 WKT로 저장하며 좌표계는 별도 기록합니다.
현재 실행 결과는 `outputs/실행ID/`, 최근 결과 안내는 `outputs/report.md`에 기록합니다.
노트북에 정의한 함수는 독립 Python 모듈로도 내보내며, 전역 CFG·ROOT 등을 쓰는 NEON 전용 로더와
입력 배열·계수만 받는 일반 추정 함수를 구분합니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
def write_geo_csv(df,path):
    df=df.copy()
    if 'geometry' in df: df['geometry']=df.geometry.apply(lambda g:g.wkt if g is not None else None)
    pd.DataFrame(df).to_csv(path,index=False)

def build_site_summary(plot_table,detection_metrics,matched_table,requested_sites):
    rows=[]
    for site in requested_sites:
        site_plots=plot_table[plot_table.siteID==site]
        if site_plots.empty:
            rows.append(dict(siteID=site,n_plots=0,comparison_note='분석 결과 없음')); continue
        for (field_year,image_year),q in site_plots.groupby(['field_year','imagery_year']):
            ids=set(q.plotID)
            dm=detection_metrics[detection_metrics.plotID.isin(ids)]
            mt=matched_table[matched_table.plotID.isin(ids)]
            row=dict(siteID=site,field_year=field_year,imagery_year=image_year,
                comparison_note='영상 연도 = 실측 연도' if field_year==image_year else f'{site}은 영상 연도 ≠ 실측 연도',
                n_plots=len(q),n_field_trees=int(q.n_truth.sum()),n_coord_trees=int(q.n_truth_coord.sum()),
                n_detections=int(q.n_detected.sum()))
            for population,label in [('visible','F1_visible'),('all','F1_all')]:
                z=dm[dm.population==population]; tp=z.TP.sum(); fp=z.FP.sum(); fn=z.FN.sum()
                row[label]=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
            for label,refcol,predcol in [('height_Bias_m','field_H_m','H_m'),
                ('DBH_Bias_cm','field_DBH_cm','DBH_jucker_cm'),('individual_AGB_Bias_kg','ref_AGB_kg','AGB_default_kg')]:
                row[label]=errors(mt[refcol],mt[predcol])['Bias']
            stats=comparison_statistics(q.C_ref_tC_ha,q.C_est_tC_ha)
            row.update(n_carbon_plots=stats['n'],C_est_mean_tC_ha=stats['mean_estimate'],
                C_ref_mean_tC_ha=stats['mean_reference'],estimate_reference_ratio=stats['estimate_reference_ratio'])
            rows.append(row)
    return pd.DataFrame(rows)

def micro_detection(df):
    rows=[]
    for (year,pop),z in df.groupby(['field_year','population']):
        tp=z.TP.sum(); fp=z.FP.sum(); fn=z.FN.sum()
        rows.append(dict(field_year=year,population=pop,TP=tp,FP=fp,FN=fn,
            precision=tp/(tp+fp) if tp+fp else np.nan,recall=tp/(tp+fn) if tp+fn else np.nan,
            F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan))
    return pd.DataFrame(rows)

if not RESOLUTION_ONLY:


    SUMMARY_LABELS={'siteID':'사이트','field_year':'실측 연도','imagery_year':'영상 연도','comparison_note':'연도 구분',
        'n_plots':'조사구 수','n_field_trees':'실측 나무 수(전체)','n_coord_trees':'매칭 좌표 나무 수',
        'n_detections':'탐지 수','F1_visible':'F1(보이는 나무)','F1_all':'F1(전체 매핑 나무)',
        'height_Bias_m':'수고 Bias(m)','DBH_Bias_cm':'DBH Bias(cm)','individual_AGB_Bias_kg':'개체 AGB Bias(kg)',
        'n_carbon_plots':'탄소 비교 조사구 수','C_est_mean_tC_ha':'평균 추정(tC/ha)',
        'C_ref_mean_tC_ha':'평균 참조(tC/ha)','estimate_reference_ratio':'추정/참조'}
    site_summary=build_site_summary(plot_comparison,detmetrics,matched,CFG['sites'])
    summary_display=site_summary.rename(columns=SUMMARY_LABELS)
    site_summary.to_csv(OUT/'site_summary.csv',index=False)

    write_geo_csv(detections,OUT/'detected_trees.csv')
    write_geo_csv(field_geo,OUT/'field_trees.csv')
    field_all.to_csv(OUT/'field_trees_all.csv',index=False)
    matched.to_csv(OUT/'matched_trees_visible.csv',index=False)
    for key,df in MATCHES.items(): df.to_csv(OUT/f'matches_{key}.csv',index=False)
    detmetrics.to_csv(OUT/'detection_metrics.csv',index=False)
    stage_errors.to_csv(OUT/'stage_errors.csv',index=False)
    plot_comparison.to_csv(OUT/'plot_comparison.csv',index=False)
    sensitivity.to_csv(OUT/'height_sensitivity.csv',index=False)
    pd.DataFrame(AUDIT,columns=['stage','item','reason']).to_csv(OUT/'audit.csv',index=False)
    species_valid=matched[~matched.get('species_coeff_missing',pd.Series(True,index=matched.index)).astype(bool)]
    species_plot=species_valid.groupby(['plotID','field_year']).agg(n=('individualID','size'),
        matched_default_AGB_kg=('AGB_default_kg','sum'),matched_known_species_AGB_kg=('AGB_known_species_kg','sum')).reset_index()
    species_plot['change_pct']=100*(species_plot.matched_known_species_AGB_kg/species_plot.matched_default_AGB_kg-1)
    species_plot['change_C_kg']=(species_plot.matched_known_species_AGB_kg-species_plot.matched_default_AGB_kg)*CFG['carbon_fraction']
    species_plot.to_csv(OUT/'species_sensitivity_matched.csv',index=False)


    species=matched[~matched.get('species_coeff_missing',pd.Series(True,index=matched.index)).astype(bool)]
    species_summary=species.groupby('field_year')[['AGB_default_kg','AGB_known_species_kg','species_effect_kg','species_effect_pct']].mean()
    report=f"""# GRSM·ORNL 2021 나무·탄소 비교 보고서

    ## 나무 찾기 방법 비교 및 이후 기본값
    기본 방법: **{DEFAULT_DETECTION_METHOD}**, 선택 기준: 같은 23개 조사구의 탄소 RMSE 최소.
    {method_comparison.to_markdown(index=False,floatfmt='.3f')}
    A/C는 바깥 GroupKFold 평가 탐지만 합친 OOF 결과다. 각 fold의 조합은 그 fold의 학습 조사구에서만 보이는 나무 F1로 선택했다.
    전 표본 선택 조합은 A_deployment_grid.csv와 selected_detection_method.json에 새 자료 적용용으로 별도 저장했다.
    기본 방법의 선택 오차는 동일 비교 표본의 값이며 독립적인 지역 일반화 성능으로 해석하지 않는다.
    B: sigma={CFG['chm_sigma_m']}m Gaussian, 창 지름=2.51+0.00901H², H=꼭대기 픽셀 원본 CHM 높이,
    CD=2√(watershed 면적/π). Popescu & Wynne (2004), [원 논문](https://doi.org/10.14358/pers.70.5.589).
    매칭은 B도 수관 envelope 상자를 사용해 기존 point-in-box 1:1 규칙을 유지했다.
    C: A OOF 상자 안 최고 CHM 꼭대기로 수고 보완, 상자 CD 유지; 꼭대기 없는 DF 상자는 보존;
    상자 밖 CHM 꼭대기는 B의 수고·수관으로 추가, {CFG['fusion_dedup_m']}m 중복 제거.
    시드 watershed는 [scikit-image 공식 API](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.watershed)를 사용한다.

    ## 사이트별 결과 요약
    {summary_display.to_markdown(index=False,floatfmt='.3f')}
    F1은 보이는 나무 기준을 기본으로 제시하고 전체 매핑 나무 F1도 함께 표시한다.
    수고·DBH·개체 AGB Bias는 보이는 나무 매칭 표본 기준이다.
    조사구 탄소 평균·비율은 추정과 참조가 모두 유효한 같은 조사구 집합의 평균이다.

    실행 ID: {RUN_ID}; 작업 CRS: {WORK_CRS.to_string()}
    분석 범위: 실제 조사 소구획의 살아 있는 DBH≥{CFG['min_field_dbh_cm']}cm. 조사구 참조 총량에는 좌표 없는 나무도 포함.
    2021년 실측과 인접연도 보완을 분리하며, 보완 연도={sorted(set(plots.field_year)-{CFG['year']})}.
    실행한 조사구={len(plots)}, 전체 실측 개체={len(field_all)}, 매칭 좌표 개체={len(field_geo)}, 탐지={len(detections)}, 보이는 나무 매칭={len(matched)}.

    ## 영상 연월·잠정 자료 여부
    {pd.DataFrame([dict(siteID=s,**v) for s,v in AOP_SELECTION.items()]).to_markdown(index=False)}
    1차 수정 시 API 조회에서 ORNL 2021-06은 PROVISIONAL로 확인했다. 현재 선택 상태는 위 표를 따른다.
    영상 대체가 필요할 때만 ORNL 2022→2020을 검토하며 실측 연도는 유지한다.
    실제 제공 연월은 aop_availability.csv, 타일 수는 aop_download_inventory.csv,
    조사구별 커버 여부는 aop_plot_coverage.csv에 기록했다.

    ## 탐지 정확도 (마이크로 집계)
    {micro_detection(detmetrics).to_markdown(index=False)}

    ## 단계별 오차
    Bias=추정−참조; Pearson r. DBH 회귀는 조사구 GroupKFold의 OOF 결과.
    {stage_errors.to_markdown(index=False)}

    ## 조사구 AGB·탄소 비교
    {plot_comparison.to_markdown(index=False)}
    총량 차이 = 매칭 개체 추정오차 + 미매칭 탐지 추정량 − 놓친 실측 개체 참조량.
    참조 AGB 자체가 누락된 경우에만 총량은 NaN 처리. 면적은 totalSampledAreaTrees를 우선한다.
    area_m2·reported_area_m2는 분모, polygon_area_m2·area_difference_m2는 기하 면적 진단이다.
    AGB_ref_coordonly_t_ha는 좌표 있는 나무만의 보조 참조값이다.
    unmapped_truth_t_ha는 좌표/경계 문제로 매칭할 수 없던 실측량이며 실제 탐지 누락과 구분해야 한다.

    ## 수종 미지정 영향 (계수가 확인된 동일 매칭 표본의 평균)
    {species_summary.to_markdown()}
    수종 미대응 개체는 이 표에서 제외하고 개별 CSV에 표시.
    조사구별 동일 매칭 표본 합계 비교는 species_sensitivity_matched.csv 참고.

    ## 높이 민감도
    {sensitivity.groupby('height_error_pct')[['change_pct','change_tC_ha']].mean().to_markdown()}
    탐지와 CD를 고정한 가상 H 변화. CHM 오차가 탐지/매칭에 미치는 영향은 포함하지 않음.
    탄소계수 0.47 사용 시 0.5 대비 모든 탄소량은 6% 감소.

    ## 공식·계수·출처
    - DBH(cm)=0.367×(H(m)×CD(m))^0.887. itcSegment biome 11; g=0.
      [Jucker 구현 원문]({JUCKER_SOURCE}); Jucker et al. (2017), GCB 23:177–190.
    - AGB(kg)=exp(β0+β1×ln(DBH(cm))). 기본 β0={default['beta0']}, β1={default['beta1']}.
      이는 Chojnacky의 일반 활엽수 Cor/Eri/Lau/etc 대체 그룹. ‘mixed hardwood’라는 독립 공식 아님.
      [MasterTaxonList]({TAXON_URL}), [Chojnacky 원 논문](https://doi.org/10.1093/forestry/cpt053).
    - 수종 기지 시 동일 식에 taxonID별 beta0/beta1 적용; 계수 결측은 개별 표시.
    - C=AGB×0.5 (민감도 0.47). kg/1000×10000/㎡ = t/ha.
    - [NEONForestAGBv2]({AGB_URL}), [저장소](https://github.com/atkinsjeff/NEONForestAGB),
      [데이터 DOI](https://doi.org/10.6084/m9.figshare.25625352.v1).
    - [NEON 조사 사용자 안내서](https://data.neonscience.org/api/v0/documents/NEON_vegStructure_userGuide_vE),
      [neonutilities 공식 API](https://neon-utilities-python.readthedocs.io/en/latest/api.html),
      [DeepForest 공식 예측 안내](https://deepforest.readthedocs.io/en/stable/getting_started/intro_tutorials/04_predict_large_tile.html).
      설치 버전은 versions.json, 원문 해시는 formula_provenance.json 참고.

    ## 해석상 한계
    - NEONForestAGB는 실측 DBH 기반 모델 추정값이다. 개체 바이오매스를 파괴 조사한 실측 정답이 아니다.
    - 나무 아래층은 영상에 가려진다. 개체 매칭의 ‘전체’는 DBH≥10cm 좌표 확인 나무이며 조사구 총량은 좌표 없는 나무도 포함한다.
    - 줄기 위치 기반 매칭은 수관 IoU 검증을 대체하지 않으며 수평 위치 오차·기울어진 줄기·겹친 수관 영향을 받는다.
    - 탐지는 중심점, 실측은 줄기점 기준으로 경계를 적용한다. 경계 근처의 외부 나무 수관 유입/유출 편향이 남는다.
    - CHM 최대값은 이웃 큰 나무 높이를 담을 수 있다. 상자 기반 CD는 실제 수관 직경과 다르다.
    - 현재 사전학습 가중치는 NEON 자료를 사용한 모델이다. 모델 로더/버전에 따라 2021 논문의 모델과 다를 수 있다.
    - subplotID와 총 조사 면적이 일치하지 않는 조사구는 추측하지 않고 제외했다. audit.csv 확인.
    - 2021 현장 조사일과 항공 촬영일은 일치하지 않을 수 있다. field date/ref_date와 영상 파일 경로를 확인한다.
    - 품질 플래그는 원본 표와 field_trees.csv에 보존했다. 최신 실측 배포본과 2024년 고정 AGBv2의 개정 차이가 있을 수 있다.
    - 다줄기 나무는 NEON의 줄기별 individualID를 유지한다. 중복 방문은 마지막 측정으로 정리한다.
    - AGB 참조값이 없는 개체의 총량을 0으로 간주하지 않는다. 좌표 미확인 개체는 audit에 기록한다.
    - coordinates_complete=False는 개체 매칭의 자료 완전성 진단이다. 조사구 참조 총량을 제외하는 조건이 아니다.
    - 한국에서는 CRS·센서 해상도·수관 구조·수종식·CHM 정확도가 달라 지역 검증과 필요 시 모델 재학습이 필요하다.
    """
    (OUT/'report.md').write_text(report,encoding='utf-8')
    (ROOT/'outputs/report.md').write_text(f'최근 완료 결과: [{RUN_ID}/report.md]({RUN_ID}/report.md)\n',encoding='utf-8')
    print('완료:',OUT/'report.md')
    display(micro_detection(detmetrics))

## 17A. 해상도 실험 함수와 캐시 재개

3차 결과와 좌표 없는 나무를 포함한 참조표, 조사구 경계, 기존RGB·CHM crop을 불러옵니다.
원본10cm의 네 방법은 캐시 파일을 그대로 재사용합니다. 25/51cm는 면적평균 축소→bilinear 확대하며,
확대본의 CRS·범위·transform·shape는 원본과 정확히 같습니다. 정수 픽셀 크기에 따른 실제GSD를 출력합니다.
모델 입력은 항상10cm 격자이므로 patch600의 지상 범위는60m로 유지됩니다.

In [ ]:
RESOLUTION_CODE_SHA = '15f59840aded39d66d1aabf2b2961794871a785efde480f7721a19467792e51c'
"""Colab-only extension; embedded in the notebook by resolution_extension.py.

Reuses the completed third run. Never downloads imagery or reruns old methods.
Dependencies/functions/globals come from the original notebook definition cells.
"""
import shutil
import zipfile
from contextlib import contextmanager
from rasterio.enums import Resampling
from rasterio.transform import array_bounds
from rasterio.io import MemoryFile
from shapely import wkt

RESOLUTION_SCHEMA = 'resolution-v1-average-native-bilinear-10cm'
RESOLUTION_GSDS = (10, 25, 51)
RESOLUTION_THRESHOLDS = (0.1, 0.2, 0.3, 0.4, 0.5)
RESOLUTION_PATCH = 600
RESOLUTION_FIXED_SCORE = 0.2
RESOLUTION_OVERLAYS = ('GRSM_016', 'ORNL_006', 'GRSM_051')


def resolution_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def resolution_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + '.part')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def resolution_read_geo(path):
    frame = pd.read_csv(path)
    if 'geometry' not in frame:
        raise ValueError(f'WKT geometry column missing: {path}')
    frame['geometry'] = frame.geometry.map(wkt.loads)
    return gpd.GeoDataFrame(frame, geometry='geometry', crs=WORK_CRS)


def resolution_bool(series):
    return series.map(lambda v: str(v).strip().lower() in ('true', '1'))


def resolution_required_files():
    files = ['config.json', 'versions.json', 'selected_detection_method.json',
             'comparison_cohort.csv', 'sampled_footprints.gpkg', 'raster_manifest.csv',
             'field_trees_all.csv', 'field_trees.csv', 'method_comparison.csv',
             'A_outer_groupkfold.csv', 'A_inner_grid_scores.csv',
             'A_fold_plot_assignments.csv', 'A_deployment_grid.csv', 'report.md']
    for method in ('baseline_015', 'A_DF_nested_OOF', 'B_CHM', 'C_fusion_nested_OOF'):
        files += [f'methods/{method}/{f}' for f in ('detected_trees.csv', 'matched_trees_visible.csv',
                  'detection_metrics.csv', 'plot_comparison.csv', 'matches_visible.csv', 'matches_all.csv')]
    return files


def resolution_find_base(root, explicit=None):
    required = resolution_required_files()
    expected = set(CFG['comparison_plot_ids'])

    def valid(path):
        if not all((path / f).is_file() for f in required):
            return False
        selection = json.loads((path / 'selected_detection_method.json').read_text())
        cohort = pd.read_csv(path / 'comparison_cohort.csv')
        return (selection['default_method'] == 'A_DF_nested_OOF'
                and selection['deployment_DF_patch_size'] == 600
                and np.isclose(selection['deployment_DF_score_threshold'], 0.2)
                and len(cohort) == 23 and set(cohort.plotID) == expected)

    if explicit:
        chosen = Path(explicit)
        if not valid(chosen):
            raise FileNotFoundError(f'3차 완료 결과/필수 파일/23개 cohort 확인 필요: {chosen}')
        return chosen
    folders = [root / 'outputs', root / 'outputs_v3'] + sorted(p for p in (root / 'outputs').glob('*') if p.is_dir())
    # An original result ZIP may have been unpacked on an earlier run. Keep using
    # that immutable snapshot even after the final ZIP is replaced by this run.
    folders += [p.parent for p in (root / 'data/cache/resolution_base').rglob('selected_detection_method.json')]
    candidates = [p for p in folders if valid(p)]
    # Prefer original result folders, not copies produced by this extension.
    originals = [p for p in candidates if not (p / 'resolution_provenance.json').exists()]
    candidates = originals or candidates
    if not candidates and (root / 'outputs/neon_results.zip').is_file():
        archive = root / 'outputs/neon_results.zip'
        dest = root / 'data/cache/resolution_base' / resolution_sha(archive)[:16]
        dest.mkdir(parents=True, exist_ok=True)
        if not (dest / '.complete').exists():
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (dest / member.filename).resolve()
                    if not target.is_relative_to(dest.resolve()):
                        raise ValueError('Unsafe member in local result ZIP')
                z.extractall(dest)
            (dest / '.complete').touch()
        candidates = [p for p in [dest] + [f.parent for f in dest.rglob('selected_detection_method.json')] if valid(p)]
    if not candidates:
        raise FileNotFoundError('Drive의 3차 결과 폴더/outputs/neon_results.zip을 찾지 못했습니다. '
                                'RESOLUTION_BASE_DIR에 완료된 결과 폴더를 지정하세요. 다운로드/기존 방법 재계산은 하지 않습니다.')
    signatures = {tuple(resolution_sha(p / f) for f in
                       ('method_comparison.csv', 'field_trees_all.csv', 'field_trees.csv', 'A_outer_groupkfold.csv')) for p in candidates}
    if len(signatures) != 1:
        raise RuntimeError('서로 다른 완료 결과가 여러 개입니다. RESOLUTION_BASE_DIR로 지난주 3차 폴더를 지정하세요.')
    return sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))[0]


def resolution_crop_path(recorded, plot_id, kind, year):
    recorded = Path(recorded)
    if recorded.is_file():
        return recorded
    relocated = ROOT / 'data/crops' / recorded.name
    if relocated.is_file():
        return relocated
    # Never substitute a different crop with a similar name or download a tile.
    raise FileNotFoundError(f'{plot_id} {year} {kind} 기존 crop 없음: {recorded}; '
                            f'Drive data/crops/{recorded.name}를 확인하세요.')


def resolution_load_state():
    global CFG, OUT, WORK_CRS, default, plots, field_all, field_geo, RASTERS
    global RESOLUTION_BASE, RESOLUTION_CACHE, RESOLUTION_BASE_SIGNATURE, RESOLUTION_INPUTS
    global RESOLUTION_BASE_METHODS, RESOLUTION_BASE_SUMMARY, RESOLUTION_SELECTION, RUN_ID
    global RESOLUTION_DETECTOR
    RESOLUTION_DETECTOR = None
    RESOLUTION_BASE = resolution_find_base(ROOT, RESOLUTION_BASE_DIR)
    original_cfg = json.loads((RESOLUTION_BASE / 'config.json').read_text())
    required_cfg = dict(jucker=dict(a=0.367, b=0.887, g=0.0, biome=11), carbon_fraction=0.5,
                        min_height_m=3.0, match_size='stemDiameter', cv_folds=5, search_inner_folds=4)
    for key, value in required_cfg.items():
        if original_cfg[key] != value:
            raise ValueError(f'기존 설정 불일치: {key}: {original_cfg[key]} != {value}')
    assert np.isclose(original_cfg['agb_default']['beta0'], -2.2118)
    assert np.isclose(original_cfg['agb_default']['beta1'], 2.4133)
    assert original_cfg['aop_include_provisional'] is True
    versions = json.loads((RESOLUTION_BASE / 'versions.json').read_text())
    if importlib.metadata.version('deepforest') != versions['deepforest']:
        raise RuntimeError(f'DeepForest {versions["deepforest"]} 필요. 첫 설치 셀/런타임 재시작을 확인하세요.')
    CFG = original_cfg
    default = CFG['agb_default']
    WORK_CRS = CRS.from_epsg(32617)
    cohort = pd.read_csv(RESOLUTION_BASE / 'comparison_cohort.csv')
    if cohort.siteID.value_counts().to_dict() != {'GRSM': 15, 'ORNL': 8}:
        raise ValueError('GRSM15 + ORNL8 이외의 cohort')
    assert set(cohort.plotID) == set(CFG['comparison_plot_ids'])
    plots = gpd.read_file(RESOLUTION_BASE / 'sampled_footprints.gpkg').to_crs(WORK_CRS)
    plots = plots[plots.plotID.isin(cohort.plotID)].copy()
    for col in ('siteID', 'field_year', 'imagery_year'):
        plots[col] = plots.plotID.map(cohort.set_index('plotID')[col])
    assert plots.plotID.is_unique and len(plots) == 23
    field_all = pd.read_csv(RESOLUTION_BASE / 'field_trees_all.csv')
    field_geo = resolution_read_geo(RESOLUTION_BASE / 'field_trees.csv')
    field_geo['visible'] = resolution_bool(field_geo.visible)
    for name, frame in [('field_all', field_all), ('field_geo', field_geo)]:
        if not set(frame.plotID).issubset(set(cohort.plotID)):
            raise ValueError(f'{name}에 다른 조사구 포함')
        assert not frame.individualID.duplicated().any()
    manifest = pd.read_csv(RESOLUTION_BASE / 'raster_manifest.csv').set_index('plotID')
    assert set(cohort.plotID).issubset(set(manifest.index))
    plots['imagery_provisional'] = plots.plotID.map(resolution_bool(manifest.imagery_provisional))
    assert plots.loc[plots.siteID == 'ORNL', 'imagery_provisional'].all()
    RASTERS = {}; inputs = []
    for _, p in plots.iterrows():
        row = manifest.loc[p.plotID]
        paths = tuple(resolution_crop_path(row[k], p.plotID, k, p.imagery_year) for k in ('rgb', 'chm'))
        RASTERS[p.plotID] = paths
        for kind, path, gsd in zip(('rgb', 'chm'), paths, (0.1, 1.0)):
            with rasterio.open(path) as ds:
                if not ds.crs or not ds.crs.is_projected or not np.allclose(ds.res, gsd, atol=1e-6):
                    raise ValueError(f'{kind} 원본 CRS/GSD 확인: {path}: {ds.crs}, {ds.res}')
                if kind == 'rgb' and (ds.count < 3 or ds.dtypes[0] != 'uint8'):
                    raise ValueError('원본 RGB uint8 3밴드 필요')
            inputs.append(dict(plotID=p.plotID, kind=kind, path=str(path), sha256=resolution_sha(path)))
    RESOLUTION_INPUTS = pd.DataFrame(inputs)
    original_hashes = {f: resolution_sha(RESOLUTION_BASE / f) for f in resolution_required_files()}
    spec = dict(schema=RESOLUTION_SCHEMA, code_sha=RESOLUTION_CODE_SHA,
                source_files=original_hashes, rasters=inputs,
                resolutions=list(RESOLUTION_GSDS), patch=600, thresholds=list(RESOLUTION_THRESHOLDS))
    RESOLUTION_BASE_SIGNATURE = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
    RUN_ID = 'resolution_' + RESOLUTION_BASE_SIGNATURE[:12]
    OUT = ROOT / 'outputs' / RUN_ID
    RESOLUTION_CACHE = ROOT / 'data/cache/resolution' / RESOLUTION_BASE_SIGNATURE[:16]
    OUT.mkdir(parents=True, exist_ok=True); RESOLUTION_CACHE.mkdir(parents=True, exist_ok=True)
    if OUT.resolve() == RESOLUTION_BASE.resolve():
        raise ValueError('기존 결과와 해상도 실험 결과 폴더를 분리해야 합니다.')
    # Preserve all old method files, reports and figures verbatim in the new ZIP.
    for source in list(RESOLUTION_BASE.rglob('*')):
        rel = source.relative_to(RESOLUTION_BASE)
        if (not source.is_file() or source.is_relative_to(OUT) or source.suffix.lower() == '.zip'
                or (len(rel.parts) > 1 and rel.parts[0] not in ('methods', 'figures'))):
            continue
        dest = OUT / rel
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(source, dest)
    RESOLUTION_INPUTS.to_csv(OUT / 'resolution_input_manifest.csv', index=False)
    resolution_json(OUT / 'resolution_provenance.json', dict(base_directory=str(RESOLUTION_BASE),
                    base_signature=RESOLUTION_BASE_SIGNATURE, schema=RESOLUTION_SCHEMA,
                    source_files=original_hashes, settings=spec, no_imagery_download=True,
                    old_methods_recomputed=False))
    RESOLUTION_BASE_METHODS = {}
    for name in ('baseline_015', 'A_DF_nested_OOF', 'B_CHM', 'C_fusion_nested_OOF'):
        folder = RESOLUTION_BASE / 'methods' / name
        RESOLUTION_BASE_METHODS[name] = dict(detections=resolution_read_geo(folder / 'detected_trees.csv'),
            matched=pd.read_csv(folder / 'matched_trees_visible.csv'),
            metrics=pd.read_csv(folder / 'detection_metrics.csv'), plots=pd.read_csv(folder / 'plot_comparison.csv'),
            pairs={pop: pd.read_csv(folder / f'matches_{pop}.csv') for pop in ('visible', 'all')})
    RESOLUTION_BASE_SUMMARY = pd.read_csv(RESOLUTION_BASE / 'method_comparison.csv')
    RESOLUTION_SELECTION = json.loads((RESOLUTION_BASE / 'selected_detection_method.json').read_text())
    if (RESOLUTION_BASE / 'excluded_taxon_coefficients.csv').exists():
        conflict = pd.read_csv(RESOLUTION_BASE / 'excluded_taxon_coefficients.csv')
        if conflict.astype(str).apply(lambda s: s.str.contains('QUDO')).any().any():
            warnings.warn('기존 QUDO 계수 충돌: 해당 수종 계수 제외 상태를 유지합니다. 전체 참조 나무는 제거하지 않습니다.')
    print('기존 3차 결과:', RESOLUTION_BASE)
    print('해상도 실험 결과:', OUT)
    print('캐시:', RESOLUTION_CACHE)
    print('조사구:', plots.siteID.value_counts().to_dict(), '전체 참조 나무:', len(field_all), '좌표 나무:', len(field_geo))


def resolution_grid(width, height, xres, yres, requested_m):
    """Integer dimensions preserve the original extent; record actual GSD."""
    w = max(1, int(round(width * xres / requested_m)))
    h = max(1, int(round(height * yres / requested_m)))
    return w, h


def resolution_rgb(source, requested_cm, cache_directory):
    source = Path(source)
    with rasterio.open(source) as ds:
        profile = ds.profile.copy(); shape0 = (ds.height, ds.width)
        transform0 = ds.transform; crs0 = ds.crs; bounds0 = tuple(ds.bounds)
        if not np.isclose(transform0.b, 0) or not np.isclose(transform0.d, 0):
            raise ValueError('북향 정렬 원본 RGB 필요')
        w, h = resolution_grid(ds.width, ds.height, *ds.res, requested_cm / 100)
        low_transform = transform0 * rasterio.Affine.scale(ds.width / w, ds.height / h)
        actual = (abs(low_transform.a), abs(low_transform.e))
    metadata = dict(schema=RESOLUTION_SCHEMA, source_sha256=resolution_sha(source),
        nominal_cm=requested_cm, coarse_width=w, coarse_height=h,
        actual_gsd_x_m=actual[0], actual_gsd_y_m=actual[1], bounds=list(bounds0),
        coarse_transform=list(low_transform), input_transform=list(transform0),
        input_width=shape0[1], input_height=shape0[0], crs=crs0.to_string(),
        downsampling='rasterio.Resampling.average', upsampling='rasterio.Resampling.bilinear',
        rasterio_version=rasterio.__version__, gdal_version=rasterio.__gdal_version__)
    cache_directory = Path(cache_directory); cache_directory.mkdir(parents=True, exist_ok=True)
    coarse = cache_directory / 'coarse_native.tif'; expanded = cache_directory / 'model_input_10cm.tif'
    stamp = cache_directory / 'resampling.json'
    if not (coarse.is_file() and expanded.is_file() and stamp.is_file() and json.loads(stamp.read_text()) == metadata):
        with rasterio.open(source) as ds:
            # Float average avoids a premature uint8 rounding before bilinear expansion.
            low = ds.read([1, 2, 3], out_shape=(3, h, w), out_dtype='float32',
                          masked=True, resampling=Resampling.average)
        low_profile = profile.copy()
        low_profile.update(driver='GTiff', count=3, dtype='float32', nodata=np.nan,
                           height=h, width=w, transform=low_transform, compress='deflate')
        tmp = coarse.with_name(coarse.stem + '.part.tif')
        with rasterio.open(tmp, 'w', **low_profile) as ds:
            ds.write(low.filled(np.nan)); ds.update_tags(nominal_gsd_cm=str(requested_cm), experiment=RESOLUTION_SCHEMA)
        tmp.replace(coarse)
        with rasterio.open(coarse) as ds:
            expanded_data = ds.read(out_shape=(3, *shape0), resampling=Resampling.bilinear, masked=True)
        good = ~np.ma.getmaskarray(expanded_data).any(axis=0)
        expanded_data = np.clip(np.rint(expanded_data.filled(0)), 0, 255).astype('uint8')
        model_profile = profile.copy(); model_profile.update(driver='GTiff', count=3, dtype='uint8', compress='deflate')
        tmp = expanded.with_name(expanded.stem + '.part.tif')
        with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
            with rasterio.open(tmp, 'w', **model_profile) as ds:
                ds.write(expanded_data); ds.write_mask(good.astype('uint8') * 255)
                ds.update_tags(effective_nominal_gsd_cm=str(requested_cm), downsample='average', upsample='bilinear')
        tmp.replace(expanded)
        resolution_json(stamp, metadata)
    # Always read the files themselves, including on a cache hit.
    with rasterio.open(coarse) as low, rasterio.open(expanded) as model_input:
        assert low.crs == model_input.crs == crs0
        assert np.allclose(low.bounds, bounds0, rtol=0, atol=1e-8)
        assert np.allclose(low.res, actual, rtol=0, atol=1e-10)
        assert model_input.transform == transform0 and model_input.shape == shape0
        assert tuple(model_input.bounds) == bounds0
    return expanded, coarse, metadata


@contextmanager
def resolution_context(**changes):
    previous_cfg = CFG.copy(); previous_out = globals()['OUT']
    try:
        if 'output' in changes:
            globals()['OUT'] = Path(changes.pop('output'))
        CFG.update(changes)
        yield
    finally:
        CFG.clear(); CFG.update(previous_cfg); globals()['OUT'] = previous_out


def resolution_model():
    global RESOLUTION_DETECTOR
    if globals().get('RESOLUTION_DETECTOR') is not None:
        return RESOLUTION_DETECTOR
    location = RESOLUTION_CACHE / 'model'; location.mkdir(exist_ok=True)
    stamp = location / 'resolved_model.json'
    if stamp.exists():
        info = json.loads(stamp.read_text())
    else:
        # Resolve once; partial reruns retain the same immutable HF commit.
        from huggingface_hub import HfApi
        model_info = HfApi().model_info(CFG['model_name'], revision=CFG['model_revision'])
        info = dict(model_name=CFG['model_name'], source_revision=CFG['model_revision'],
                    resolved_revision=model_info.sha, deepforest_version=importlib.metadata.version('deepforest'),
                    limitation='3차 결과는 main으로만 기록되어 당시 가중치 SHA 동일 여부는 직접 검증할 수 없음')
        resolution_json(stamp, info)
    with resolution_context(output=location, model_revision=info['resolved_revision']):
        RESOLUTION_DETECTOR = load_detector()
    resolution_json(OUT / 'resolution_model_provenance.json', info)
    return RESOLUTION_DETECTOR


def resolution_candidates(plot, cm, rgb, chm):
    directory = RESOLUTION_CACHE / 'detections' / f'{cm}cm' / plot.plotID
    directory.mkdir(parents=True, exist_ok=True)
    processed = directory / 'candidates.csv'; stamp = directory / 'candidates_input.json'
    fingerprint = dict(schema=RESOLUTION_SCHEMA, code_sha=RESOLUTION_CODE_SHA,
        rgb_sha256=resolution_sha(rgb), chm_sha256=resolution_sha(chm), plot_wkt=plot.geometry.wkt,
        config=CFG, deepforest_version=importlib.metadata.version('deepforest'),
        resampling=json.loads((Path(rgb).parent / 'resampling.json').read_text()))
    # Model revision is stored once per experiment and in every prediction stamp.
    model_stamp = RESOLUTION_CACHE / 'model/resolved_model.json'
    fingerprint['resolved_model'] = json.loads(model_stamp.read_text()) if model_stamp.exists() else None
    if processed.exists() and stamp.exists() and json.loads(stamp.read_text()) == fingerprint:
        return resolution_read_geo(processed)
    detector = resolution_model()
    fingerprint['resolved_model'] = json.loads(model_stamp.read_text())
    with resolution_context(model_revision=fingerprint['resolved_model']['resolved_revision']):
        frame = deepforest_patch_candidates(detector, rgb, chm, plot, 600, directory / 'raw')
    frame['det_id'] = frame.det_id.map(lambda s: f'r{cm}_' + s)
    write_geo_csv(frame, processed.with_suffix('.part.csv'))
    processed.with_suffix('.part.csv').replace(processed)
    resolution_json(stamp, fingerprint)
    return frame


def resolution_verify_cached_winners(outer_records, inner_records):
    """Pure Python proof that the cached full-grid winner wins the subset."""
    winners = {}; totals = {}
    for row in inner_records:
        if int(row['patch_size']) != 600:
            continue
        key = (int(row['outer_fold']), float(row['score_threshold']))
        counts = totals.setdefault(key, [0, 0, 0])
        for i, column in enumerate(('TP', 'FP', 'FN')):
            counts[i] += int(row[column])
    assert {t for fold, t in totals} == set(RESOLUTION_THRESHOLDS)
    for row in outer_records:
        fold = int(row['outer_fold']); threshold = float(row['score_threshold'])
        assert int(row['patch_size']) == 600 and math.isclose(threshold, .2)
        def rank(t):
            tp, fp, fn = totals[fold, t]; den = 2 * tp + fp + fn
            return (-(2 * tp / den if den else -1), t)
        best = min(RESOLUTION_THRESHOLDS, key=rank)
        assert math.isclose(best, threshold), f'Cached subset winner differs in fold{fold}'
        winners[fold] = best
    assert set(winners) == set(range(5))
    return winners


def resolution_cached_10cm():
    outer = pd.read_csv(RESOLUTION_BASE / 'A_outer_groupkfold.csv')
    inner = pd.read_csv(RESOLUTION_BASE / 'A_inner_grid_scores.csv')
    splits = pd.read_csv(RESOLUTION_BASE / 'A_fold_plot_assignments.csv')
    deployment = pd.read_csv(RESOLUTION_BASE / 'A_deployment_grid.csv')
    # The original unrestricted optimum is also optimal in the patch600 subset.
    # Verify this explicitly rather than relabel a mixed-patch OOF as fixed600.
    restricted = inner[inner.patch_size == 600].copy()
    resolution_verify_cached_winners(outer.to_dict('records'), inner.to_dict('records'))
    result = RESOLUTION_BASE_METHODS['A_DF_nested_OOF']
    return result, outer, restricted, splits, deployment[deployment.patch_size == 600].copy()


def resolution_runtime_checks():
    # Real Rasterio average/bilinear test on a small synthetic checkerboard.
    with tempfile.TemporaryDirectory(prefix='resolution_check_') as td:
        td = Path(td); source = td / 'synthetic.tif'
        yy, xx = np.indices((100, 100)); tile = ((xx + yy) % 2 * 200 + 20).astype('uint8')
        with rasterio.open(source, 'w', driver='GTiff', width=100, height=100, count=3,
                           dtype='uint8', crs='EPSG:32617', transform=rasterio.Affine(.1, 0, 100, 0, -.1, 200)) as ds:
            ds.write(np.stack([tile] * 3))
        for cm in (25, 51):
            expanded, coarse, meta = resolution_rgb(source, cm, td / str(cm))
            with rasterio.open(coarse) as ds:
                assert np.var(ds.read(1)) < np.var(tile)
                assert abs(float(ds.read(1).mean()) - float(tile.mean())) < 1
            with rasterio.open(expanded) as ds:
                assert ds.width == ds.height == 100 and np.allclose(ds.res, .1)
            # The second request must use the finished cache without a rewrite.
            previous = expanded.stat().st_mtime_ns
            resolution_rgb(source, cm, td / str(cm))
            assert expanded.stat().st_mtime_ns == previous
    detection_algorithm_smoke_checks()
    original = RESOLUTION_BASE_METHODS['A_DF_nested_OOF']['plots']
    for _, p in original.iterrows():
        fa = field_all[field_all.plotID == p.plotID]
        assert len(fa) == p.n_truth
        if fa.ref_AGB_kg.notna().all():
            expected = fa.ref_AGB_kg.sum() / 1000 * 10000 / p.area_m2 * .5
            assert np.isclose(expected, p.C_ref_tC_ha)
    print('해상도/격자/실제 GSD/평균 보존/캐시 재사용/매칭·CV 격리/좌표 없는 참조 포함 검증 통과')


def resolution_save_result(result, folder):
    folder.mkdir(parents=True, exist_ok=True)
    for key, filename in [('detections', 'detected_trees.csv'), ('matched', 'matched_trees_visible.csv'),
                          ('metrics', 'detection_metrics.csv'), ('plots', 'plot_comparison.csv')]:
        write_geo_csv(result[key], folder / filename)
    for pop, frame in result['pairs'].items():
        frame.to_csv(folder / f'matches_{pop}.csv', index=False)


def resolution_score_text(outer, plot_ids):
    ids = set(plot_ids)
    selected = {int(r.outer_fold): float(r.score_threshold) for _, r in outer.iterrows()
                if ids.intersection(str(r.test_plots).split(';'))}
    values = set(selected.values())
    return next(iter(values)) if len(values) == 1 else json.dumps(selected, sort_keys=True)


def resolution_summaries(results, selections):
    rows = []; plotrows = []
    for (cm, setting), result in results.items():
        if cm == 10:
            # Reuse saved 10cm numbers, including the individual-error population.
            table = RESOLUTION_BASE_SUMMARY[RESOLUTION_BASE_SUMMARY.method == 'A_DF_nested_OOF'].copy()
        else:
            table = method_comparison_rows({f'{cm}_{setting}': result}, CFG['sites'])
        outer = selections[cm]['outer']
        for _, row in table.iterrows():
            r = row.to_dict(); r.pop('method', None)
            ids = plots.plotID if row.siteID == 'ALL' else plots.loc[plots.siteID == row.siteID, 'plotID']
            r.update(resolution_cm=cm, setting=setting, patch_size=600,
                     selected_score_threshold=.2 if setting == 'fixed' else resolution_score_text(outer, ids),
                     deployment_score_threshold=float(selections[cm]['deployment'].iloc[0].score_threshold),
                     source='cached_3rd_run_A' if cm == 10 else 'new_RGB_resolution_experiment')
            for old, new in [('height_Bias_m', 'height_Bias'), ('crown_Bias_m', 'crown_Bias'), ('DBH_Bias_cm', 'DBH_Bias'),
                             ('individual_AGB_Bias_kg', 'individual_AGB_Bias'), ('C_est_mean_tC_ha', 'C_est_mean'),
                             ('C_ref_mean_tC_ha', 'C_ref_mean'), ('carbon_RMSE_tC_ha', 'carbon_RMSE'), ('carbon_MAE_tC_ha', 'carbon_MAE')]:
                r[new] = r[old]
            rows.append(r)
        frame = result['plots'].copy(); frame['resolution_cm'] = cm; frame['setting'] = setting
        frame['selected_score_threshold'] = .2
        frame['outer_fold'] = -1
        if setting == 'retuned':
            for _, selection in outer.iterrows():
                mask = frame.plotID.isin(str(selection.test_plots).split(';'))
                frame.loc[mask, 'selected_score_threshold'] = selection.score_threshold
                frame.loc[mask, 'outer_fold'] = selection.outer_fold
        plotrows.append(frame)
    summary = pd.DataFrame(rows).sort_values(['resolution_cm', 'setting', 'siteID']).reset_index(drop=True)
    per_plot = pd.concat(plotrows, ignore_index=True).sort_values(['plotID', 'resolution_cm', 'setting'])
    assert len(summary) == 3 * 2 * 3 and len(per_plot) == 23 * 3 * 2
    assert per_plot.groupby('plotID').C_ref_tC_ha.nunique().eq(1).all()
    summary.to_csv(OUT / 'resolution_comparison.csv', index=False)
    per_plot.to_csv(OUT / 'resolution_plot_comparison.csv', index=False)
    return summary, per_plot


def resolution_validate_10cm(summary):
    base = RESOLUTION_BASE_SUMMARY.query("method == 'A_DF_nested_OOF' and siteID == 'ALL'").iloc[0]
    current = summary.query("resolution_cm == 10 and setting == 'fixed' and siteID == 'ALL'").iloc[0]
    rows = []
    for key, target, tolerance in [('F1_visible', .55, .02), ('estimate_reference_ratio', .68, .02)]:
        delta = float(current[key] - base[key]); passed = abs(delta) <= 1e-8
        approximate = abs(float(current[key]) - target) <= tolerance
        rows.append(dict(metric=key, cached_A=float(base[key]), fixed_10cm=float(current[key]),
                         difference=delta, matches_cached_A=passed, approximate_expected=target,
                         within_approximate_tolerance=approximate))
        if not passed or not approximate:
            warnings.warn(f'10cm 고정 결과와 기존 A/예상값 차이: {rows[-1]}')
    pd.DataFrame(rows).to_csv(OUT / 'resolution_10cm_validation.csv', index=False)
    print('10cm 고정 vs 기존 A 자동 비교 (각 outer fold600/.2 동일 확인):')
    display(pd.DataFrame(rows))
    return rows


def resolution_figures(summary, results, images):
    figures = OUT / 'figures'; figures.mkdir(exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for setting, label, color in [('fixed', 'Fixed: score .2 / patch600', '#3877a8'),
                                  ('retuned', 'Retuned: nested OOF / patch600', '#df8639')]:
        q = summary[(summary.siteID == 'ALL') & (summary.setting == setting)].sort_values('resolution_cm')
        for ax, y in zip(axes, ('F1_visible', 'estimate_reference_ratio')):
            ax.plot(q.resolution_cm, q[y], 'o-' if setting == 'fixed' else 's--', label=label, color=color)
            ax.set_xticks([10, 25, 51]); ax.set_xlabel('Nominal RGB GSD (cm)'); ax.grid(alpha=.2)
    axes[0].set_ylabel('Visible-tree F1'); axes[0].set_ylim(0, 1)
    axes[1].set_ylabel('Mean estimated / mean reference carbon'); axes[1].axhline(1, ls=':', c='gray')
    axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
    fig.suptitle('Same 23 plots; CHM held at 1m; RGB expanded to original 10cm grid')
    fig.tight_layout(); fig.savefig(figures / 'resolution_trend.png', dpi=170, bbox_inches='tight'); plt.close(fig)
    # Three columns; rows distinguish fixed settings from honest outer-OOF retuning.
    for pid in RESOLUTION_OVERLAYS:
        fig, axes = plt.subplots(2, 3, figsize=(17, 11), sharex=True, sharey=True)
        p = plots.set_index('plotID').loc[pid]
        for column, cm in enumerate(RESOLUTION_GSDS):
            rgb = images[cm][pid]
            for row, setting in enumerate(('fixed', 'retuned')):
                ax = axes[row, column]
                with rasterio.open(rgb) as ds:
                    show(ds, ax=ax); crs = ds.crs
                f = field_geo[field_geo.plotID == pid].to_crs(crs)
                d = results[cm, setting]['detections']; d = d[d.plotID == pid].to_crs(crs)
                if len(d): d.boundary.plot(ax=ax, color='cyan', linewidth=.8)
                for visible, color in [(True, 'lime'), (False, 'magenta')]:
                    z = f[f.visible == visible]
                    ax.scatter(z.geometry.x, z.geometry.y, c=color, s=19, edgecolors='black', linewidths=.35, zorder=4)
                footprint = gpd.GeoSeries([p.geometry], crs=WORK_CRS).to_crs(crs)
                footprint.boundary.plot(ax=ax, color='yellow', linewidth=1.5)
                bounds = footprint.total_bounds; padding = 8
                ax.set_xlim(bounds[0] - padding, bounds[2] + padding); ax.set_ylim(bounds[1] - padding, bounds[3] + padding)
                score = .2 if setting == 'fixed' else float(d.selected_score_threshold.iloc[0]) if len(d) and 'selected_score_threshold' in d else None
                ax.set_title(f'{cm}cm | {setting} | n={len(d)}' + (f' | score={score:g}' if score is not None else ''))
                ax.ticklabel_format(useOffset=False, style='plain'); ax.tick_params(labelsize=7)
        fig.suptitle(pid + ' | RGB resolution only; same CHM and footprints')
        fig.text(.5, .012, 'Cyan: detections | Green: visible field stems | Pink: occluded field stems | Yellow: plot boundary | Rows: fixed / nested OOF', ha='center', fontsize=10)
        fig.tight_layout(rect=(0, .035, 1, .96))
        fig.savefig(figures / f'{pid}_resolution_side_by_side.png', dpi=160, bbox_inches='tight'); plt.close(fig)


def resolution_report(summary, validation):
    columns = ['resolution_cm', 'setting', 'siteID', 'n_detected', 'F1_visible', 'F1_all',
               'C_est_mean', 'C_ref_mean', 'estimate_reference_ratio', 'carbon_RMSE', 'selected_score_threshold']
    table = summary[columns].to_markdown(index=False, floatfmt='.3f')
    all_sites = summary[summary.siteID == 'ALL'].set_index(['resolution_cm', 'setting'])
    changes = []
    for setting in ('fixed', 'retuned'):
        base = all_sites.loc[10, setting]
        for cm in (25, 51):
            q = all_sites.loc[cm, setting]
            changes.append(f'- {setting} {cm}cm: 10cm 대비 F1_visible {q.F1_visible-base.F1_visible:+.3f}, '
                           f'탄소 추정/참조 {q.estimate_reference_ratio-base.estimate_reference_ratio:+.3f}, '
                           f'탄소 RMSE {q.carbon_RMSE-base.carbon_RMSE:+.3f}tC/ha.')
    intro = '\n'.join(changes)
    model_file = RESOLUTION_CACHE / 'model/resolved_model.json'
    if model_file.exists(): shutil.copy2(model_file, OUT / 'resolution_model_provenance.json')
    section = f'''<!-- resolution-experiment:start -->
# 해상도 실험

원본10cm RGB를 면적 평균(Resampling.average)으로25/51cm로 줄이고, DeepForest 입력은
bilinear로 원본10cm 격자에 확대했다. **확대해도 원래 선명도와 세부 정보는 늘지 않는다.**
CHM은 동일한 원본1m 파일을 사용했다. H=상자 내부 CHM최대, CD=상자 가로·세로 평균,
CHM≥3m, Jucker biome11, Chojnacky 일반활엽수 대체식, CF0.5, point-in-box 헝가리안 매칭을 유지했다.

## 결과 요약

탄소 평균·RMSE 단위는 tC/ha, 높이·수관 Bias는m, DBH Bias는cm, 개체AGB Bias는kg이다.
탄소 비율은 유효 조사구의 평균 추정/평균 참조다. 상세 Bias·precision/recall은 resolution_comparison.csv 참고.

{table}

{intro}

## 실행·비교 검증

- GRSM15+ORNL8, 총23개 조사구 고정. 조사구 참조 총량에는 좌표 없는 나무를 포함했다.
  F1_all의 모집단은 기존과 동일하게 매칭 좌표가 있는 전체 나무이며, 좌표 없는 나무는 F1 분모로 만들지 않았다.
- 10cm baseline_015/A/B/C는 기존 결과를 재사용했다. 기존 A의5개 outer fold가 모두600/.2이고,
  patch600 안쪽 탐색에서도 같은 score가 최적임을 확인해10cm 고정/재선택 행에 그대로 재사용했다.
- 10cm 고정 결과 vs 기존A: {json.dumps(validation, ensure_ascii=False)}.
- 25/51cm 고정은 score.2, 재선택은 patch600에서.1/.2/.3/.4/.5를 안쪽 보이는 나무F1로 선택했다.
  바깥 평가 조사구의 정답은 선택에 쓰지 않았다. fold별 임계값과 train/test목록을 별도CSV로 저장했다.
  deployment_score_threshold는 전 조사구 선택값으로, OOF 평가에 역으로 적용하지 않았다.
- 실제 파일 CRS·bounds·GSD와 원본/확대본transform·shape의 일치를 검사했다.
  coarse 격자는 원본 범위와 좌상단을 보존하고 정수 크기로 만들므로 실제 GSD는25/51cm와 소폭 다를 수 있다.
  nominal과 실제x/y GSD는 resolution_raster_metadata.csv에 모두 기록했다.
- Drive crop만 읽고 새 RGB·CHM 다운로드는 수행하지 않았다. 줄인RGB/10cm 확대본/탐지 후보는Drive캐시에 보존했다.
- QUDO 충돌 수종계수 제외, gzip 응답 크기검사 예외, ORNL2021 provisional, HF_HOME=/content/hf_cache를 유지했다.

## 해석상 한계

- 인위적 평균 축소 영상은 실제25/51cm 촬영의 센서·광학특성·압축·촬영각·노출·계절과 다를 수 있다.
- CHM을 고정했으므로 RGB 해상도 변화만 분리한 실험이며, 국내 CHM의 오차/해상도 영향은 평가하지 않는다.
  상자가 달라지면 같은 CHM에서도 추출 최대 수고와 CD가 달라진다.
- 이23개 조사구에서 배포 설정이 이미 선택되었으므로 고정 설정의10cm 점수는 독립적인 외부 검증이 아니다.
  NEON 사전학습 모델·지역·수종·산림구조가 국내 영상과 달라 국내 성능을 보장하지 않는다.
- 기존3차 모델은 revision=main으로만 기록되었다. 새 추론은 동일DeepForest버전과 실험 첫 실행 때 고정한
  HF커밋을 사용하나, 과거 가중치 SHA가 없어 당시와 동일한 가중치인지 직접 검증할 수 없다.
  resolution_model_provenance.json을 보존한다. 재실행 시 새해상도들 사이의 커밋은 바꾸지 않는다.
- 실측 기반NEONForestAGB 역시 상대생장식 참조 추정값이다. 파괴조사 무게 정답이나 수관IoU 검증이 아니다.

[Rasterio 리샘플링 문서](https://rasterio.readthedocs.io/en/stable/topics/resampling.html)

그림: figures/resolution_trend.png 및 세 *_resolution_side_by_side.png.
기존 결과 출처: {RESOLUTION_BASE}; 해상도 실험ID: {RUN_ID}.
<!-- resolution-experiment:end -->

'''
    previous = (RESOLUTION_BASE / 'report.md').read_text(encoding='utf-8')
    if '<!-- resolution-experiment:end -->' in previous:
        previous = previous.split('<!-- resolution-experiment:end -->', 1)[1].lstrip()
    (OUT / 'report.md').write_text(section + previous, encoding='utf-8')
    pointer = ROOT / 'outputs' / ('resolution_latest.md' if RESOLUTION_BASE.resolve() == (ROOT/'outputs').resolve() else 'report.md')
    pointer.write_text(f'최근 완료 결과: [{RUN_ID}/report.md]({RUN_ID}/report.md)\n', encoding='utf-8')


def resolution_completed_outputs():
    outputs = ['resolution_comparison.csv', 'resolution_plot_comparison.csv',
            'resolution_raster_metadata.csv', 'resolution_10cm_validation.csv', 'report.md',
            'figures/resolution_trend.png'] + [f'figures/{pid}_resolution_side_by_side.png' for pid in RESOLUTION_OVERLAYS]
    for cm in RESOLUTION_GSDS:
        outputs += [f'resolution/{cm}cm/{name}' for name in ('outer_groupkfold.csv', 'inner_grid_scores.csv',
                    'fold_plot_assignments.csv', 'deployment_grid.csv')]
        for setting in ('fixed', 'retuned'):
            outputs += [f'resolution/{cm}cm/{setting}/{name}' for name in ('detected_trees.csv',
                        'matched_trees_visible.csv', 'detection_metrics.csv', 'plot_comparison.csv',
                        'matches_visible.csv', 'matches_all.csv')]
    return outputs


def run_resolution_experiment():
    global RESOLUTION_SUMMARY, RESOLUTION_PLOT_COMPARISON, RESOLUTION_RESULTS
    resolution_load_state()
    completed = OUT / 'resolution_completed.json'
    if completed.is_file():
        record = json.loads(completed.read_text())
        if record['base_signature'] == RESOLUTION_BASE_SIGNATURE and all(
            (OUT / name).is_file() and resolution_sha(OUT / name) == digest
            for name, digest in record['outputs'].items()
        ):
            RESOLUTION_SUMMARY = pd.read_csv(OUT / 'resolution_comparison.csv')
            RESOLUTION_PLOT_COMPARISON = pd.read_csv(OUT / 'resolution_plot_comparison.csv')
            print('완료된 동일 실험 재사용: RGB 재생성·GPU 추론·CV/평가 재계산 없이 저장 결과를 불러왔습니다.')
            display(pd.read_csv(OUT / 'resolution_10cm_validation.csv'))
            return RESOLUTION_SUMMARY
    resolution_runtime_checks()
    results = {}; selections = {}; metadata = []
    images = {10: {pid: paths[0] for pid, paths in RASTERS.items()}}
    cached, outer, inner, splits, deployment = resolution_cached_10cm()
    for setting in ('fixed', 'retuned'): results[10, setting] = cached
    selections[10] = dict(outer=outer, inner=inner, splits=splits, deployment=deployment)
    for cm in (25, 51):
        frames = []; images[cm] = {}
        for _, p in plots.iterrows():
            rgb, chm = RASTERS[p.plotID]
            model_input, coarse, meta = resolution_rgb(rgb, cm, RESOLUTION_CACHE / 'rgb' / f'{cm}cm' / p.plotID)
            images[cm][p.plotID] = model_input
            meta.update(plotID=p.plotID, resolution_cm=cm, coarse_path=str(coarse), model_input_path=str(model_input),
                        chm_path=str(chm), chm_sha256=resolution_sha(chm), chm_gsd_m=1.0)
            metadata.append(meta)
            print(f'{p.plotID} {cm}cm: actual GSD={meta["actual_gsd_x_m"]:.6f} × '
                  f'{meta["actual_gsd_y_m"]:.6f}m; input=10cm original grid; CHM=1m unchanged')
            frames.append(resolution_candidates(p, cm, model_input, chm))
        candidates = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry='geometry', crs=WORK_CRS)
        grid = {(600, t): candidates[candidates.score >= t].copy() for t in RESOLUTION_THRESHOLDS}
        oof, outer, inner, splits, deployment = nested_group_parameter_selection(
            grid, plots, field_geo[field_geo.visible], CFG['cv_folds'])
        reference_splits = selections[10]['splits'].sort_values(['outer_fold', 'plotID', 'role']).reset_index(drop=True)
        pd.testing.assert_frame_equal(splits.sort_values(['outer_fold', 'plotID', 'role']).reset_index(drop=True), reference_splits)
        selections[cm] = dict(outer=outer, inner=inner, splits=splits, deployment=deployment)
        results[cm, 'fixed'] = evaluate_detection_method(grid[600, .2], plots, field_all, field_geo)
        results[cm, 'retuned'] = evaluate_detection_method(oof, plots, field_all, field_geo)
    for pid, rgb in images[10].items():
        with rasterio.open(rgb) as ds:
            metadata.append(dict(plotID=pid, resolution_cm=10, nominal_cm=10,
                actual_gsd_x_m=ds.res[0], actual_gsd_y_m=ds.res[1], bounds=list(ds.bounds),
                input_transform=list(ds.transform), input_width=ds.width, input_height=ds.height,
                coarse_path=str(rgb), model_input_path=str(rgb), crs=ds.crs.to_string(),
                chm_path=str(RASTERS[pid][1]), chm_sha256=resolution_sha(RASTERS[pid][1]), chm_gsd_m=1.0,
                downsampling='none: original 10cm', upsampling='none'))
    pd.DataFrame(metadata).to_csv(OUT / 'resolution_raster_metadata.csv', index=False)
    for cm, selection in selections.items():
        folder = OUT / 'resolution' / f'{cm}cm'; folder.mkdir(parents=True, exist_ok=True)
        for key, filename in [('outer', 'outer_groupkfold.csv'), ('inner', 'inner_grid_scores.csv'),
                              ('splits', 'fold_plot_assignments.csv'), ('deployment', 'deployment_grid.csv')]:
            selection[key].to_csv(folder / filename, index=False)
        for setting in ('fixed', 'retuned'):
            if cm == 10:
                # Byte-for-byte copies; no old inference, evaluation or carbon recomputation.
                shutil.copytree(RESOLUTION_BASE / 'methods/A_DF_nested_OOF', folder / setting, dirs_exist_ok=True)
            else:
                resolution_save_result(results[cm, setting], folder / setting)
    summary, per_plot = resolution_summaries(results, selections)
    validation = resolution_validate_10cm(summary)
    resolution_figures(summary, results, images)
    resolution_report(summary, validation)
    # Confirm old CSVs have not been mutated by evaluation/plotting or copying.
    provenance = json.loads((OUT / 'resolution_provenance.json').read_text())
    for name, digest in provenance['source_files'].items():
        assert resolution_sha(RESOLUTION_BASE / name) == digest, f'기존 결과 변경 감지: {name}'
    RESOLUTION_SUMMARY = summary; RESOLUTION_PLOT_COMPARISON = per_plot; RESOLUTION_RESULTS = results
    resolution_json(completed, dict(base_signature=RESOLUTION_BASE_SIGNATURE,
                    outputs={name: resolution_sha(OUT / name) for name in resolution_completed_outputs()}))
    return summary

## 17B. 10·25·51cm × 고정·nested OOF 비교 실행

25/51cm에서만 추론합니다. patch600을 고정하고 score.1/.2/.3/.4/.5를 같은GroupKFold로 선택합니다.
10cm 고정 결과의F1·탄소 비율을 기존A와 자동 비교하고, 다르면 경고합니다.
실제 수치·18행 사이트 비교표·138행 조사구 비교표·추세도·요청한3개 나란히 그림을 저장합니다.
중간 탐지 후보까지 Drive에 저장하므로 재실행하면 가중치 로딩과GPU 추론도 건너뛸 수 있습니다.

In [ ]:
RESOLUTION_SUMMARY = run_resolution_experiment()

## 18. 재사용 함수 모듈 내보내기

탐지·공식·매칭·회귀·평가 함수를 `neon_pipeline.py`로 저장합니다.
다른 Colab에서 모듈을 불러온 후 `configure(설정dict, 결과폴더, 작업CRS)`를 호출하세요.
`estimate_dbh`와 `estimate_agb`에는 공식 계수를 직접 전달할 수도 있습니다.

해상도 실험 기본 실행에서는 기존 처리·다운로드를 건너뜁니다. 함수 정의만 재사용합니다.


In [ ]:
if not RESOLUTION_ONLY:
    (OUT/'neon_pipeline.py').write_text('"""Reusable inference/evaluation functions. Requires Colab scientific dependencies.\nCall configure(config, output_directory, projected_crs) before inference.\nDBH inputs: H, CD in metres. AGB input DBH in centimetres; output kilograms.\n"""\nfrom pathlib import Path\nimport inspect\nimport math\nimport json, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport geopandas as gpd\nimport rasterio\nfrom rasterio.merge import merge\nfrom rasterio.mask import mask\nfrom shapely.geometry import Point, box, Polygon\nfrom shapely.ops import unary_union\nfrom scipy.optimize import linear_sum_assignment\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.linear_model import LinearRegression\nimport matplotlib.pyplot as plt\nfrom scipy.ndimage import gaussian_filter, maximum_filter, label as ndi_label\nfrom skimage.segmentation import watershed\nfrom rasterio.features import shapes as raster_shapes\nfrom shapely.geometry import shape\nAUDIT=[]\ndef configure(config, output_directory, projected_crs):\n    global CFG, OUT, WORK_CRS, default\n    CFG=config\n    OUT=Path(output_directory)\n    OUT.mkdir(parents=True,exist_ok=True)\n    WORK_CRS=projected_crs\n    default=config[\'agb_default\']\ndef audit(stage,item,reason):\n    AUDIT.append(dict(stage=stage,item=str(item),reason=str(reason)))\n\n\ndef require_columns(df, cols, label):\n    missing = set(cols) - set(df.columns)\n    if missing:\n        raise ValueError(f\'{label} 필드 누락: {missing}; 실제 필드={list(df.columns)}\')\n\ndef offset_xy(e, n, distance, azimuth_deg):\n    rad = np.deg2rad(azimuth_deg)\n    return (e + distance * np.sin(rad), n + distance * np.cos(rad))\n\ndef estimate_dbh(H_m, CD_m, coefficients=None):\n    c = coefficients or CFG[\'jucker\']\n    H = np.asarray(H_m, dtype=float)\n    CD = np.asarray(CD_m, dtype=float)\n    return np.where((H > 0) & (CD > 0), c[\'a\'] * (H * CD) ** c[\'b\'] * np.exp(c.get(\'g\', 0) ** 2 / 2), np.nan)\n\ndef estimate_agb(dbh_cm, beta0=None, beta1=None):\n    b0 = default[\'beta0\'] if beta0 is None else beta0\n    b1 = default[\'beta1\'] if beta1 is None else beta1\n    D = np.asarray(dbh_cm, dtype=float)\n    with np.errstate(invalid=\'ignore\', divide=\'ignore\'):\n        return np.where(D > 0, np.exp(b0 + b1 * np.log(D)), np.nan)\n\ndef kg_to_t_ha(kg, area_m2):\n    return np.asarray(kg) / 1000 * 10000 / np.asarray(area_m2)\n\ndef carbon_kg(agb_kg, fraction=0.5):\n    return np.asarray(agb_kg) * fraction\n\ndef tile_coordinates(centers, buffer_m=50):\n    tiles = set()\n    for x, y in centers:\n        for e in range(int((x - buffer_m) // 1000), int((x + buffer_m) // 1000) + 1):\n            for n in range(int((y - buffer_m) // 1000), int((y + buffer_m) // 1000) + 1):\n                tiles.add((e * 1000, n * 1000))\n    return sorted(tiles)\n\ndef product_site_entry(product, site):\n    records = product.get(\'data\', product).get(\'siteCodes\', [])\n    return next((r for r in records if r.get(\'siteCode\') == site), dict(siteCode=site, availableMonths=[], availableReleases=[]))\n\ndef select_aop_pair(rgb_entry, chm_entry, target_year, allow_adjacent=False, priority=(2022, 2020), include_provisional=True):\n    common = sorted(set(rgb_entry.get(\'availableMonths\', [])) & set(chm_entry.get(\'availableMonths\', [])))\n    years = [int(target_year)] + ([int(y) for y in priority] if allow_adjacent else [])\n    selected = next((y for y in years if any((m.startswith(str(y) + \'-\') for m in common))), None)\n    if selected is None:\n        raise ValueError(f\'RGB/CHM 공통 연월 없음: 요청 연도 {years}; 공통 월 {common}\')\n    months = [m for m in common if m.startswith(str(selected) + \'-\')]\n    provisional = False\n    for entry in [rgb_entry, chm_entry]:\n        stable = {m for r in entry.get(\'availableReleases\', []) if r.get(\'release\') != \'PROVISIONAL\' for m in r.get(\'availableMonths\', [])}\n        prov = {m for r in entry.get(\'availableReleases\', []) if r.get(\'release\') == \'PROVISIONAL\' for m in r.get(\'availableMonths\', [])}\n        provisional |= any((m in prov and m not in stable for m in months))\n    if provisional and (not include_provisional):\n        raise ValueError(\'선택 연도에 PROVISIONAL 자료가 있습니다. aop_include_provisional=True 필요; 다른 해로 숨겨 대체하지 않습니다.\')\n    return dict(imagery_year=selected, common_months=months, include_provisional=bool(provisional), provisional=bool(provisional), adjacent_imagery=selected != int(target_year))\n\ndef aop_tiff_files(folder):\n    return sorted((p for p in Path(folder).rglob(\'*\') if p.is_file() and p.suffix.lower() in [\'.tif\', \'.tiff\']))\n\ndef raster_index(folder, expected_resolution):\n    rows = []\n    for path in sorted(Path(folder).rglob(\'*\')):\n        if path.suffix.lower() not in [\'.tif\', \'.tiff\']:\n            continue\n        with rasterio.open(path) as ds:\n            if not ds.crs or not ds.crs.is_projected:\n                raise ValueError(f\'미터 투영 CRS 필요: {path}\')\n            if not np.allclose(ds.res, expected_resolution, atol=expected_resolution * 0.05):\n                audit(\'resolution\', path, f\'예상 {expected_resolution}; 실제 {ds.res}\')\n                continue\n            rows.append(dict(path=str(path), crs=ds.crs, geometry=box(*ds.bounds)))\n    return rows\n\ndef crop_mosaic(index, region, region_crs, dest, bands):\n    if dest.exists():\n        return dest\n    datasets = []\n    coverage = []\n    try:\n        for r in index:\n            geom = gpd.GeoSeries([region], crs=region_crs).to_crs(r[\'crs\']).iloc[0]\n            if r[\'geometry\'].intersects(geom):\n                datasets.append(rasterio.open(r[\'path\']))\n                coverage.append(r[\'geometry\'])\n        if not datasets:\n            raise ValueError(\'해당 위치를 덮는 타일 없음\')\n        crs = datasets[0].crs\n        if any((d.crs != crs for d in datasets)):\n            raise ValueError(\'mosaic 입력 CRS 불일치\')\n        geom = gpd.GeoSeries([region], crs=region_crs).to_crs(crs).iloc[0]\n        if not unary_union(coverage).buffer(0.2).covers(geom):\n            raise ValueError(\'조사구 주변 영상 coverage 불완전\')\n        data, trans = merge(datasets, bounds=geom.bounds, indexes=bands)\n        profile = datasets[0].profile.copy()\n        profile.update(driver=\'GTiff\', height=data.shape[1], width=data.shape[2], count=len(bands), transform=trans, compress=\'deflate\', tiled=True, blockxsize=256, blockysize=256)\n        with rasterio.open(dest, \'w\', **profile) as out:\n            out.write(data)\n    finally:\n        for d in datasets:\n            d.close()\n    return dest\n\ndef raster_covers_geometry(index, geometry, geometry_crs):\n    if not index:\n        return False\n    bounds = [gpd.GeoSeries([r[\'geometry\']], crs=r[\'crs\']).to_crs(geometry_crs).iloc[0] for r in index]\n    return unary_union(bounds).buffer(0.2).covers(geometry)\n\ndef zonal_max(path, geom, geom_crs):\n    with rasterio.open(path) as ds:\n        g = gpd.GeoSeries([geom], crs=geom_crs).to_crs(ds.crs).iloc[0]\n        try:\n            arr, _ = mask(ds, [g], crop=True, filled=False, indexes=1)\n        except ValueError:\n            return np.nan\n        v = arr.compressed()\n        v = v[np.isfinite(v) & (v >= 0)]\n        return float(v.max()) if len(v) else np.nan\n\ndef predict_crowns(m, rgb, chm, plot):\n    cache = OUT / f\'{plot.plotID}_raw_predictions.csv\'\n    if cache.exists():\n        raw = pd.read_csv(cache)\n    else:\n        params = inspect.signature(m.predict_tile).parameters\n        path_key = \'path\' if \'path\' in params else \'raster_path\'\n        args = {path_key: str(rgb), \'patch_size\': CFG[\'patch_size\'], \'patch_overlap\': CFG[\'patch_overlap\']}\n        if \'iou_threshold\' in params:\n            args[\'iou_threshold\'] = CFG[\'nms_iou\']\n        raw = m.predict_tile(**args)\n        if isinstance(raw, tuple):\n            raw = raw[0]\n        if raw is None or len(raw) == 0:\n            raw = pd.DataFrame(columns=[\'xmin\', \'ymin\', \'xmax\', \'ymax\', \'score\'])\n        raw.to_csv(cache, index=False)\n    require_columns(raw, [\'xmin\', \'ymin\', \'xmax\', \'ymax\', \'score\'], \'DeepForest predictions\')\n    rows = []\n    with rasterio.open(rgb) as ds:\n        trans = ds.transform\n        crs = ds.crs\n        for idx, r in raw[raw.score >= CFG[\'score_threshold\']].iterrows():\n            corners = [trans * (x, y) for x, y in [(r.xmin, r.ymin), (r.xmax, r.ymin), (r.xmax, r.ymax), (r.xmin, r.ymax)]]\n            geom = Polygon(corners)\n            geom = gpd.GeoSeries([geom], crs=crs).to_crs(WORK_CRS).iloc[0]\n            if not plot.geometry.covers(geom.centroid):\n                continue\n            h = zonal_max(chm, geom, WORK_CRS)\n            if not np.isfinite(h):\n                audit(\'CHM_nodata\', f\'{plot.plotID}_{idx}\', \'CHM 없음\')\n                continue\n            if h < CFG[\'min_height_m\']:\n                continue\n            width = (r.xmax - r.xmin) * abs(trans.a)\n            length = (r.ymax - r.ymin) * abs(trans.e)\n            rows.append(dict(det_id=f\'{plot.plotID}_{idx}\', plotID=plot.plotID, siteID=plot.siteID, field_year=plot.field_year, score=r.score, xmin_px=r.xmin, ymin_px=r.ymin, xmax_px=r.xmax, ymax_px=r.ymax, H_m=h, CD_m=(width + length) / 2, width_m=width, length_m=length, geometry=geom))\n    return rows\n\ndef label_visibility(f, rasters):\n    f = f.copy()\n    local = []\n    for _, r in f.iterrows():\n        local.append(zonal_max(rasters[r.plotID][1], r.geometry.buffer(CFG[\'visible_radius_m\']), WORK_CRS))\n    f[\'local_CHM_m\'] = local\n    canopy = f.get(\'canopyPosition\', pd.Series(\'\', index=f.index)).fillna(\'\').str.strip().str.lower()\n    f[\'visible_by_position\'] = canopy.isin([\'full sun\', \'open grown\', \'partially shaded\'])\n    f[\'visible_by_height\'] = f.field_H_m.ge(CFG[\'visible_height_ratio\'] * f.local_CHM_m) & f.local_CHM_m.gt(0)\n    f[\'visible\'] = f.visible_by_position | f.visible_by_height\n    return f\n\ndef match_one_to_one(det, truth):\n    if det.empty or truth.empty:\n        return pd.DataFrame(columns=[\'det_id\', \'individualID\', \'plotID\', \'distance_m\'])\n    d = det.reset_index(drop=True)\n    t = truth.reset_index(drop=True)\n    n, m = (len(d), len(t))\n    weights = np.zeros((n, m + n))\n    valid = np.zeros((n, m), dtype=bool)\n    sizes = pd.to_numeric(t[CFG[\'match_size\']], errors=\'coerce\').fillna(0).to_numpy()\n    rank = pd.Series(sizes).rank(method=\'dense\').to_numpy() / (m + 1)\n    for i, r in d.iterrows():\n        for j, s in t.iterrows():\n            if r.geometry.covers(s.geometry):\n                valid[i, j] = True\n                distance = r.geometry.centroid.distance(s.geometry)\n                weights[i, j] = (n + 1) * 10 + rank[j] + 1e-06 / (1 + distance)\n    ri, ci = linear_sum_assignment(-weights)\n    rows = []\n    for i, j in zip(ri, ci):\n        if j < m and valid[i, j]:\n            rows.append(dict(det_id=d.iloc[i].det_id, individualID=t.iloc[j].individualID, plotID=d.iloc[i].plotID, distance_m=d.iloc[i].geometry.centroid.distance(t.iloc[j].geometry)))\n    return pd.DataFrame(rows, columns=[\'det_id\', \'individualID\', \'plotID\', \'distance_m\'])\n\ndef grouped_dbh_regression(df, n_splits=5):\n    result = pd.Series(np.nan, index=df.index)\n    fold_id = pd.Series(np.nan, index=df.index)\n    good = df[[\'H_m\', \'CD_m\', \'field_DBH_cm\']].gt(0).all(axis=1)\n    good &= np.isfinite(df[[\'H_m\', \'CD_m\', \'field_DBH_cm\']]).all(axis=1)\n    z = df.loc[good]\n    groups = z.plotID\n    if groups.nunique() < 2:\n        audit(\'regression\', \'GroupKFold\', \'독립 조사구 2개 미만\')\n        return (result, fold_id, None)\n    X = np.log(z[[\'H_m\', \'CD_m\']].to_numpy())\n    y = np.log(z.field_DBH_cm.to_numpy())\n    folds = GroupKFold(n_splits=min(n_splits, groups.nunique()))\n    for fold, (tr, te) in enumerate(folds.split(X, y, groups)):\n        if len(tr) < 4:\n            audit(\'regression\', fold, \'학습 나무 4개 미만\')\n            continue\n        m = LinearRegression().fit(X[tr], y[tr])\n        smear = float(np.mean(np.exp(y[tr] - m.predict(X[tr]))))\n        result.loc[z.index[te]] = np.exp(m.predict(X[te])) * smear\n        fold_id.loc[z.index[te]] = fold\n    m = LinearRegression().fit(X, y)\n    fit = dict(intercept=float(m.intercept_), log_H=float(m.coef_[0]), log_CD=float(m.coef_[1]), smearing=float(np.mean(np.exp(y - m.predict(X)))), n=len(z), n_plots=groups.nunique())\n    return (result, fold_id, fit)\n\ndef errors(truth, pred):\n    a = np.asarray(truth, dtype=float)\n    b = np.asarray(pred, dtype=float)\n    ok = np.isfinite(a) & np.isfinite(b)\n    a = a[ok]\n    b = b[ok]\n    if not len(a):\n        return dict(n=0, MAE=np.nan, RMSE=np.nan, Bias=np.nan, r=np.nan)\n    e = b - a\n    r = np.corrcoef(a, b)[0, 1] if len(a) > 1 and np.std(a) > 0 and (np.std(b) > 0) else np.nan\n    return dict(n=len(a), MAE=float(np.mean(abs(e))), RMSE=float(np.sqrt(np.mean(e ** 2))), Bias=float(np.mean(e)), r=r)\n\ndef plot_biomass_totals(detections_all, truth_all, truth_coord, pairs, plot, carbon_fraction=0.5, carbon_alternative=0.47):\n\n    def number(value):\n        try:\n            return float(value)\n        except (ValueError, TypeError):\n            return float(\'nan\')\n\n    def sum_valid(rows, column):\n        values = [number(r.get(column)) for r in rows]\n        return sum(values) if all((math.isfinite(v) for v in values)) else float(\'nan\')\n    area = number(plot.get(\'reported_area_m2\'))\n    if not math.isfinite(area) or area <= 0:\n        area = number(plot.get(\'area_m2\'))\n    if not math.isfinite(area) or area <= 0:\n        raise ValueError(\'양수 조사면적 필요\')\n    scale = lambda value: value / 1000 * 10000 / area\n    by_id = {r[\'individualID\']: r for r in truth_all}\n    if len(by_id) != len(truth_all):\n        raise ValueError(\'조사구 참조 개체 중복\')\n    det_by_id = {r[\'det_id\']: r for r in detections_all}\n    paired_truth = {r[\'individualID\'] for r in pairs}\n    paired_det = {r[\'det_id\'] for r in pairs}\n    coord_ids = {r[\'individualID\'] for r in truth_coord}\n    if not paired_truth <= by_id.keys() or not paired_det <= det_by_id.keys():\n        raise ValueError(\'매칭 ID가 조사구 모집단에 없음\')\n    reference = sum_valid(truth_all, \'ref_AGB_kg\')\n    pred = sum_valid(detections_all, \'AGB_default_kg\')\n    coord_reference = sum_valid(truth_coord, \'ref_AGB_kg\')\n    missing_agb = sum((not math.isfinite(number(r.get(\'ref_AGB_kg\'))) for r in truth_all))\n    matched_ref = sum_valid([by_id[k] for k in paired_truth], \'ref_AGB_kg\')\n    matched_pred = sum_valid([det_by_id[k] for k in paired_det], \'AGB_default_kg\')\n    unmatched_pred = sum_valid([r for r in detections_all if r[\'det_id\'] not in paired_det], \'AGB_default_kg\')\n    missed_ref = sum_valid([r for r in truth_all if r[\'individualID\'] not in paired_truth], \'ref_AGB_kg\')\n    no_coord = sum_valid([r for r in truth_all if r[\'individualID\'] not in coord_ids], \'ref_AGB_kg\')\n    missed_coord = sum_valid([r for r in truth_coord if r[\'individualID\'] not in paired_truth], \'ref_AGB_kg\')\n    field_year = int(plot[\'field_year\'])\n    imagery_year = int(plot.get(\'imagery_year\', field_year))\n    row = dict(plotID=plot[\'plotID\'], siteID=plot[\'siteID\'], field_year=field_year, imagery_year=imagery_year, same_year=field_year == imagery_year, imagery_provisional=plot.get(\'imagery_provisional\', False), area_m2=area, reported_area_m2=number(plot.get(\'reported_area_m2\', area)), polygon_area_m2=number(plot.get(\'polygon_area_m2\', area)), area_difference_m2=number(plot.get(\'polygon_area_m2\', area)) - area, n_truth=len(truth_all), n_truth_coord=len(truth_coord), n_detected=len(detections_all), n_matched=len(pairs), n_field_before_coordinate_filter=len(truth_all), n_without_matching_coordinate=len(truth_all) - len(truth_coord), n_missing_ref_AGB=missing_agb, AGB_reference_coverage=1 - missing_agb / len(truth_all) if truth_all else 1.0, reference_complete=missing_agb == 0, coordinates_complete=len(truth_all) == len(truth_coord), AGB_est_t_ha=scale(pred), AGB_ref_t_ha=scale(reference), AGB_ref_coordonly_t_ha=scale(coord_reference), matched_error_t_ha=scale(matched_pred - matched_ref), unmatched_detection_t_ha=scale(unmatched_pred), missed_truth_t_ha=scale(missed_ref), unmapped_truth_t_ha=scale(no_coord), missed_mapped_truth_t_ha=scale(missed_coord))\n    row[\'C_est_tC_ha\'] = row[\'AGB_est_t_ha\'] * carbon_fraction\n    row[\'C_ref_tC_ha\'] = row[\'AGB_ref_t_ha\'] * carbon_fraction\n    row[\'C_ref_coordonly_tC_ha\'] = row[\'AGB_ref_coordonly_t_ha\'] * carbon_fraction\n    row[\'C_est_047_tC_ha\'] = row[\'AGB_est_t_ha\'] * carbon_alternative\n    row[\'C_ref_047_tC_ha\'] = row[\'AGB_ref_t_ha\'] * carbon_alternative\n    row[\'C_bias_tC_ha\'] = row[\'C_est_tC_ha\'] - row[\'C_ref_tC_ha\']\n    row[\'C_missed_tC_ha\'] = row[\'missed_truth_t_ha\'] * carbon_fraction\n    row[\'estimate_reference_ratio\'] = pred / reference if reference > 0 else float(\'nan\')\n    if math.isfinite(reference) and math.isfinite(pred):\n        assert math.isclose(pred - reference, matched_pred - matched_ref + unmatched_pred - missed_ref, rel_tol=1e-09, abs_tol=1e-06)\n    return row\n\ndef comparison_statistics(truth, pred):\n    result = errors(truth, pred)\n    a = np.asarray(truth, dtype=float)\n    b = np.asarray(pred, dtype=float)\n    ok = np.isfinite(a) & np.isfinite(b)\n    a = a[ok]\n    b = b[ok]\n    mean_ref = float(a.mean()) if len(a) else np.nan\n    mean_est = float(b.mean()) if len(b) else np.nan\n    result.update(mean_reference=mean_ref, mean_estimate=mean_est, estimate_reference_ratio=mean_est / mean_ref if mean_ref > 0 else np.nan)\n    return result\n\ndef finalize_detection_frame(records, method):\n    cols = DET_COLUMNS if \'DET_COLUMNS\' in globals() else [\'det_id\', \'plotID\', \'siteID\', \'field_year\', \'score\', \'H_m\', \'CD_m\', \'geometry\']\n    frame = gpd.GeoDataFrame(records, geometry=\'geometry\', crs=WORK_CRS) if len(records) else gpd.GeoDataFrame(columns=cols, geometry=\'geometry\', crs=WORK_CRS)\n    frame[\'method\'] = method\n    frame[\'DBH_jucker_cm\'] = estimate_dbh(frame.H_m, frame.CD_m)\n    frame[\'AGB_default_kg\'] = estimate_agb(frame.DBH_jucker_cm)\n    frame[\'C_default_kg\'] = carbon_kg(frame.AGB_default_kg, CFG[\'carbon_fraction\'])\n    return frame\n\ndef deepforest_patch_candidates(detector, rgb, chm, plot, patch_size, cache_dir):\n    saved = []\n    for obj in [detector.model, getattr(detector.model, \'model\', None)]:\n        if obj is not None and hasattr(obj, \'score_thresh\'):\n            saved.append((obj, \'score_thresh\', obj.score_thresh))\n            obj.score_thresh = 0.05\n    configs = []\n    for obj in [detector.config, getattr(detector.config, \'retinanet\', None)]:\n        if obj is None:\n            continue\n        try:\n            old = obj[\'score_thresh\']\n            obj[\'score_thresh\'] = 0.05\n            configs.append((obj, old))\n        except (KeyError, TypeError, AttributeError):\n            pass\n    if not saved and (not configs):\n        raise RuntimeError(\'DeepForest 내부 score_thresh를 0.05로 설정하지 못했습니다. 낮은 점수 탐색을 숨겨 진행하지 않습니다.\')\n    oldcfg = CFG.copy()\n    oldout = globals()[\'OUT\']\n    directory = Path(cache_dir) / f\'patch_{patch_size}\'\n    directory.mkdir(parents=True, exist_ok=True)\n    fingerprint = dict(rgb=str(rgb), rgb_bytes=Path(rgb).stat().st_size, patch_size=int(patch_size), patch_overlap=CFG[\'patch_overlap\'], nms_iou=CFG[\'nms_iou\'], min_score=0.1, model_version=importlib.metadata.version(\'deepforest\'), model=CFG[\'model_name\'], revision=CFG[\'model_revision\'])\n    stamp = directory / f\'{plot.plotID}_input.json\'\n    rawpath = directory / f\'{plot.plotID}_raw_predictions.csv\'\n    if rawpath.exists() and (not stamp.exists() or json.loads(stamp.read_text()) != fingerprint):\n        rawpath.unlink()\n    try:\n        CFG[\'patch_size\'] = int(patch_size)\n        CFG[\'score_threshold\'] = 0.1\n        globals()[\'OUT\'] = directory\n        rows = predict_crowns(detector, rgb, chm, plot)\n        stamp.write_text(json.dumps(fingerprint, indent=2))\n    finally:\n        CFG.clear()\n        CFG.update(oldcfg)\n        globals()[\'OUT\'] = oldout\n        for obj, key, value in saved:\n            setattr(obj, key, value)\n        for obj, value in configs:\n            obj[\'score_thresh\'] = value\n    for row in rows:\n        row[\'det_id\'] = f"{plot.plotID}_p{patch_size}_{row[\'det_id\'].split(\'_\')[-1]}"\n    return finalize_detection_frame(rows, f\'DF_patch_{patch_size}\')\n\ndef variable_window_peaks(height, valid, resolution_m, diameter_a=2.51, diameter_b=0.00901, min_height=3.0):\n    mask = valid & (height >= min_height)\n    cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\n    if resolution_m > (diameter_a + diameter_b * min_height ** 2) / 2:\n        raise ValueError(\'해상도가 최소 가변 창 반지름보다 큽니다. 더 세밀한 CHM 또는 현지 창 계수를 사용하세요.\')\n    maxima = mask & (height >= maximum_filter(np.where(valid, height, -np.inf), footprint=cross, mode=\'constant\', cval=-np.inf))\n    components, count = ndi_label(maxima, structure=cross.astype(int))\n    candidates = []\n    for k in range(1, count + 1):\n        coords = np.argwhere(components == k)\n        order = np.lexsort((coords[:, 1], coords[:, 0], -height[coords[:, 0], coords[:, 1]]))\n        r, c = coords[order[0]]\n        H = float(height[r, c])\n        radius = (diameter_a + diameter_b * H * H) / 2\n        pixels = int(np.ceil(radius / resolution_m))\n        r0 = max(0, r - pixels)\n        r1 = min(height.shape[0], r + pixels + 1)\n        c0 = max(0, c - pixels)\n        c1 = min(height.shape[1], c + pixels + 1)\n        rr, cc = np.ogrid[r0:r1, c0:c1]\n        circle = ((rr - r) * resolution_m) ** 2 + ((cc - c) * resolution_m) ** 2 <= radius ** 2\n        neighborhood = height[r0:r1, c0:c1][circle & valid[r0:r1, c0:c1]]\n        if len(neighborhood) and H >= float(neighborhood.max()) - 1e-07:\n            candidates.append((int(r), int(c), H, float(radius * 2)))\n    return sorted(candidates, key=lambda x: (-x[2], x[0], x[1]))\n\ndef chm_watershed_crowns(chm_path, plot, sigma_m=1.0):\n    with rasterio.open(chm_path) as ds:\n        raw = ds.read(1, masked=True)\n        transform = ds.transform\n        source_crs = ds.crs\n        resolution = float(abs(transform.a))\n        if not np.isclose(abs(transform.a), abs(transform.e)):\n            raise ValueError(\'CHM 정사각형 픽셀 필요\')\n        rawdata = np.asarray(raw.astype(float).filled(np.nan), dtype=float)\n        valid = np.isfinite(rawdata) & (rawdata >= 0)\n        weight = gaussian_filter(valid.astype(float), sigma=sigma_m / resolution)\n        smooth = gaussian_filter(np.where(valid, rawdata, 0), sigma=sigma_m / resolution) / np.maximum(weight, 1e-12)\n        peaks = variable_window_peaks(smooth, valid, resolution, CFG[\'chm_window_a\'], CFG[\'chm_window_b\'], CFG[\'min_height_m\'])\n        markers = np.zeros(rawdata.shape, dtype=np.int32)\n        accepted = []\n        for r, c, H, diameter in peaks:\n            if rawdata[r, c] < CFG[\'min_height_m\']:\n                continue\n            k = len(accepted) + 1\n            markers[r, c] = k\n            accepted.append((k, r, c, float(rawdata[r, c]), diameter))\n        if not accepted:\n            return finalize_detection_frame([], \'B_CHM\')\n        segments = watershed(-smooth, markers, mask=valid & (rawdata >= CFG[\'min_height_m\']))\n        crown_parts = {}\n        for geom, value in raster_shapes(segments.astype(\'int32\'), mask=segments > 0, transform=transform):\n            crown_parts.setdefault(int(value), []).append(shape(geom))\n    rows = []\n    for k, r, c, H, diameter in accepted:\n        xy = transform * (c + 0.5, r + 0.5)\n        point = gpd.GeoSeries([Point(*xy)], crs=source_crs).to_crs(WORK_CRS).iloc[0]\n        if not plot.geometry.covers(point):\n            continue\n        polygon = gpd.GeoSeries([unary_union(crown_parts[k])], crs=source_crs).to_crs(WORK_CRS).iloc[0]\n        area = float(polygon.area)\n        if area <= 0:\n            continue\n        rows.append(dict(det_id=f\'{plot.plotID}_CHM_{k}\', plotID=plot.plotID, siteID=plot.siteID, field_year=plot.field_year, imagery_year=plot.imagery_year, score=1.0, H_m=H, CD_m=2 * np.sqrt(area / np.pi), crown_area_m2=area, crown_geometry_wkt=polygon.wkt, peak_x=point.x, peak_y=point.y, window_diameter_m=diameter, geometry=polygon.envelope))\n    return finalize_detection_frame(rows, \'B_CHM\')\n\ndef combine_boxes_and_peaks(boxes, chm, dedup_m=1.0):\n    rows = []\n    used = set()\n    for _, boxrow in boxes.sort_values([\'score\', \'det_id\'], ascending=[False, True]).iterrows():\n        candidates = chm[chm.apply(lambda p: boxrow.geometry.covers(Point(p.peak_x, p.peak_y)), axis=1)] if len(chm) else chm\n        had_seed = len(candidates) > 0\n        candidates = candidates[~candidates.det_id.isin(used)]\n        if had_seed and candidates.empty:\n            continue\n        row = boxrow.to_dict()\n        row[\'det_id\'] = \'C_\' + str(row[\'det_id\'])\n        if len(candidates):\n            peak = candidates.sort_values([\'H_m\', \'det_id\'], ascending=[False, True]).iloc[0]\n            used.add(peak.det_id)\n            row.update(H_m=peak.H_m, peak_x=peak.peak_x, peak_y=peak.peak_y)\n        else:\n            row.update(peak_x=boxrow.geometry.centroid.x, peak_y=boxrow.geometry.centroid.y)\n        point = Point(row[\'peak_x\'], row[\'peak_y\'])\n        if any((point.distance(Point(x[\'peak_x\'], x[\'peak_y\'])) < dedup_m for x in rows)):\n            continue\n        row[\'fusion_source\'] = \'DeepForest_box\'\n        rows.append(row)\n    for _, p in chm.iterrows():\n        point = Point(p.peak_x, p.peak_y)\n        if any((g.covers(point) for g in boxes.geometry)):\n            continue\n        if any((point.distance(Point(x[\'peak_x\'], x[\'peak_y\'])) < dedup_m for x in rows)):\n            continue\n        row = p.to_dict()\n        row[\'det_id\'] = \'C_\' + row[\'det_id\']\n        row[\'fusion_source\'] = \'outside_CHM_peak\'\n        rows.append(row)\n    return finalize_detection_frame(rows, \'C_fusion\')\n\ndef detection_counts(detections, truth, plot_ids):\n    tp = fp = fn = 0\n    for pid in plot_ids:\n        d = detections[detections.plotID == pid]\n        f = truth[truth.plotID == pid]\n        hits = len(match_one_to_one(d, f))\n        tp += hits\n        fp += len(d) - hits\n        fn += len(f) - hits\n    return dict(TP=tp, FP=fp, FN=fn, precision=tp / (tp + fp) if tp + fp else np.nan, recall=tp / (tp + fn) if tp + fn else np.nan, F1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan)\n\ndef nested_group_parameter_selection(candidates, plot_table, truth, n_splits=5):\n    ids = np.array(sorted(plot_table.plotID.unique()))\n    outer = GroupKFold(n_splits=min(n_splits, len(ids)))\n    predictions = []\n    selections = []\n    inner_rows = []\n    split_rows = []\n    keys = sorted(candidates)\n    for fold, (train, test) in enumerate(outer.split(ids, groups=ids)):\n        trainids = ids[train]\n        testids = ids[test]\n        assert set(trainids).isdisjoint(testids)\n        for pid in trainids:\n            split_rows.append(dict(outer_fold=fold, plotID=pid, role=\'train\'))\n        for pid in testids:\n            split_rows.append(dict(outer_fold=fold, plotID=pid, role=\'test\'))\n        inner = GroupKFold(n_splits=min(CFG[\'search_inner_folds\'], len(trainids)))\n        scores = {key: dict(TP=0, FP=0, FN=0) for key in keys}\n        for inner_fold, (itr, ival) in enumerate(inner.split(trainids, groups=trainids)):\n            assert set(trainids[itr]).isdisjoint(trainids[ival])\n            assert set(trainids[ival]).isdisjoint(testids)\n            for key in keys:\n                result = detection_counts(candidates[key], truth, trainids[ival])\n                inner_rows.append(dict(outer_fold=fold, inner_fold=inner_fold, patch_size=key[0], score_threshold=key[1], **result))\n                for k in [\'TP\', \'FP\', \'FN\']:\n                    scores[key][k] += result[k]\n\n        def ranking(key):\n            counts = scores[key]\n            den = 2 * counts[\'TP\'] + counts[\'FP\'] + counts[\'FN\']\n            f1 = 2 * counts[\'TP\'] / den if den else -1\n            return (-f1, key[0], key[1])\n        best = min(keys, key=ranking)\n        chosen = candidates[best][candidates[best].plotID.isin(testids)].copy()\n        chosen[\'outer_fold\'] = fold\n        chosen[\'selected_patch_size\'] = best[0]\n        chosen[\'selected_score_threshold\'] = best[1]\n        predictions.append(chosen)\n        selections.append(dict(outer_fold=fold, patch_size=best[0], score_threshold=best[1], train_plots=\';\'.join(trainids), test_plots=\';\'.join(testids), **detection_counts(chosen, truth, testids)))\n    oof = gpd.GeoDataFrame(pd.concat(predictions, ignore_index=True), geometry=\'geometry\', crs=WORK_CRS)\n    deployment_rows = []\n    for key in keys:\n        deployment_rows.append(dict(patch_size=key[0], score_threshold=key[1], **detection_counts(candidates[key], truth, ids)))\n    deployment = pd.DataFrame(deployment_rows).sort_values([\'F1\', \'patch_size\', \'score_threshold\'], ascending=[False, True, True])\n    return (oof, pd.DataFrame(selections), pd.DataFrame(inner_rows), pd.DataFrame(split_rows), deployment)\n\ndef evaluate_detection_method(det, plot_table, all_truth, coordinate_truth):\n    metric_rows = []\n    pair_tables = {}\n    joined = None\n    for population in [\'visible\', \'all\']:\n        pairs = []\n        for _, p in plot_table.iterrows():\n            d = det[det.plotID == p.plotID]\n            f = coordinate_truth[coordinate_truth.plotID == p.plotID]\n            if population == \'visible\':\n                f = f[f.visible]\n            match = match_one_to_one(d, f)\n            pairs.append(match)\n            metric_rows.append(dict(plotID=p.plotID, siteID=p.siteID, field_year=p.field_year, population=population, **detection_counts(d, f, [p.plotID])))\n        pair_tables[population] = pd.concat(pairs, ignore_index=True)\n    visible = pair_tables[\'visible\']\n    joined = visible.merge(det.drop(columns=\'geometry\'), on=[\'det_id\', \'plotID\'], validate=\'one_to_one\')\n    joined = joined.merge(coordinate_truth.drop(columns=[\'geometry\', \'plotID\', \'siteID\', \'field_year\']), on=\'individualID\', validate=\'one_to_one\')\n    joined[\'DBH_regression_oof_cm\'], joined[\'cv_fold\'], _ = grouped_dbh_regression(joined, CFG[\'cv_folds\'])\n    joined[\'AGB_regression_oof_kg\'] = estimate_agb(joined.DBH_regression_oof_cm)\n    joined[\'AGB_known_species_kg\'] = estimate_agb(joined.DBH_jucker_cm, joined.species_beta0, joined.species_beta1)\n    joined[\'species_effect_kg\'] = joined.AGB_known_species_kg - joined.AGB_default_kg\n    joined[\'species_effect_pct\'] = 100 * (joined.AGB_known_species_kg / joined.AGB_default_kg - 1)\n    plotrows = []\n    for _, p in plot_table.iterrows():\n        ds = det[det.plotID == p.plotID]\n        fa = all_truth[all_truth.plotID == p.plotID]\n        fc = coordinate_truth[coordinate_truth.plotID == p.plotID]\n        pairs = pair_tables[\'all\']\n        pairs = pairs[pairs.plotID == p.plotID]\n        plotrows.append(plot_biomass_totals(ds.to_dict(\'records\'), fa.to_dict(\'records\'), fc.to_dict(\'records\'), pairs.to_dict(\'records\'), p.to_dict(), CFG[\'carbon_fraction\'], CFG[\'carbon_alternative\']))\n    return dict(detections=det, matched=joined, metrics=pd.DataFrame(metric_rows), pairs=pair_tables, plots=pd.DataFrame(plotrows))\n\ndef method_comparison_rows(method_results, requested_sites):\n    rows = []\n    for method, result in method_results.items():\n        for site in [\'ALL\'] + list(requested_sites):\n            pc = result[\'plots\']\n            mt = result[\'matched\']\n            dm = result[\'metrics\']\n            if site != \'ALL\':\n                pc = pc[pc.siteID == site]\n                mt = mt[mt.siteID == site]\n                dm = dm[dm.siteID == site]\n            stats = comparison_statistics(pc.C_ref_tC_ha, pc.C_est_tC_ha)\n            row = dict(method=method, siteID=site, n_plots=len(pc), n_field=int(pc.n_truth.sum()), n_detected=int(pc.n_detected.sum()), n_carbon_plots=stats[\'n\'], C_est_mean_tC_ha=stats[\'mean_estimate\'], C_ref_mean_tC_ha=stats[\'mean_reference\'], estimate_reference_ratio=stats[\'estimate_reference_ratio\'], carbon_r=stats[\'r\'], carbon_RMSE_tC_ha=stats[\'RMSE\'], carbon_MAE_tC_ha=stats[\'MAE\'])\n            for pop in [\'visible\', \'all\']:\n                q = dm[dm.population == pop]\n                tp = q.TP.sum()\n                fp = q.FP.sum()\n                fn = q.FN.sum()\n                row[\'F1_\' + pop] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan\n                row[\'precision_\' + pop] = tp / (tp + fp) if tp + fp else np.nan\n                row[\'recall_\' + pop] = tp / (tp + fn) if tp + fn else np.nan\n            for name, a, b in [(\'height_Bias_m\', \'field_H_m\', \'H_m\'), (\'crown_Bias_m\', \'field_CD_m\', \'CD_m\'), (\'DBH_Bias_cm\', \'field_DBH_cm\', \'DBH_jucker_cm\'), (\'individual_AGB_Bias_kg\', \'ref_AGB_kg\', \'AGB_default_kg\')]:\n                row[name] = errors(mt[a], mt[b])[\'Bias\']\n            rows.append(row)\n    return pd.DataFrame(rows)\n\ndef detection_algorithm_smoke_checks():\n    from types import SimpleNamespace\n    yy, xx = np.mgrid[:64, :64]\n    canopy = 20 * np.exp(-((xx - 18) ** 2 + (yy - 32) ** 2) / 8) + 30 * np.exp(-((xx - 46) ** 2 + (yy - 32) ** 2) / 8)\n    path = OUT / \'synthetic_CHM_smoke.tif\'\n    with rasterio.open(path, \'w\', driver=\'GTiff\', width=64, height=64, count=1, dtype=\'float32\', crs=WORK_CRS, transform=rasterio.transform.from_origin(0, 64, 1, 1), nodata=-9999) as ds:\n        ds.write(canopy.astype(\'float32\'), 1)\n    p = SimpleNamespace(plotID=\'SMOKE\', siteID=\'SMOKE\', field_year=2021, imagery_year=2021, geometry=box(0, 0, 64, 64))\n    result = chm_watershed_crowns(path, p, 1.0)\n    assert len(result) == 2, \'합성 두 나무의 꼭대기 2개를 찾아야 합니다\'\n    assert np.allclose(sorted(result.H_m), [20, 30], atol=0.01)\n    assert np.allclose(result.CD_m, 2 * np.sqrt(result.crown_area_m2 / np.pi))\n    assert np.isclose(result.crown_area_m2.sum(), np.count_nonzero(canopy >= CFG[\'min_height_m\']))\n    first_peak = result.iloc[0].to_dict()\n    duplicate = dict(first_peak)\n    duplicate[\'det_id\'] = \'duplicate_box\'\n    duplicate[\'score\'] = 0.8\n    first_peak[\'score\'] = 0.9\n    boxes = gpd.GeoDataFrame([first_peak, duplicate], geometry=\'geometry\', crs=WORK_CRS)\n    fused = combine_boxes_and_peaks(boxes, result, 1.0)\n    assert len(fused) == 2, \'동일 꼭대기 중복 상자는 제거하고 상자 밖 두 번째 꼭대기는 추가해야 합니다\'\n    ids = [\'P0\', \'P1\', \'P2\', \'P3\']\n    drows = []\n    frows = []\n    for i, pid in enumerate(ids):\n        x = i * 20\n        drows.append(dict(det_id=\'d\' + pid, plotID=pid, score=0.8, H_m=20, CD_m=5, geometry=box(x - 3, -3, x + 3, 3)))\n        frows.append(dict(individualID=\'f\' + pid, plotID=pid, stemDiameter=30, geometry=Point(x, 0)))\n    ds = gpd.GeoDataFrame(drows, geometry=\'geometry\', crs=WORK_CRS)\n    fs = gpd.GeoDataFrame(frows, geometry=\'geometry\', crs=WORK_CRS)\n    plotdf = pd.DataFrame({\'plotID\': ids})\n    grid = {(300, 0.1): ds, (400, 0.2): ds.iloc[:2].copy()}\n    _, first, _, _, _ = nested_group_parameter_selection(grid, plotdf, fs, n_splits=2)\n    changed = fs.copy()\n    changed.loc[changed.plotID == \'P0\', \'geometry\'] = Point(9999, 9999)\n    _, second, _, _, _ = nested_group_parameter_selection(grid, plotdf, changed, n_splits=2)\n    fold = first[first.test_plots.str.split(\';\').apply(lambda z: \'P0\' in z)].index[0]\n    assert first.loc[fold, [\'patch_size\', \'score_threshold\']].equals(second.loc[fold, [\'patch_size\', \'score_threshold\']]), \'바깥 평가 정답이 선택에 새어 들어갔습니다\'\n    for _, r in first.iterrows():\n        assert set(r.train_plots.split(\';\')).isdisjoint(r.test_plots.split(\';\'))\n    path.unlink(missing_ok=True)\n    print(\'합성 CHM 꼭대기·watershed 면적·CD, GroupKFold 정답 격리 검증 통과\')\n\ndef plot_carbon_scatter(plot_table, output_folder):\n    for (year, image_year), cohort in plot_table.groupby([\'field_year\', \'imagery_year\']):\n        z = cohort.replace([np.inf, -np.inf], np.nan).dropna(subset=[\'C_ref_tC_ha\', \'C_est_tC_ha\'])\n        fig, ax = plt.subplots(figsize=(6, 6))\n        for site, q in z.groupby(\'siteID\'):\n            ax.scatter(q.C_ref_tC_ha, q.C_est_tC_ha, s=35, alpha=0.8, label=site, color={\'GRSM\': \'tab:blue\', \'ORNL\': \'tab:orange\'}.get(site))\n        if len(z):\n            hi = max(z[[\'C_ref_tC_ha\', \'C_est_tC_ha\']].max()) * 1.05\n            hi = max(hi, 1.0)\n            ax.plot([0, hi], [0, hi], \'k--\', label=\'1:1\')\n            ax.set_xlim(0, hi)\n            ax.set_ylim(0, hi)\n        ax.set(xlabel=\'Field-based reference carbon (tC/ha)\', ylabel=\'Estimated carbon (tC/ha)\', title=f\'Plot carbon: field {year} / imagery {image_year} (n={len(z)})\')\n        ax.legend()\n        ax.set_aspect(\'equal\', adjustable=\'box\')\n        filename = f\'plot_carbon_{year}.png\' if year == image_year else f\'plot_carbon_field{year}_imagery{image_year}.png\'\n        fig.savefig(Path(output_folder) / filename, dpi=170, bbox_inches=\'tight\')\n        plt.close(fig)\n\ndef build_site_summary(plot_table, detection_metrics, matched_table, requested_sites):\n    rows = []\n    for site in requested_sites:\n        site_plots = plot_table[plot_table.siteID == site]\n        if site_plots.empty:\n            rows.append(dict(siteID=site, n_plots=0, comparison_note=\'분석 결과 없음\'))\n            continue\n        for (field_year, image_year), q in site_plots.groupby([\'field_year\', \'imagery_year\']):\n            ids = set(q.plotID)\n            dm = detection_metrics[detection_metrics.plotID.isin(ids)]\n            mt = matched_table[matched_table.plotID.isin(ids)]\n            row = dict(siteID=site, field_year=field_year, imagery_year=image_year, comparison_note=\'영상 연도 = 실측 연도\' if field_year == image_year else f\'{site}은 영상 연도 ≠ 실측 연도\', n_plots=len(q), n_field_trees=int(q.n_truth.sum()), n_coord_trees=int(q.n_truth_coord.sum()), n_detections=int(q.n_detected.sum()))\n            for population, label in [(\'visible\', \'F1_visible\'), (\'all\', \'F1_all\')]:\n                z = dm[dm.population == population]\n                tp = z.TP.sum()\n                fp = z.FP.sum()\n                fn = z.FN.sum()\n                row[label] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan\n            for label, refcol, predcol in [(\'height_Bias_m\', \'field_H_m\', \'H_m\'), (\'DBH_Bias_cm\', \'field_DBH_cm\', \'DBH_jucker_cm\'), (\'individual_AGB_Bias_kg\', \'ref_AGB_kg\', \'AGB_default_kg\')]:\n                row[label] = errors(mt[refcol], mt[predcol])[\'Bias\']\n            stats = comparison_statistics(q.C_ref_tC_ha, q.C_est_tC_ha)\n            row.update(n_carbon_plots=stats[\'n\'], C_est_mean_tC_ha=stats[\'mean_estimate\'], C_ref_mean_tC_ha=stats[\'mean_reference\'], estimate_reference_ratio=stats[\'estimate_reference_ratio\'])\n            rows.append(row)\n    return pd.DataFrame(rows)\n',encoding='utf-8')
    print(OUT/'neon_pipeline.py')

## 19. 한국 자료로 재사용하기

1. 설정의 `WORK_CRS`를 한국 입력의 미터 투영 CRS로 바꾸세요.
2. `RASTERS={plotID:(RGB경로, CHM경로)}`, `plots`(실제 조사 geometry·area_m2·siteID·field_year·중심 좌표),
   `field_all`(좌표 유무와 무관한 전체 실측·참조 AGB)과 `field_geo`(매칭 가능한 실측 점)를 같은 필드로 준비하면
   NEON 다운로드 단계를 건너뛸 수 있습니다. `plots.imagery_year`로 영상 연도를 별도로 지정하세요.
3. Jucker 계수와 AGB 계수를 한국 수종·지역 검증식으로 교체하세요. 북미 계수를 그대로 적용한 값은 탐색적 추정입니다.
4. `predict_crowns`, `estimate_dbh`, `estimate_agb`, `label_visibility`, `match_one_to_one`,
   `grouped_dbh_regression`, `errors`, `kg_to_t_ha`를 재사용하세요.
5. 실측 DBH≥10cm라는 조사 범위와 탐지 가능한 모든 수관의 모집단 차이를 고려하세요.
   정확한 전체 산림 탄소 추정에는 작은 나무·관목·아래층을 포함하는 별도 설계가 필요합니다.

**결과 확인 순서:** audit.csv → sampled_footprints.gpkg/overlay 그림 → detection_metrics.csv →
stage_errors.csv → plot_comparison.csv → height_sensitivity.csv → report.md.

이 노트북은 로컬에서 문법·구조를 검사했으며 Colab에서의 다운로드/GPU 실행 결과는 포함하지 않습니다.

## 20. 사람이 읽는 최종 요약

사이트별 조사구·전체 실측·매칭 좌표 나무·탐지 수와 정확도를 한 표로 봅니다.
보고서 맨 위와 동일한 요약이며, 영상 연도와 실측 연도가 다르면 행을 분리합니다.
Bias는 추정−참조, F1 기본은 보이는 나무, 탄소 비율은 동일한 유효 조사구들의 평균 추정/평균 참조입니다.

해상도별 F1_visible·탄소 추정/참조·RMSE를 마지막에 추가 출력합니다.

In [ ]:
if not RESOLUTION_ONLY:
    display(summary_display.round(3))
    display(method_comparison.round(3))
    print('이후 기본 나무 찾기 방법:',DEFAULT_DETECTION_METHOD)
    print('전체 참조값에는 좌표 없는 나무가 포함됩니다. 자세한 자료는',OUT/'field_trees_all.csv')
    print('보고서:',OUT/'report.md')
print('해상도별 최종 요약 — ALL, 탄소 RMSE 단위:tC/ha')
resolution_summary_display = RESOLUTION_SUMMARY.loc[RESOLUTION_SUMMARY.siteID=='ALL',
    ['resolution_cm','setting','F1_visible','estimate_reference_ratio','carbon_RMSE','selected_score_threshold']].copy()
resolution_summary_display['setting'] = resolution_summary_display.setting.map({'fixed':'고정','retuned':'재선택(OOF)'})
resolution_summary_display = resolution_summary_display.rename(columns={
    'resolution_cm':'해상도(cm)','setting':'설정','F1_visible':'F1(보이는 나무)',
    'estimate_reference_ratio':'탄소 추정/참조','carbon_RMSE':'RMSE(tC/ha)','selected_score_threshold':'선택 score'})
display(resolution_summary_display.round(3))
print('fixed=.2/600; retuned=patch600, inner F1 선택 → outer OOF 평가')
print('10cm 자동 비교:', OUT/'resolution_10cm_validation.csv')
print('보고서:', OUT/'report.md')

## 21. 결과 ZIP 저장 및 다운로드

현재 실행 결과 폴더만 ZIP으로 묶습니다.
`/content/neon_results.zip`을 만들고 Drive의 outputs/에도 사본을 남긴 뒤 브라우저 다운로드를 시작합니다.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/neon_results','zip',root_dir=str(OUT))
shutil.copy2(archive,ROOT/'outputs/neon_results.zip')
print('ZIP:',archive,'/ Drive 사본:',ROOT/'outputs/neon_results.zip')
files.download('/content/neon_results.zip')